# E-CUP 2026 — bootstrap

**Run All** воссоздаёт дерево проекта в текущем каталоге и ставит зависимости.
Каждый файл едет своей `%%writefile`-ячейкой, поэтому тетрадь самодостаточна:
ни git, ни scp на машине не нужны — хватает одного Jupyter.

Перезапускать можно сколько угодно, файлы просто перезаписываются.

Данные и веса сюда не входят: данные качаются на месте
(`src/scripts/01_fetch_data.py`, хранилище организаторов), веса обучаются заново
тетрадями `01_mmbert.ipynb` и `02_bge.ipynb`.

Раздел 4 выравнивает оболочку с ядром Jupyter: `python3` и `pip` в `!`-командах
всех тетрадей — это тот же интерпретатор, в который ставятся зависимости.

Собрана `build.py` из дерева проекта; править файлы здесь бессмысленно —
следующая сборка затрёт правку.

После прогона этой тетради — по порядку:
`01_mmbert.ipynb` → `02_bge.ipynb` → `03_blend.ipynb`.

## 1. Каталоги

In [ ]:
import os, pathlib, sys

os.makedirs('src', exist_ok=True)
os.makedirs('src/ecup', exist_ok=True)
os.makedirs('src/scripts', exist_ok=True)
os.makedirs('src/solution', exist_ok=True)
os.makedirs('src/utils', exist_ok=True)

ROOT = pathlib.Path.cwd().resolve()
os.environ['ECUP_ROOT'] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('каталогов создано:', 5, '| ECUP_ROOT =', ROOT)

## 2. Файлы проекта — 49 штук

In [ ]:
%%writefile run.py
from __future__ import annotations

import argparse
import os
import sys
import time

T0 = time.time()
HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

ROOT_DIR = os.environ.get("ECUP_ROOT") or HERE
os.environ.setdefault("ECUP_ROOT", ROOT_DIR)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")


def log(msg: str) -> None:
    print(f"[{time.time() - T0:7.1f}s] {msg}", flush=True)


def canary() -> None:
    log(f"python {sys.version.split()[0]} | cwd={os.getcwd()} | HERE={HERE} "
        f"| ROOT={ROOT_DIR}")
    try:
        log(f"содержимое каталога решения: {sorted(os.listdir(HERE))}")
        art = os.path.join(ROOT_DIR, "artifacts")
        if os.path.isdir(art):
            log(f"artifacts: {sorted(os.listdir(art))}")
            for name in sorted(os.listdir(art)):
                sub = os.path.join(art, name)
                if os.path.isdir(sub):
                    log(f"  artifacts/{name}: {sorted(os.listdir(sub))}")
    except Exception as e:
        log(f"listdir не удался: {e}")
    try:
        import shutil
        shm = shutil.disk_usage("/dev/shm")
        log(f"cpu={os.cpu_count()} /dev/shm={shm.total/1e6:.0f}МБ свободно={shm.free/1e6:.0f}МБ")
    except Exception as e:
        log(f"/dev/shm недоступен: {e}")
    log(f"каталог решения на запись: {os.access(HERE, os.W_OK)} | /tmp: {os.access('/tmp', os.W_OK)}")
    for m in ("numpy", "pyarrow", "torch", "transformers"):
        try:
            x = __import__(m)
            log(f"  импорт {m:16s} OK {getattr(x, '__version__', '')}")
        except Exception as e:
            log(f"  импорт {m:16s} СБОЙ {type(e).__name__}: {e}")
    for m in ("src.ecup.config", "src.ecup.textprep", "src.ecup.serialize",
              "src.ecup.infer", "src.ecup.solve"):
        try:
            __import__(m)
            log(f"  импорт {m:16s} OK")
        except Exception as e:
            log(f"  импорт {m:16s} СБОЙ {type(e).__name__}: {e}")
    try:
        import torch
        log(f"  cuda={torch.cuda.is_available()} устройств={torch.cuda.device_count()}")
        if torch.cuda.is_available():
            log(f"  gpu={torch.cuda.get_device_name(0)}")
    except Exception as e:
        log(f"  torch.cuda недоступен: {type(e).__name__}: {e}")


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--items_path", "--items-path", "-i", dest="items_path")
    p.add_argument("--matches_path", "--matches-path", "-m", dest="matches_path")
    p.add_argument("--output_path", "--output-path", "-o", dest="output_path")
    a, unknown = p.parse_known_args()
    if unknown:
        log(f"неизвестные аргументы проигнорированы: {unknown}")
    miss = [k for k in ("items_path", "matches_path", "output_path") if not getattr(a, k)]
    if miss:
        raise SystemExit(f"не переданы обязательные аргументы: {miss}")
    return a


def write_csv(path: str, id1, id2, scores) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write("id1,id2,predict\n")
        for a, b, s in zip(id1, id2, scores):
            f.write(f"{a},{b},{s:.6g}\n")


def main() -> None:
    args = parse_args()
    canary()
    log(f"items={args.items_path} matches={args.matches_path} out={args.output_path}")

    import numpy as np
    import pyarrow.parquet as pq

    mt = pq.read_table(args.matches_path, columns=["id1", "id2"])
    id1 = mt.column("id1").to_numpy(zero_copy_only=False).tolist()
    id2 = mt.column("id2").to_numpy(zero_copy_only=False).tolist()
    n = len(id1)
    log(f"пар: {n:,}")

    it = pq.read_table(args.items_path, columns=["id", "name", "attributes", "category"])
    ids = it.column("id").to_numpy(zero_copy_only=False).tolist()
    names = it.column("name").to_pylist()
    attrs = it.column("attributes").to_pylist()
    cats = it.column("category").to_pylist()
    pos = {v: i for i, v in enumerate(ids)}
    if len(pos) != len(ids):
        log(f"ВНИМАНИЕ: в таблице товаров {len(ids) - len(pos):,} повторяющихся id, "
            f"остаётся последняя строка")
    log(f"товаров: {len(ids):,}")

    log("режим: ГРОМКИЙ (при ошибке traceback и код возврата 1, файла не будет)")
    from src.ecup.solve import predict
    scores = predict(names, attrs, cats, pos, id1, id2, ROOT_DIR, T0, log)

    write_csv(args.output_path, id1, id2, np.asarray(scores, dtype=np.float64).tolist())
    log(f"записано {n:,} строк в {args.output_path}")


if __name__ == "__main__":
    try:
        main()
    except SystemExit:
        raise
    except BaseException:
        import traceback
        log("АВАРИЯ: выходной файл НЕ записан, полный traceback ниже")
        traceback.print_exc()
        sys.stdout.flush()
        sys.stderr.flush()
        sys.exit(1)


In [ ]:
%%writefile requirements.txt
# Обучение и подготовка данных. Версии — те, на которых прогон воспроизводился.
#
# Разделение важное: КРИТИЧЕСКИЙ ПУТЬ РЕШЕНИЯ внутри контейнера обходится
# четырьмя нижними строками (numpy, pyarrow, torch, transformers) — ровно тем,
# что уже есть в официальном образе `odsai/ecup26-matching-baseline:1.0`.
# Всё остальное нужно только на своей машине: нарезка корпусов, обучение,
# сборка архива и проверка. В образ оно не едет и ехать не должно.

# --- нужно и на обучении, и на инференсе -----------------------------------
torch==2.11.0            # ставится с индекса https://download.pytorch.org/whl/cu128
transformers==5.5.3
tokenizers>=0.21
safetensors>=0.4
numpy>=2.2
pyarrow>=19

# --- только на своей машине -------------------------------------------------
polars>=1.17             # чтение parquet и нарезка фолдов (src/utils/data.py,
                         # folds.py, solution/batch.slice_folds — импорт ленивый,
                         # внутри контейнера не выполняется)
scipy>=1.14              # connected_components: компоненты связности графа пар
pandas>=2.2
scikit-learn>=1.6        # average_precision_score — метрика соревнования
tqdm>=4.67
requests>=2.32           # скачивание данных соревнования


In [ ]:
%%writefile src/__init__.py
from __future__ import annotations

import sys as _sys

for _stream in (_sys.stdout, _sys.stderr):
    try:
        _stream.reconfigure(encoding="utf-8", errors="replace")
    except (AttributeError, ValueError):
        pass


In [ ]:
%%writefile src/config.py
from __future__ import annotations

import os
from pathlib import Path


def _detect_root() -> Path:
    env = os.environ.get("ECUP_ROOT")
    if env:
        return Path(env).expanduser().resolve()

    here = Path(__file__).resolve()
    for parent in here.parents:
        if (parent / "src").is_dir():
            return parent
    return Path.cwd().resolve()


ROOT = _detect_root()

DATA = ROOT / "data"
RAW = DATA / "raw"
PROC = DATA / "processed"
LOCAL_RUN = DATA / "local_run"

ARTIFACTS = ROOT / "artifacts"
BUILD = ROOT / "build"
SUBMISSIONS = ROOT / "submissions"


BASE_URL = "https://storage.yandexcloud.net/ozon-ecup-2026/Matching"
DOCKER_IMAGE = "odsai/ecup26-matching-baseline:1.0"
ENTRY_POINT = "python -u run.py"

N_FOLDS = 5
SEED = 42

TEST_POS_RATE = {
    "Автотовары": 0.1178, "Аптека": 0.1273, "Бытовая техника": 0.2099,
    "Бытовая химия": 0.2119, "Галантерея и аксессуары": 0.0430,
    "Детские товары": 0.1468, "Дом и сад": 0.0939, "Канцелярские товары": 0.0949,
    "Красота и гигиена": 0.2158, "Мебель": 0.0490,
    "Музыкальные инструменты": 0.1347, "Обувь": 0.0458, "Одежда": 0.0573,
    "Продукты питания": 0.1146, "Спорт и отдых": 0.1115,
    "Строительство и ремонт": 0.1290, "Товары для животных": 0.1451,
    "Хобби и творчество": 0.0974, "Электроника": 0.0647,
    "Ювелирные изделия": 0.0157,
}

STAGE_LIMITS = {"check": 60, "public": 360, "private": 780}


def ensure_dirs() -> None:
    for d in (RAW, PROC):
        d.mkdir(parents=True, exist_ok=True)


In [ ]:
%%writefile src/ecup/__init__.py
from __future__ import annotations

__all__: tuple[str, ...] = ()

__version__ = "2.0.0"


In [ ]:
%%writefile src/ecup/arrowstr.py
from __future__ import annotations

import sys
from pathlib import Path
from typing import Sequence

CONVERT_BATCH = 200_000


class ArrowStrings(Sequence):

    __slots__ = ("_chunks", "_starts", "_n")

    def __init__(self, column):
        import numpy as np

        chunks = list(getattr(column, "chunks", None) or [column])
        self._chunks = [c for c in chunks if len(c) > 0]
        sizes = [len(c) for c in self._chunks]
        self._starts = np.cumsum([0] + sizes[:-1]).astype(np.int64) if sizes \
            else np.zeros(0, dtype=np.int64)
        self._n = int(sum(sizes))

    def __len__(self) -> int:
        return self._n

    def _locate(self, i: int) -> tuple:
        import numpy as np

        if i < 0:
            i += self._n
        if not 0 <= i < self._n:
            raise IndexError(i)
        k = int(np.searchsorted(self._starts, i, side="right")) - 1
        return self._chunks[k], i - int(self._starts[k])

    def __getitem__(self, i):
        if isinstance(i, slice):
            return [self[j] for j in range(*i.indices(self._n))]
        chunk, off = self._locate(int(i))
        v = chunk[off].as_py()
        return "" if v is None else v

    def __iter__(self):
        for c in self._chunks:
            for v in c:
                v = v.as_py()
                yield "" if v is None else v

    def char_lengths(self):
        import numpy as np

        out = np.empty(self._n, dtype=np.int64)
        pos = 0
        for c in self._chunks:
            bufs = c.buffers()
            offsets_buf = bufs[1]
            dtype = np.int64 if c.type in (_large_string_type(),) else np.int32
            off = np.frombuffer(offsets_buf, dtype=dtype,
                                count=len(c) + 1, offset=c.offset * dtype().itemsize)
            out[pos:pos + len(c)] = np.diff(off.astype(np.int64))
            pos += len(c)
        return out


def _large_string_type():
    import pyarrow as pa

    return pa.large_string()


def ipc_path_for(parquet_path: Path) -> Path:
    p = Path(parquet_path)
    return p.with_suffix(".arrow")


def parquet_to_ipc(parquet_path: Path, out: Path | None = None,
                   columns: Sequence[str] | None = None,
                   log=print) -> Path:
    import pyarrow as pa
    import pyarrow.parquet as pq

    src = Path(parquet_path)
    dst = Path(out) if out else ipc_path_for(src)
    if dst.is_file() and dst.stat().st_mtime >= src.stat().st_mtime:
        return dst

    pf = pq.ParquetFile(src)
    cols = list(columns) if columns else pf.schema_arrow.names
    schema = pa.schema([pf.schema_arrow.field(c) for c in cols])
    tmp = dst.with_name(dst.name + ".part")
    n = 0
    with pa.OSFile(str(tmp), "wb") as sink:
        with pa.ipc.new_file(sink, schema) as writer:
            for batch in pf.iter_batches(batch_size=CONVERT_BATCH, columns=cols):
                writer.write_batch(batch)
                n += batch.num_rows
                if n % (CONVERT_BATCH * 10) == 0:
                    log(f"  arrow: {n:,} строк")
    tmp.replace(dst)
    log(f"  arrow: {n:,} строк -> {dst} ({dst.stat().st_size / 2**30:.1f} ГиБ)")
    return dst


def load_texts(parquet_path: Path, columns: Sequence[str] | None = None,
               log=print):
    import pyarrow as pa

    ipc = parquet_to_ipc(parquet_path, columns=columns, log=log)
    source = pa.memory_map(str(ipc), "rb")
    return pa.ipc.open_file(source).read_all()


def dedup_strings(column) -> "object":
    import numpy as np
    import pyarrow.compute as pc

    d = column.dictionary_encode() if not hasattr(column, "dictionary") else column
    d = d.combine_chunks() if hasattr(d, "combine_chunks") else d
    values = [v.as_py() for v in d.dictionary]
    codes = np.asarray(pc.fill_null(d.indices, -1), dtype=np.int64)
    lookup = np.empty(len(values) + 1, dtype=object)
    lookup[:len(values)] = values
    lookup[len(values)] = ""
    codes[codes < 0] = len(values)
    return lookup[codes]


def anon_bytes() -> int:
    for p in ("/sys/fs/cgroup/memory.stat",
              "/sys/fs/cgroup/memory/memory.stat"):
        try:
            for line in Path(p).read_text().splitlines():
                if line.startswith("anon ") or line.startswith("rss "):
                    return int(line.split()[1])
        except OSError:
            continue
    return 0


def limit_bytes() -> int:
    for p in ("/sys/fs/cgroup/memory.max",
              "/sys/fs/cgroup/memory/memory.limit_in_bytes"):
        try:
            v = Path(p).read_text().strip()
            if v and v != "max":
                return int(v)
        except OSError:
            continue
    return 0


def memory_line() -> str:
    a, lim = anon_bytes(), limit_bytes()
    if not a:
        return ""
    g = a / 2**30
    return f"anon {g:.1f} ГиБ" + (f" из {lim / 2**30:.0f}" if lim else "")


if __name__ == "__main__":
    for arg in sys.argv[1:]:
        parquet_to_ipc(Path(arg))


In [ ]:
%%writefile src/ecup/backbones.py
from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True)
class Backbone:

    hf_id: str
    alias: str
    max_len: int = 320
    batch: int = 48
    lr: float = 2e-5
    max_position: int = 512
    throughput_hint: int = 600
    container_verified: bool = False
    note: str = ""
    tags: tuple[str, ...] = field(default_factory=tuple)


REGISTRY: dict[str, Backbone] = {
    "rmb-base": Backbone(
        hf_id="deepvk/RuModernBERT-base", alias="rmb-base",
        max_len=320, batch=48, lr=2e-5, max_position=8192,
        throughput_hint=620, container_verified=True,
        note="Первая база, на которой архив реально отработал в стенде: k16, "
             "две эпохи стадии B поверх предобучения на LLM-метках, доска "
             "0.4452. Дальше её сменил mmBERT — та же архитектура, но "
             "многоязычное обучение и вчетверо больший словарь.",
        tags=("modernbert", "ru")),
    "mmbert": Backbone(
        hf_id="jhu-clsp/mmBERT-base", alias="mmbert",
        max_len=576, batch=48, lr=2e-5, max_position=8192,
        throughput_hint=600,
        note="База половины 1. Архитектура ТА ЖЕ, что у `rmb-base` "
             "(ModernBERT, 22 слоя на 768 при промежуточном 1152, глобальное "
             "внимание каждый третий слой), поэтому код и образ её тянут "
             "без единой правки — в отличие от EuroBERT или gte, которым нужен "
             "trust_remote_code, а качать код модели в контейнере без сети "
             "нечем. Непохожесть даёт не архитектура, а ОБУЧЕНИЕ: 3 триллиона "
             "токенов на 1833 языках против 2 триллионов русскоцентричных, и "
             "словарь Gemma-2 на 256 тысяч вместо русского BPE на 50 тысяч. "
             "Предел позиций 8192, то есть длину 576 от k48 берёт целиком. "
             "Эмбеддинги на 196 млн параметров утяжеляют файл вдвое, но на "
             "скорость почти не влияют: поиск по таблице дёшев.",
        tags=("modernbert", "multilingual")),
    "bge-rerank": Backbone(
        hf_id="BAAI/bge-reranker-base", alias="bge-rerank",
        max_len=448, batch=64, lr=2e-5, max_position=512,
        throughput_hint=780,
        note="Кандидат в смесь РАДИ НЕПОХОЖЕСТИ, а не силы; половина 2 в "
             "итоге взяла старшую версию этой же базы, bge-reranker-v2-m3. "
             "Уже обучен как кросс-энкодер релевантности, основа XLM-R base: "
             "12 слоёв на 768 при промежуточном 3072 без гейта — примерно "
             "0.78 от стоимости `rmb-base`. Предел позиций 512, то есть "
             "длину 576 от k48 он не возьмёт; сколько именно занимает этот "
             "текст в ЕГО токенизаторе, надо мерить: словарь у XLM-R на "
             "250 тысяч и на русском заметно менее экономный, чем русский "
             "BPE `rmb-base`. Линия BGE берёт Канцелярские товары на +0.068 "
             "выше линии ModernBERT и Дом и сад на +0.039 — слепые пятна у "
             "моделей разные, на этом и живёт смесь.",
        tags=("xlm-r", "multilingual", "reranker")),
    "rmb-small": Backbone(
        hf_id="deepvk/RuModernBERT-small", alias="rmb-small",
        max_len=320, batch=96, lr=3e-5, max_position=8192,
        throughput_hint=1700,
        note="Младший брат `rmb-base`: та же архитектура и токенизатор, "
             "значит образ её тянет с той же вероятностью, а стоит втрое "
             "дешевле. Кандидат в смесь на случай, когда бюджет времени не "
             "даёт взять ещё одну base.",
        tags=("modernbert", "ru", "fast")),
    "berta": Backbone(
        hf_id="sergeyzh/BERTA", alias="berta",
        max_len=256, batch=64, lr=2e-5, max_position=512,
        throughput_hint=700,
        note="Другая архитектура (BERT-семейство) — главный источник "
             "разнообразия к ModernBERT. Контейнер НЕ проверен.",
        tags=("bert", "ru")),
    "rubert-tiny2": Backbone(
        hf_id="cointegrated/rubert-tiny2", alias="rubert-tiny2",
        max_len=256, batch=256, lr=5e-5, max_position=2048,
        throughput_hint=6000,
        note="Очень дёшево. В одиночку слаб, но в смеси по рангам дешёвая "
             "и слабо коррелированная модель часто добавляет третий знак "
             "почти бесплатно по времени.",
        tags=("bert", "ru", "fast")),
    "rubert-base": Backbone(
        hf_id="ai-forever/ruBert-base", alias="rubert-base",
        max_len=256, batch=64, lr=2e-5, max_position=512,
        throughput_hint=700,
        note="Классический русский BERT. Контейнер НЕ проверен.",
        tags=("bert", "ru")),
    "e5-base": Backbone(
        hf_id="intfloat/multilingual-e5-base", alias="e5-base",
        max_len=256, batch=64, lr=2e-5, max_position=512,
        throughput_hint=650,
        note="Многоязычный XLM-R. Другой токенизатор — значит и другие ошибки "
             "на латинице и кодах моделей, что для смеси полезно.",
        tags=("xlmr", "multi")),
}

_BY_HF: dict[str, str] = {b.hf_id: k for k, b in REGISTRY.items()}


def get(name: str) -> Backbone | None:
    if name in REGISTRY:
        return REGISTRY[name]
    if name in _BY_HF:
        return REGISTRY[_BY_HF[name]]
    return None


def resolve(name: str) -> str:
    b = get(name)
    return b.hf_id if b else name


def describe(name: str) -> str:
    b = get(name)
    if b is None:
        return (f"{name}: в реестре нет — гиперпараметры берутся из флагов, "
                f"поведение в контейнере НЕ проверено")
    v = "проверен стендом" if b.container_verified else "контейнер НЕ проверен"
    return (f"{b.alias} ({b.hf_id}): max_len {b.max_len}, batch {b.batch}, "
            f"lr {b.lr:g}, ~{b.throughput_hint} пар/с, {v}")


def table() -> str:
    rows = ["| алиас | Hugging Face | max_len | batch | lr | ~пар/с | контейнер |",
            "|---|---|---|---|---|---|---|"]
    for b in REGISTRY.values():
        rows.append(f"| `{b.alias}` | `{b.hf_id}` | {b.max_len} | {b.batch} | "
                    f"{b.lr:g} | {b.throughput_hint} | "
                    f"{'проверен' if b.container_verified else '**не проверен**'} |")
    return "\n".join(rows)


In [ ]:
%%writefile src/ecup/bucket.py
from __future__ import annotations

from typing import Iterator, Sequence

DEFAULT_WINDOW_BATCHES = 64


def pair_lengths(text_a: Sequence[str], text_b: Sequence[str]):
    import numpy as np

    if len(text_a) != len(text_b):
        raise ValueError(f"стороны пары разной длины: {len(text_a)} и {len(text_b)}")
    if hasattr(text_a, "char_lengths") and hasattr(text_b, "char_lengths"):
        s = text_a.char_lengths() + text_b.char_lengths()
        return np.minimum(s, np.iinfo(np.int32).max).astype(np.int32)
    return np.fromiter(
        (len(a) + len(b) for a, b in zip(text_a, text_b)),
        dtype=np.int32, count=len(text_a))


def bucketed_batches(lengths, batch_size: int, seed: int,
                     window_batches: int = DEFAULT_WINDOW_BATCHES,
                     drop_last: bool = True,
                     rank: int = 0, world_size: int = 1) -> list:
    import numpy as np

    n = len(lengths)
    if batch_size < 1:
        raise ValueError(f"batch_size={batch_size} должен быть положительным")
    if window_batches < 1:
        raise ValueError(f"window_batches={window_batches} должен быть положительным")

    rng = np.random.default_rng(seed)
    order = rng.permutation(n)
    lengths = np.asarray(lengths)
    window = batch_size * window_batches

    batches: list[list[int]] = []
    tail: list[int] = []
    for start in range(0, n, window):
        chunk = order[start:start + window]
        chunk = chunk[np.lexsort((chunk, lengths[chunk]))]
        full = (len(chunk) // batch_size) * batch_size
        for b in range(0, full, batch_size):
            batches.append(chunk[b:b + batch_size].tolist())
        tail.extend(chunk[full:].tolist())

    if tail:
        tail_arr = np.asarray(tail, dtype=np.int64)
        tail_arr = tail_arr[np.lexsort((tail_arr, lengths[tail_arr]))]
        full = (len(tail_arr) // batch_size) * batch_size
        for b in range(0, full, batch_size):
            batches.append(tail_arr[b:b + batch_size].tolist())
        if not drop_last and full < len(tail_arr):
            batches.append(tail_arr[full:].tolist())

    rng.shuffle(batches)

    if world_size > 1:
        n = (len(batches) // world_size) * world_size
        batches = batches[rank:n:world_size]
    return batches


def padding_share(lengths, batches) -> float:
    import numpy as np

    lengths = np.asarray(lengths)
    useful = 0
    total = 0
    for b in batches:
        ls = lengths[np.asarray(b, dtype=np.int64)]
        useful += int(ls.sum())
        total += int(ls.max()) * len(b)
    if total == 0:
        raise ValueError("пустая раскладка батчей")
    return 1.0 - useful / total


def build_sampler_class():
    import torch

    class BucketBatchSampler(torch.utils.data.Sampler):

        def __init__(self, lengths, batch_size: int, seed: int,
                     window_batches: int = DEFAULT_WINDOW_BATCHES,
                     drop_last: bool = True, rank: int = 0, world_size: int = 1):
            self.lengths = lengths
            self.batch_size = batch_size
            self.seed = seed
            self.window_batches = window_batches
            self.drop_last = drop_last
            self.rank = rank
            self.world_size = world_size
            self.epoch = 0
            self._batches = self._make()

        def _make(self) -> list:
            return bucketed_batches(
                self.lengths, self.batch_size, self.seed + self.epoch,
                window_batches=self.window_batches, drop_last=self.drop_last,
                rank=self.rank, world_size=self.world_size)

        def set_epoch(self, epoch: int) -> None:
            self.epoch = epoch
            self._batches = self._make()

        def __iter__(self) -> Iterator[list[int]]:
            return iter(self._batches)

        def __len__(self) -> int:
            return len(self._batches)

    return BucketBatchSampler


In [ ]:
%%writefile src/ecup/config.py
from __future__ import annotations

import json
import os
from dataclasses import dataclass, field, fields
from pathlib import Path

ROOT = Path(os.environ.get("ECUP_ROOT", Path(__file__).resolve().parents[2]))
DATA = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"

SERIALIZE_CONFIG_NAME = "serialize_config.json"

RENDER_ALGO_VERSION = "v2"


def text_format_version() -> str:
    from .textprep import dictionaries_digest
    return f"{RENDER_ALGO_VERSION}+{dictionaries_digest()}"


@dataclass(frozen=True)
class SerializeConfig:

    canon: bool = True
    max_keys: int = 16
    max_val_chars: int = 60
    version: str = field(default_factory=text_format_version)
    missing_key_idf: float = 1.0

    def __post_init__(self) -> None:
        if self.max_keys < 0:
            raise ValueError(f"max_keys должен быть неотрицательным, получено {self.max_keys}")
        if self.max_val_chars <= 0:
            raise ValueError(
                f"max_val_chars должен быть положительным, получено {self.max_val_chars}")
        if not isinstance(self.canon, bool):
            raise TypeError(f"canon должен быть bool, получено {type(self.canon).__name__}")

    def to_dict(self) -> dict:
        return {f.name: getattr(self, f.name) for f in fields(self)}

    @classmethod
    def from_dict(cls, d: dict) -> "SerializeConfig":
        known = {f.name for f in fields(cls)}
        if "version" not in d:
            raise ValueError(
                "в конфигурации сериализации нет поля version: неизвестно, каким "
                "алгоритмом и какими словарями ключей отрендерен обучающий текст. "
                f"Текущий код соответствует version={text_format_version()!r}; "
                "впишите его в файл, только если точно знаете, что чекпойнт "
                "обучен этим же кодом")
        unknown = sorted(set(d) - known)
        if unknown:
            raise ValueError(
                f"в конфигурации сериализации неизвестные поля {unknown}; "
                f"этот код знает только {sorted(known)}. Обновите src/ecup/config.py, "
                f"молча игнорировать нельзя — изменится текст пары")
        return cls(**{k: v for k, v in d.items() if k in known})

    def save(self, dir_path) -> Path:
        p = Path(dir_path)
        p.mkdir(parents=True, exist_ok=True)
        out = p / SERIALIZE_CONFIG_NAME
        out.write_text(
            json.dumps(self.to_dict(), ensure_ascii=False, indent=2, sort_keys=True) + "\n",
            encoding="utf-8")
        return out

    @classmethod
    def load(cls, dir_path) -> "SerializeConfig":
        p = Path(dir_path) / SERIALIZE_CONFIG_NAME
        if not p.is_file():
            raise FileNotFoundError(
                f"нет {p}: неизвестно, на каком тексте обучен этот чекпойнт. "
                f"Подставлять значения по умолчанию запрещено — именно так "
                f"обучение и инференс разъезжаются молча")
        return cls.from_dict(json.loads(p.read_text(encoding="utf-8")))


In [ ]:
%%writefile src/ecup/folds.py
from __future__ import annotations

from collections import defaultdict

import numpy as np

from .metric import macro_pr_auc
from .textprep import WORD_RE, normalize

LB_SCALE = 1.670

LB_INTERCEPT = 0.1561
LB_SLOPE = 0.6001

LB_POINTS: dict[str, tuple[float, float]] = {
    "jaccard_tokens":      (0.3579, 0.2141960717597015),
    "ce_rumodernbert_k10": (0.7321, 0.4343651883290507),
    "ce_rumodernbert_k16": (0.7358, 0.4452475828806686),
}


def first_token(name) -> str:
    t = WORD_RE.findall(normalize("" if name is None else str(name)))
    return t[0] if t else ""


def first_tokens(names) -> list[str]:
    return [first_token(n) for n in names]


def make_unseen_folds(category, first_tok, n_folds: int = 5) -> np.ndarray:
    category = list(category)
    first_tok = list(first_tok)
    if len(category) != len(first_tok):
        raise ValueError(
            f"длины не совпадают: категорий {len(category)}, токенов {len(first_tok)}")

    by_cat: dict = defaultdict(lambda: defaultdict(list))
    for i, (c, b) in enumerate(zip(category, first_tok)):
        by_cat[c][b].append(i)

    fold = np.full(len(category), -1, dtype=np.int8)
    for _, groups in by_cat.items():
        load = np.zeros(n_folds, dtype=np.int64)
        for key, idxs in sorted(groups.items(), key=lambda kv: (-len(kv[1]), str(kv[0]))):
            f = int(np.argmin(load))
            fold[idxs] = f
            load[f] += len(idxs)

    if (fold < 0).any():
        raise ValueError("остались непроставленные фолды")
    return fold


def make_brand_holdout(brand, llm_count, target_pairs: int = 25_000,
                       seed: int = 20260820) -> np.ndarray:
    b = np.asarray(brand, dtype=object)
    uniq, first = np.unique(b, return_index=True)
    counts = np.array([llm_count.get(x, 0) for x in uniq])
    n_pairs = len(b)

    edges = [0, 1, 10, 100, 1000, 1 << 62]
    rng = np.random.default_rng(seed)
    share = target_pairs / max(n_pairs, 1)
    chosen: list = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        idx = np.flatnonzero((counts >= lo) & (counts < hi))
        if not len(idx):
            continue
        k = max(1, int(round(len(idx) * share)))
        chosen.extend(uniq[rng.choice(idx, size=min(k, len(idx)), replace=False)])
    return np.isin(b, np.asarray(chosen, dtype=object))


def make_component_folds(id1, id2, category, n_folds: int = 5,
                         seed: int = 42) -> np.ndarray:
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components

    id1 = np.asarray(id1)
    id2 = np.asarray(id2)
    category = np.asarray(category).astype(str)
    all_ids, inv = np.unique(np.concatenate([id1, id2]), return_inverse=True)
    ia, ib = inv[:len(id1)], inv[len(id1):]
    g = coo_matrix((np.ones(len(ia), dtype=np.int8), (ia, ib)),
                   shape=(len(all_ids), len(all_ids)))
    _, labels = connected_components(g, directed=False)
    comp = labels[ia]

    rng = np.random.default_rng(seed)
    out = np.full(len(comp), -1, dtype=np.int8)
    for c in np.unique(category):
        m = category == c
        comps, sizes = np.unique(comp[m], return_counts=True)
        o = rng.permutation(len(comps))
        comps, sizes = comps[o], sizes[o]
        o = np.argsort(-sizes, kind="stable")
        comps, sizes = comps[o], sizes[o]
        load = np.zeros(n_folds, dtype=np.int64)
        where: dict[int, int] = {}
        for cc, sz in zip(comps.tolist(), sizes.tolist()):
            f = int(np.argmin(load))
            where[cc] = f
            load[f] += sz
        idx = np.flatnonzero(m)
        out[idx] = [where[x] for x in comp[idx].tolist()]
    if (out < 0).any():
        raise ValueError("остались непроставленные фолды")
    return out


JACCARD_FOLD0 = {"components": 0.3545, "unseen_brands": 0.3433}


def comparable_ratio(macro_ap: float, split: str = "unseen_brands") -> float:
    if split not in JACCARD_FOLD0:
        raise ValueError(f"неизвестная схема {split!r}; есть {sorted(JACCARD_FOLD0)}")
    return macro_ap / JACCARD_FOLD0[split]


def random_level(y, category) -> float:
    y = np.asarray(y)
    category = np.asarray(category).astype(str)
    shares = [y[category == c].mean() for c in np.unique(category)]
    shares = [d for d in shares if 0 < d < 1]
    if not shares:
        raise ValueError("ни одной категории с обоими классами")
    return float(np.mean(shares))


def lb_from_val(val: float, random: float) -> float:
    return LB_INTERCEPT + LB_SLOPE * (val - random)


def lb_of(y, score, category) -> float:
    return lb_from_val(macro_pr_auc(y, score, category), random_level(y, category))


def expected_lb(y, score, category, scale: float = LB_SCALE,
                i_know_it_lies: bool = False) -> float:
    if not i_know_it_lies:
        raise ValueError(
            "expected_lb годится только для прикидки порядка величины: множитель "
            "подобран одним параметром по трём точкам и на приростах ошибается "
            "впятеро (локальная дельта +0.0036 предсказывала +0.0022, реально "
            "было +0.0109). Для сравнения моделей используйте "
            "src/scripts/36_clean_metric.py. Если всё же нужен порядок величины — "
            "передайте i_know_it_lies=True")
    return macro_pr_auc(y, score, category) / scale


def delta_consistency(points: dict | None = None) -> list[dict]:
    pts = points or LB_POINTS
    names = sorted(pts)
    scale, _ = refit_scale(pts)
    out = []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            d_loc = pts[b][0] - pts[a][0]
            d_lb = pts[b][1] - pts[a][1]
            pred = d_loc / scale
            out.append({
                "пара": f"{a} -> {b}",
                "дельта_локально": round(d_loc, 4),
                "дельта_предсказано": round(pred, 4),
                "дельта_на_ЛБ": round(d_lb, 4),
                "во_сколько_раз_врёт": round(d_lb / pred, 2) if pred else float("inf"),
            })
    return out


def bootstrap_lb(y, score, category, fold: int, frac: float = 0.30,
                 n: int = 300, seed: int = 0):
    from . import lb

    y = np.asarray(y)
    score = np.asarray(score)
    category = np.asarray(category)
    rng = np.random.default_rng(seed)
    take = min(max(int(len(y) * frac), 1000), len(y))
    v = []
    for _ in range(n):
        idx = rng.choice(len(y), take, replace=False)
        ap = macro_pr_auc(y[idx], score[idx], category[idx])
        v.append(lb.expected(ap, fold))
    a = np.asarray(v)
    return float(a.mean()), float(np.percentile(a, 5)), float(np.percentile(a, 95))


def refit_scale(points: dict | None = None) -> tuple[float, float]:
    pts = points or LB_POINTS
    if not pts:
        raise ValueError("нет ни одной калибровочной точки")
    r = np.array([loc / lb for loc, lb in pts.values()])
    return float(r.mean()), float(100 * r.std() / r.mean())


In [ ]:
%%writefile src/ecup/infer.py
from __future__ import annotations

import time as _t
from pathlib import Path

import numpy as np


def _per_model(value, md: Path, what: str) -> int:
    if isinstance(value, dict):
        v = value.get(md)
        if v is None:
            raise KeyError(
                f"для {md.name} не задан {what}; заданы: "
                f"{sorted(p.name for p in value)}. Подставлять умолчание нельзя: "
                f"обрезка меняет то, что видит модель, а ошибки бы не было")
        return int(v)
    return int(value)


def score_pairs(model_dirs: list[Path], text_a: list[str], text_b: list[str],
                max_len: int | dict = 256, batch: int | dict = 256, log=print,
                deadline: float | None = None) -> list[tuple[Path, np.ndarray]]:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    n = len(text_a)
    if len(text_b) != n:
        raise ValueError(f"стороны пар разной длины: {n} против {len(text_b)}")

    oom_types = tuple({t for t in (getattr(torch, "OutOfMemoryError", None),
                                   getattr(torch.cuda, "OutOfMemoryError", None))
                       if t is not None}) or (RuntimeError,)

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    outs: list[tuple[Path, np.ndarray]] = []
    order = None

    for md in model_dirs:
        if deadline is not None and _t.time() > deadline:
            log(f"watchdog: пропускаю {md.name}, время вышло")
            break
        t0 = _t.time()
        md_len = _per_model(max_len, md, "max_len")
        md_batch = _per_model(batch, md, "batch")
        tok = AutoTokenizer.from_pretrained(str(md))
        if order is None:
            tl = np.fromiter((len(text_a[i]) + len(text_b[i]) for i in range(n)),
                             dtype=np.int32, count=n)
            order = np.argsort(tl, kind="stable")
            log(f"длины в символах: p50={np.percentile(tl,50):.0f} "
                f"p95={np.percentile(tl,95):.0f} max={tl.max()} "
                f"(ключ сортировки за {_t.time()-t0:.1f}s)")
        model = AutoModelForSequenceClassification.from_pretrained(
            str(md), dtype=torch.bfloat16, attn_implementation="sdpa").eval()
        try:
            model = model.to(dev)
        except RuntimeError as e:
            log(f"{md.name}: на {dev} не влез ({type(e).__name__}: {e}), "
                f"МОДЕЛЬ ПРОПУЩЕНА и в смесь не войдёт")
            del model
            if dev == "cuda":
                torch.cuda.empty_cache()
            continue
        res = np.zeros(n, dtype=np.float32)
        cut = False

        def score_chunk(sub, model=model, tok=tok, res=res) -> bool:
            try:
                enc = tok([text_a[i] for i in sub], [text_b[i] for i in sub],
                          padding=True, truncation=True, max_length=md_len,
                          return_tensors="pt")
                enc = {k: v.to(dev, non_blocking=True) for k, v in enc.items()}
                res[sub] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
                return True
            except oom_types:
                if dev == "cuda":
                    torch.cuda.empty_cache()
                if len(sub) <= 1:
                    return False
                half = len(sub) // 2
                return score_chunk(sub[:half]) and score_chunk(sub[half:])

        with torch.inference_mode():
            for s in range(0, n, md_batch):
                sl = order[s:s + md_batch]
                if not score_chunk(sl):
                    log(f"{md.name}: не хватает памяти даже на одну пару "
                        f"(позиция {s:,}/{n:,}) — результат ОТБРАСЫВАЮ целиком: "
                        f"недосчитанная модель испортила бы смесь")
                    cut = True
                    break
                if deadline is not None and _t.time() > deadline:
                    done = s + len(sl)
                    if outs:
                        log(f"watchdog: {md.name} оборван на {done:,}/{n:,} "
                            f"({done/n:.1%}) — ОТБРАСЫВАЮ: {len(outs)} модель(и) "
                            f"уже досчитаны, смесь обойдётся без неё")
                        cut = True
                        break
                    if done == 0:
                        log(f"watchdog: {md.name} оборван, не посчитано ни одной "
                            f"пары — сохранять нечего")
                        cut = True
                        break
                    tail = order[done:]
                    res[tail] = res[order[:done]].min() - 1.0
                    log(f"watchdog: {md.name} оборван на {done:,}/{n:,} "
                        f"({done/n:.1%}) — результат СОХРАНЯЮ: других моделей "
                        f"нет, а частичный ответ лучше отсутствия файла. "
                        f"Остаток {len(tail):,} пар уходит в конец ранжирования")
                    break
        if not cut:
            outs.append((md, res))
            log(f"{md.name}: {_t.time()-t0:.1f}s ({n/max(_t.time()-t0,1e-9):,.0f} пар/с, "
                f"max_len={md_len}, batch={md_batch})")
        del model
        if dev == "cuda":
            torch.cuda.empty_cache()
    return outs


def rank_within(scores: np.ndarray, category: np.ndarray) -> np.ndarray:
    category = np.asarray(category)
    if category.dtype == object:
        category = category.astype(str)
    out = np.zeros(len(scores), dtype=np.float64)
    for c in np.unique(category):
        k = np.where(category == c)[0]
        s = scores[k]
        r = np.empty(len(s), dtype=np.float64)
        r[np.argsort(s, kind="stable")] = np.arange(len(s), dtype=np.float64)
        out[k] = r / max(len(s) - 1, 1)
    return out


def blend(score_list: list[np.ndarray], weights: list[float],
          category: np.ndarray) -> np.ndarray:
    if len(score_list) != len(weights):
        raise ValueError(
            f"число моделей и весов не совпало: {len(score_list)} против {len(weights)}")
    tot = float(sum(weights))
    if tot <= 0:
        raise ValueError(f"сумма весов смеси должна быть положительной, получено {tot}")
    acc = np.zeros(len(score_list[0]), dtype=np.float64)
    for s, w in zip(score_list, weights):
        acc += (w / tot) * rank_within(s, category)
    return acc


In [ ]:
%%writefile src/ecup/lb.py
from __future__ import annotations

import numpy as np

JACCARD_LB = 0.2141960717597015

LB_POINTS: list[tuple[str, float, float, float]] = [
    ("v7-ce-only",  0.7321,    0.3433, 0.4343651883290507),
    ("v8-hum-f1",   0.7552,    0.3583, 0.4119489596),
    ("v9-k16",      0.7358,    0.3433, 0.4452475828806686),
    ("v10-pre-f1",  0.7647,    0.3583, 0.4327306009),
    ("v11-prefull", 0.7471272, 0.3433, 0.4801),
]

SLOPE = 1.0061
INTERCEPT = -1.7110

JACCARD_UNSEEN_FOLDS: dict[int, float] = {
    0: 0.3433, 1: 0.3583, 2: 0.3569, 3: 0.3649, 4: 0.3654,
}


def expected(macro_ap: float, fold: int) -> float:
    if fold not in JACCARD_UNSEEN_FOLDS:
        raise ValueError(
            f"нет Jaccard для фолда {fold}; есть {sorted(JACCARD_UNSEEN_FOLDS)}. "
            f"Посчитайте Jaccard на этом фолде (`jaccard_scores`) и добавьте "
            f"его сюда")
    return predict(macro_ap, JACCARD_UNSEEN_FOLDS[fold])


def predict(macro_ap: float, macro_ap_jaccard: float,
            slope: float = SLOPE, intercept: float = INTERCEPT) -> float:
    if macro_ap_jaccard <= 0:
        raise ValueError("macro-AP Jaccard должен быть положительным: без него "
                         "трудность среза не измерить")
    return slope * (macro_ap / macro_ap_jaccard) + intercept


def refit(points: list | None = None) -> dict:
    pts = points if points is not None else LB_POINTS
    if len(pts) < 3:
        raise ValueError(f"нужно хотя бы три точки, есть {len(pts)}")
    x = np.array([ap / j for _, ap, j, _ in pts])
    y = np.array([lb for *_, lb in pts])
    slope, intercept = np.polyfit(x, y, 1)

    errs = []
    for i in range(len(pts)):
        m = [j for j in range(len(pts)) if j != i]
        s, b = np.polyfit(x[m], y[m], 1)
        errs.append(s * x[i] + b - y[i])
    errs = np.asarray(errs)
    return {
        "slope": float(slope),
        "intercept": float(intercept),
        "loo_mean_abs": float(np.abs(errs).mean()),
        "loo_max_abs": float(np.abs(errs).max()),
        "residuals": {pts[i][0]: float(slope * x[i] + intercept - y[i])
                      for i in range(len(pts))},
        "n_points": len(pts),
    }


def jaccard_scores(names, p1, p2) -> np.ndarray:
    from .textprep import WORD_RE, normalize

    memo: dict[int, set] = {}

    def tok(i: int) -> set:
        t = memo.get(i)
        if t is None:
            t = set(WORD_RE.findall(normalize(names[i])))
            memo[i] = t
        return t

    l1 = p1.tolist() if hasattr(p1, "tolist") else list(p1)
    l2 = p2.tolist() if hasattr(p2, "tolist") else list(p2)
    out = np.empty(len(l1), dtype=np.float32)
    for k, (a, b) in enumerate(zip(l1, l2)):
        ta, tb = tok(a), tok(b)
        u = len(ta | tb)
        out[k] = len(ta & tb) / u if u else 0.0
    return out


JACCARD_TEST_CAT: dict[str, float] = {
    "Автотовары": 0.2554, "Аптека": 0.2437, "Бытовая техника": 0.3143,
    "Бытовая химия": 0.3826, "Галантерея и аксессуары": 0.0675,
    "Детские товары": 0.3221, "Дом и сад": 0.2591, "Канцелярские товары": 0.1778,
    "Красота и гигиена": 0.3252, "Мебель": 0.1050, "Музыкальные инструменты": 0.3042,
    "Обувь": 0.0922, "Одежда": 0.0697, "Продукты питания": 0.3145,
    "Спорт и отдых": 0.1748, "Строительство и ремонт": 0.2737,
    "Товары для животных": 0.2553, "Хобби и творчество": 0.2117,
    "Электроника": 0.1122, "Ювелирные изделия": 0.0228,
}


In [ ]:
%%writefile src/ecup/losses.py
from __future__ import annotations


def _groups(logits, labels, groups):
    import torch
    if groups is None:
        yield logits, labels
        return
    for g in torch.unique(groups):
        m = groups == g
        y = labels[m]
        if (y > 0.5).any() and (y <= 0.5).any():
            yield logits[m], y


def pairwise_logistic(logits, labels, groups=None):
    import torch
    import torch.nn.functional as F
    parts = []
    for s, y in _groups(logits, labels, groups):
        pos_s, neg_s = s[y > 0.5], s[y <= 0.5]
        parts.append(F.softplus(-(pos_s.unsqueeze(1) - neg_s.unsqueeze(0))).mean())
    if not parts:
        return logits.sum() * 0.0
    return torch.stack(parts).mean()


def smooth_ap(logits, labels, groups=None, tau: float = 0.01):
    import torch
    parts = []
    for s, y in _groups(logits, labels, groups):
        pos_s = y > 0.5
        d = (s.unsqueeze(0) - s.unsqueeze(1)) / tau
        sigm = torch.sigmoid(d)
        sigm = sigm * (1.0 - torch.eye(len(s), device=s.device, dtype=sigm.dtype))
        R_pos = 1.0 + (sigm * pos_s.unsqueeze(0)).sum(1)[pos_s]
        R_all = 1.0 + sigm.sum(1)[pos_s]
        parts.append((R_pos / R_all).mean())
    if not parts:
        return logits.sum() * 0.0
    return 1.0 - torch.stack(parts).mean()


In [ ]:
%%writefile src/ecup/metric.py
from __future__ import annotations

import numpy as np
from sklearn.metrics import average_precision_score


def per_category_ap(y_true, y_score, category) -> dict[str, float]:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    category = np.asarray(category)
    out: dict[str, float] = {}
    for c in np.unique(category):
        m = category == c
        yy = y_true[m]
        if 0 < int(yy.sum()) < int(m.sum()):
            out[str(c)] = float(average_precision_score(yy, y_score[m]))
    return out


def macro_pr_auc(y_true, y_score, category) -> float:
    per = per_category_ap(y_true, y_score, category)
    if not per:
        raise ValueError(
            "ни одной категории с обоими классами: метрику посчитать не на чем. "
            "Проверьте разбиение — вероятно, в срез попали только позитивы "
            "либо только негативы")
    return float(np.mean(list(per.values())))


def bootstrap_ci(y_true, y_score, category, frac: float = 0.30,
                 n_iter: int = 200, seed: int = 0) -> tuple[float, float, float]:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    category = np.asarray(category)
    rng = np.random.default_rng(seed)
    n = len(y_true)
    k = max(int(n * frac), 1)
    vals: list[float] = []
    for _ in range(n_iter):
        idx = rng.choice(n, size=k, replace=False)
        try:
            vals.append(macro_pr_auc(y_true[idx], y_score[idx], category[idx]))
        except ValueError:
            continue
    if not vals:
        raise ValueError("все подвыборки выродились: увеличьте frac или n_iter")
    a = np.asarray(vals)
    return float(a.mean()), float(np.percentile(a, 5)), float(np.percentile(a, 95))


In [ ]:
%%writefile src/ecup/oof.py
from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np

OOF_DIRNAME = "oof"

INDEX_NAME = "_index.npz"


def pairs_digest(id1, id2) -> str:
    h = hashlib.blake2b(digest_size=16)
    a = np.asarray(id1, dtype=np.int64)
    b = np.asarray(id2, dtype=np.int64)
    if len(a) != len(b):
        raise ValueError(f"стороны пар разной длины: {len(a)} против {len(b)}")
    h.update(np.ascontiguousarray(a).tobytes())
    h.update(np.ascontiguousarray(b).tobytes())
    h.update(str(len(a)).encode())
    return h.hexdigest()


@dataclass(frozen=True)
class Index:

    y: np.ndarray
    category: np.ndarray
    fold: np.ndarray
    digest: str

    def __len__(self) -> int:
        return len(self.y)

    @property
    def n_folds(self) -> int:
        return int(self.fold.max()) + 1


def index_path(artifacts: Path) -> Path:
    return Path(artifacts) / OOF_DIRNAME / INDEX_NAME


def save_index(artifacts: Path, y, category, fold, digest: str) -> Path:
    p = index_path(artifacts)
    p.parent.mkdir(parents=True, exist_ok=True)
    y = np.asarray(y, dtype=np.int8)
    category = np.asarray(category).astype(str)
    fold = np.asarray(fold, dtype=np.int8)
    if not (len(y) == len(category) == len(fold)):
        raise ValueError(
            f"длины не совпадают: y={len(y)}, category={len(category)}, fold={len(fold)}")
    if (fold < 0).any():
        raise ValueError("в разбиении есть строки без фолда (-1)")
    np.savez_compressed(p, y=y, category=category, fold=fold,
                        digest=np.array(digest))
    return p


def load_index(artifacts: Path) -> Index:
    p = index_path(artifacts)
    if not p.is_file():
        raise FileNotFoundError(
            f"нет {p}: не построен общий индекс OOF (метки, категории, фолды). "
            f"Без него куски предсказаний нечем выровнять между собой. "
            f"Создайте его: src/scripts/05_build_index.py")
    z = np.load(p, allow_pickle=False)
    return Index(y=z["y"], category=z["category"], fold=z["fold"],
                 digest=str(z["digest"]))


def recipe_dir(artifacts: Path, recipe: str) -> Path:
    if not recipe or "/" in recipe or recipe.startswith("_"):
        raise ValueError(
            f"недопустимое имя рецепта {recipe!r}: нужно непустое имя без «/», "
            f"не начинающееся с подчёркивания (оно занято служебными файлами)")
    return Path(artifacts) / OOF_DIRNAME / recipe


def save_fold(artifacts: Path, recipe: str, fold: int, score, index: Index,
              meta: dict | None = None) -> Path:
    d = recipe_dir(artifacts, recipe)
    d.mkdir(parents=True, exist_ok=True)
    mask = index.fold == fold
    n_fold = int(mask.sum())
    if n_fold == 0:
        raise ValueError(f"в индексе нет ни одной строки с фолдом {fold}")
    score = np.asarray(score, dtype=np.float32)
    if len(score) == len(index):
        score = score[mask]
    elif len(score) != n_fold:
        raise ValueError(
            f"длина предсказаний {len(score)} не равна ни размеру таблицы "
            f"{len(index)}, ни размеру фолда {n_fold}")
    if not np.isfinite(score).all():
        raise ValueError(
            f"в предсказаниях фолда {fold} есть NaN или inf "
            f"({int((~np.isfinite(score)).sum())} шт.). NaN в OOF молча "
            f"выключил бы часть строк из подбора весов")
    np.save(d / f"fold{fold}.npy", score)

    m = dict(meta or {})
    m["pairs_digest"] = index.digest
    mp = d / "meta.json"
    if mp.is_file():
        old = json.loads(mp.read_text(encoding="utf-8"))
        if old.get("pairs_digest") not in (None, index.digest):
            raise ValueError(
                f"у рецепта {recipe} уже лежат предсказания с другим отпечатком "
                f"таблицы пар ({old['pairs_digest']} против {index.digest}). "
                f"Складывать их нельзя: это разный порядок строк. Удалите "
                f"каталог {d} либо пересчитайте старые фолды")
        old.update(m)
        m = old
    mp.write_text(json.dumps(m, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
                  encoding="utf-8")
    return d / f"fold{fold}.npy"


def list_recipes(artifacts: Path) -> list[str]:
    root = Path(artifacts) / OOF_DIRNAME
    if not root.is_dir():
        return []
    return sorted(p.name for p in root.iterdir()
                  if p.is_dir() and not p.name.startswith("_"))


def load(artifacts: Path, recipe: str, index: Index) -> tuple[np.ndarray, dict]:
    d = recipe_dir(artifacts, recipe)
    if not d.is_dir():
        raise FileNotFoundError(
            f"нет каталога {d}; известные рецепты: {list_recipes(artifacts)}")
    mp = d / "meta.json"
    meta = json.loads(mp.read_text(encoding="utf-8")) if mp.is_file() else {}
    got = meta.get("pairs_digest")
    if got is not None and got != index.digest:
        raise ValueError(
            f"рецепт {recipe} посчитан на таблице пар с отпечатком {got}, "
            f"а индекс собран по {index.digest}. Это разный порядок строк: "
            f"смешивать нельзя, метрика получилась бы правдоподобной и неверной")

    out = np.full(len(index), np.nan, dtype=np.float32)
    have: list[int] = []
    for f in range(index.n_folds):
        p = d / f"fold{f}.npy"
        if not p.is_file():
            continue
        s = np.load(p)
        mask = index.fold == f
        if len(s) != int(mask.sum()):
            raise ValueError(
                f"{p}: {len(s)} значений при размере фолда {int(mask.sum())}. "
                f"Скорее всего, разбиение пересчитано после обучения")
        out[mask] = s
        have.append(f)
    if not have:
        raise FileNotFoundError(f"у рецепта {recipe} нет ни одного файла fold*.npy")
    meta = dict(meta)
    meta["folds_present"] = have
    meta["coverage"] = float(np.isfinite(out).mean())
    return out, meta


def common_mask(scores: list[np.ndarray]) -> np.ndarray:
    if not scores:
        raise ValueError("нет ни одного массива предсказаний")
    m = np.isfinite(scores[0])
    for s in scores[1:]:
        if len(s) != len(m):
            raise ValueError(
                f"массивы разной длины: {len(m)} против {len(s)} — "
                f"они посчитаны на разных таблицах пар")
        m &= np.isfinite(s)
    return m


In [ ]:
%%writefile src/ecup/serialize.py
from __future__ import annotations

import math
import sys
import time
from collections import Counter
from typing import Iterable, Sequence

from .config import SerializeConfig, text_format_version
from .textprep import ParsedItem, key_space, parse_item

MAX_UNPARSED_SHARE = 0.20

_WATCH_EVERY = 20_000


class BudgetExceeded(RuntimeError):
    pass


def check_cfg_matches_code(cfg: SerializeConfig) -> None:
    now = text_format_version()
    if cfg.version != now:
        raise ValueError(
            f"формат текста чекпойнта {cfg.version!r} не совпадает с текущим кодом "
            f"{now!r}. Метка состоит из версии алгоритма рендера и отпечатка "
            f"словарей ключей (DROP_KEYS, KEY_SYNONYMS, приоритетные и вариативные "
            f"ключи, таблицы единиц). Значит, эта модель обучена на другом тексте: "
            f"либо возьмите код той версии, либо пересоберите тексты и переобучите. "
            f"Продолжать нельзя — ошибки бы не было, просто упало бы качество")


def select_keys(kv_a: dict[str, str], kv_b: dict[str, str],
                key_idf: dict[str, float] | None,
                cfg: SerializeConfig) -> list[str]:
    space = key_space(cfg.canon)
    all_keys = (set(kv_a) | set(kv_b)) - space.drop
    if not all_keys:
        return []
    idf = key_idf or {}
    missing = cfg.missing_key_idf

    def rank(k: str) -> tuple[int, float, str]:
        if k in space.priority:
            tier = 0
        elif k in kv_a and k in kv_b and kv_a[k] != kv_b[k]:
            tier = 1
        elif (k in kv_a) != (k in kv_b):
            tier = 2
        elif k in space.variant:
            tier = 3
        else:
            tier = 4
        return (tier, -idf.get(k, missing), k)

    return sorted(all_keys, key=rank)[:cfg.max_keys]


def render_side(name_norm: str, kv: dict[str, str], keys: Sequence[str],
                cfg: SerializeConfig) -> str:
    parts = [name_norm]
    for k in keys:
        v = kv.get(k)
        if not v:
            continue
        parts.append(f"{k}={v[:cfg.max_val_chars]}")
    return " | ".join(parts)


def render_pair(a: ParsedItem, b: ParsedItem, category: str,
                key_idf: dict[str, float] | None,
                cfg: SerializeConfig) -> tuple[str, str]:
    kv_a = a.kv(cfg.canon)
    kv_b = b.kv(cfg.canon)
    keys = select_keys(kv_a, kv_b, key_idf, cfg)
    return (f"{category} :: {render_side(a.name_norm, kv_a, keys, cfg)}",
            render_side(b.name_norm, kv_b, keys, cfg))


_FORK_SHARED: dict = {}


def _render_range(bounds: tuple[int, int]) -> tuple[list[str], list[str]]:
    s = _FORK_SHARED
    names, attrs, cats = s["names"], s["attrs"], s["cats"]
    l1, l2, key_idf, cfg = s["l1"], s["l2"], s["key_idf"], s["cfg"]
    lo, hi = bounds
    local: dict[int, ParsedItem] = {}

    def get(q: int) -> ParsedItem:
        it = local.get(q)
        if it is None:
            it = parse_item(names[q], attrs[q])
            local[q] = it
        return it

    ta: list[str] = []
    tb: list[str] = []
    for r in range(lo, hi):
        x, z = render_pair(get(l1[r]), get(l2[r]), cats[l1[r]], key_idf, cfg)
        ta.append(x)
        tb.append(z)
    return ta, tb


def _serialize_parallel(names, attrs, cats, l1, l2, key_idf, cfg,
                        workers: int, log) -> tuple[list[str], list[str]] | None:
    import multiprocessing as mp

    n = len(l1)
    try:
        ctx = mp.get_context("fork")
    except ValueError as e:
        log(f"  параллельная сериализация недоступна ({type(e).__name__}: {e}), "
            f"иду в один процесс")
        return None

    torch_threads = None
    torch_mod = sys.modules.get("torch")
    if torch_mod is not None:
        try:
            torch_threads = torch_mod.get_num_threads()
            torch_mod.set_num_threads(1)
        except Exception as e:
            log(f"  не удалось прижать потоки torch ({type(e).__name__}); "
                f"параллельная сериализация всё равно идёт")
            torch_threads = None

    _FORK_SHARED.update(names=names, attrs=attrs, cats=cats,
                        l1=l1, l2=l2, key_idf=key_idf, cfg=cfg)
    n_chunks = max(workers * 4, 1)
    step = (n + n_chunks - 1) // n_chunks
    bounds = [(i, min(i + step, n)) for i in range(0, n, step)]
    try:
        with ctx.Pool(processes=workers) as pool:
            parts = pool.map(_render_range, bounds)
    except OSError as e:
        log(f"  пул процессов не поднялся ({type(e).__name__}: {e}) — "
            f"иду в один процесс. Это влияет только на время, не на результат")
        return None
    finally:
        _FORK_SHARED.clear()
        if torch_threads is not None:
            torch_mod.set_num_threads(torch_threads)

    ta: list[str] = []
    tb: list[str] = []
    for x, z in parts:
        ta.extend(x)
        tb.extend(z)
    if len(ta) != n:
        raise RuntimeError(
            f"параллельная сериализация вернула {len(ta):,} пар вместо {n:,}")
    return ta, tb


def serialize_pairs(names: Sequence, attrs: Sequence, cats: Sequence,
                    p1, p2, key_idf: dict[str, float] | None,
                    cfg: SerializeConfig, log=print,
                    deadline: float | None = None,
                    parsed: dict | None = None,
                    release_raw: bool = False,
                    workers: int = 1) -> tuple[list[str], list[str]]:
    t0 = time.time()
    check_cfg_matches_code(cfg)
    l1 = p1.tolist() if hasattr(p1, "tolist") else list(p1)
    l2 = p2.tolist() if hasattr(p2, "tolist") else list(p2)
    if len(l1) != len(l2):
        raise ValueError(f"длины сторон пар не совпадают: {len(l1)} против {len(l2)}")

    if workers > 1 and parsed is None and l1:
        log(f"  сериализация в {workers} процессов ({len(l1):,} пар)")
        got = _serialize_parallel(names, attrs, cats, l1, l2, key_idf, cfg,
                                  workers, log)
        if got is not None:
            log(f"  пар отрендерено: {len(got[0]):,} за {time.time() - t0:.1f}s "
                f"(в {workers} процессов)")
            return got

    if parsed is None:
        parsed = {}
    need = sorted(set(l1) | set(l2))
    failed = 0
    for i, q in enumerate(need):
        it = parsed.get(q)
        if it is None:
            it = parse_item(names[q], attrs[q])
            if cfg.canon and release_raw:
                it.release_raw()
            parsed[q] = it
        failed += it.attrs_failed
        if i and i % _WATCH_EVERY == 0 and deadline is not None \
                and time.time() > deadline:
            raise BudgetExceeded(
                f"бюджет исчерпан на разборе товаров: {i:,} из {len(need):,} "
                f"за {time.time() - t0:.0f}s. Дальше процесс всё равно убили бы "
                f"снаружи по лимиту, но уже без traceback и кода возврата")
        if i and i % 200_000 == 0:
            log(f"  разобрано товаров {i:,}/{len(need):,} ({time.time() - t0:.0f}s)")
    log(f"  товаров разобрано: {len(need):,} за {time.time() - t0:.1f}s "
        f"(ветка: {'канонические' if cfg.canon else 'сырые'} ключи, "
        f"max_keys={cfg.max_keys}, атрибуты не разобраны у {failed:,})")
    if need and failed / len(need) > MAX_UNPARSED_SHARE:
        raise RuntimeError(
            f"атрибуты не разобрались у {failed:,} товаров из {len(need):,} "
            f"({100 * failed / len(need):.1f}% при пороге "
            f"{100 * MAX_UNPARSED_SHARE:.0f}%). Скорее всего, колонка attributes "
            f"приехала в другом формате (dict от pyarrow, список записей, "
            f"двойная сериализация). Текст пары выродился бы в одно имя, "
            f"а скор — до уровня имени, без единого признака аварии")

    ta: list[str] = []
    tb: list[str] = []
    t1 = time.time()
    for r in range(len(l1)):
        x, z = render_pair(parsed[l1[r]], parsed[l2[r]], cats[l1[r]], key_idf, cfg)
        ta.append(x)
        tb.append(z)
        if r and r % _WATCH_EVERY == 0 and deadline is not None \
                and time.time() > deadline:
            raise BudgetExceeded(
                f"бюджет исчерпан на рендере пар: {r:,} из {len(l1):,} "
                f"за {time.time() - t1:.0f}s")
    log(f"  пар отрендерено: {len(ta):,} за {time.time() - t1:.1f}s "
        f"(всего {time.time() - t0:.1f}s)")
    return ta, tb


def key_idf_from_parsed(items: Iterable[ParsedItem], cfg: SerializeConfig,
                        min_df: int = 5) -> dict[str, float]:
    df: Counter[str] = Counter()
    n = 0
    for it in items:
        n += 1
        df.update(it.kv(cfg.canon).keys())
    n = max(n, 1)
    return {k: math.log(n / c) for k, c in df.items() if c >= min_df}


def suggested_missing_idf(key_idf: dict[str, float]) -> float:
    return max(key_idf.values()) if key_idf else 1.0


In [ ]:
%%writefile src/ecup/solve.py
from __future__ import annotations

import json
import os
import time as _t
from pathlib import Path

import numpy as np

from .config import SERIALIZE_CONFIG_NAME, SerializeConfig
from .infer import blend, rank_within, score_pairs
from .serialize import BudgetExceeded, check_cfg_matches_code, serialize_pairs

TRAIN_META_NAME = "train_meta.json"


def stage_budget(n_pairs: int) -> float:
    override = os.environ.get("ECUP_BUDGET_S")
    if override:
        return float(override)
    if n_pairs <= 5_000:
        return 50.0
    if n_pairs <= 150_000:
        return 320.0
    return 730.0


def _load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))


def _checkpoint_config(model_dir: Path, log) -> SerializeConfig:
    cfg = SerializeConfig.load(model_dir)
    check_cfg_matches_code(cfg)
    log(f"{model_dir.name}: сериализация из чекпойнта {cfg.to_dict()}")
    return cfg


def _checkpoint_max_len(model_dir: Path) -> int | None:
    p = model_dir / TRAIN_META_NAME
    if not p.is_file():
        return None
    v = _load_json(p).get("max_len")
    return int(v) if v is not None else None


def _key_idf_source(model_dir: Path, art: Path, cfg: SerializeConfig, log) -> Path:
    local = model_dir / "key_idf.json"
    if local.is_file():
        return local
    shared = art / "key_idf.json"
    if shared.is_file() and not cfg.canon:
        log(f"{model_dir.name}: key_idf из общего {shared}")
        return shared
    raise RuntimeError(
        f"для {model_dir.name} нет key_idf.json в каталоге чекпойнта"
        + (f", а общий {shared} для канонической ветки не подходит: его "
           f"пространство ключей не проверить" if cfg.canon and shared.is_file()
           else f" (общего {shared} тоже нет)")
        + ". Без словаря все ключи получают IDF по умолчанию, отбор атрибутов "
          "вырождается в один ярус и текст расходится с обучающим — ошибки бы "
          "не было, просто упало бы качество. Собирайте архив "
          "src/scripts/40_build_submission.py: он кладёт словарь рядом с весами")


def _resolve_ce(art: Path, w_ce_list, log) -> tuple[list[Path], list[float]]:
    if not art.is_dir():
        raise RuntimeError(f"нет каталога артефактов {art}")
    ce_dirs = sorted(d for d in art.glob("ce_f*") if d.is_dir())
    if not ce_dirs:
        raise RuntimeError(
            f"в {art} нет ни одного каталога ce_f*; содержимое: "
            f"{sorted(p.name for p in art.iterdir())}")
    if w_ce_list is None:
        weights = [1.0] * len(ce_dirs)
    elif len(w_ce_list) != len(ce_dirs):
        raise RuntimeError(
            f"в blend.json {len(w_ce_list)} весов, а кросс-энкодеров {len(ce_dirs)}: "
            f"{[d.name for d in ce_dirs]}")
    else:
        weights = [float(w) for w in w_ce_list]
    order = sorted(range(len(ce_dirs)), key=lambda i: -weights[i])
    ce_dirs = [ce_dirs[i] for i in order]
    weights = [weights[i] for i in order]
    log(f"кросс-энкодеры: {[(d.name, w) for d, w in zip(ce_dirs, weights)]}")
    return ce_dirs, weights


def predict(names, attrs, cats, pos, id1, id2, root: str, t0: float, log=print) -> np.ndarray:
    art = Path(root) / "artifacts"
    blend_cfg = _load_json(art / "blend.json") if (art / "blend.json").is_file() else {}
    blend_max_len = blend_cfg.get("max_len")
    blend_max_len = int(blend_max_len) if blend_max_len is not None else None
    ce_batch = int(blend_cfg.get("ce_batch", 256))
    batch_by_name = {str(k): int(v) for k, v in
                     (blend_cfg.get("ce_batch_by_model") or {}).items()}
    w_ce_list = blend_cfg.get("w_ce_list")
    workers = int(blend_cfg.get("serialize_workers", 1))
    if workers > 1:
        log(f"сериализация: {workers} процессов (текст побайтово тот же, "
            f"меняется только время)")
    if "max_keys" in blend_cfg:
        log(f"ВНИМАНИЕ: blend.json задаёт max_keys={blend_cfg['max_keys']}, "
            f"и это значение ИГНОРИРУЕТСЯ: max_keys теперь едет вместе с весами "
            f"в {SERIALIZE_CONFIG_NAME}. Уберите ключ из blend.json, чтобы он "
            f"не создавал ложного впечатления настройки.")

    n_pairs = len(id1)
    budget = stage_budget(n_pairs)
    reserve = max(4.0, min(30.0, budget * 0.06))
    deadline = t0 + budget - reserve
    log(f"бюджет этапа {budget:.0f}s на {n_pairs:,} пар (резерв {reserve:.0f}s)")

    def left() -> float:
        return deadline - _t.time()

    known = np.fromiter(((a in pos) and (b in pos) for a, b in zip(id1, id2)),
                        dtype=bool, count=n_pairs)
    if not known.all():
        log(f"ВНИМАНИЕ: {int((~known).sum()):,} пар с неизвестными товарами -> 0")
    idx = np.where(known)[0]
    if len(idx) == 0:
        raise RuntimeError("ни одной пары с известными товарами")

    p1 = np.fromiter((pos[id1[i]] for i in idx), dtype=np.int64, count=len(idx))
    p2 = np.fromiter((pos[id2[i]] for i in idx), dtype=np.int64, count=len(idx))
    l1, l2 = p1.tolist(), p2.tolist()
    pair_cat = np.array([str(cats[i]) for i in l1])
    empty_cat = sum(1 for i in l1 if cats[i] is None or str(cats[i]).strip() == "")
    if empty_cat:
        log(f"ВНИМАНИЕ: у {empty_cat:,} пар категория пустая; они ранжируются "
            f"в отдельной группе, метрика по ним ничего не значит")
    mixed = sum(1 for i, j in zip(l1, l2) if cats[i] != cats[j])
    if mixed:
        log(f"ВНИМАНИЕ: у {mixed:,} пар категории сторон различаются; "
            f"ранг и префикс берутся по первому товару")

    ce_dirs, weights = _resolve_ce(art, w_ce_list, log)

    groups: dict[tuple, list[Path]] = {}
    max_len_by_dir: dict[Path, int] = {}
    batch_by_dir: dict[Path, int] = {}
    for d in ce_dirs:
        cfg = _checkpoint_config(d, log)
        ckpt_len = _checkpoint_max_len(d)
        if ckpt_len is None:
            if blend_max_len is None:
                raise RuntimeError(
                    f"у {d.name} нет {TRAIN_META_NAME} с max_len, и в blend.json "
                    f"его тоже нет. Это длина обрезки текста, она обязана "
                    f"совпадать с обучением; подставлять умолчание нельзя — "
                    f"модель получит не тот текст, а ошибки не будет. Соберите "
                    f"архив src/scripts/40_build_submission.py")
            log(f"{d.name}: нет {TRAIN_META_NAME}, max_len взят из blend.json "
                f"({blend_max_len})")
            ckpt_len = blend_max_len
        elif blend_max_len is not None and blend_max_len != ckpt_len:
            raise RuntimeError(
                f"{d.name} обучен с max_len={ckpt_len}, а blend.json задаёт "
                f"{blend_max_len}. Обрезка меняет то, что видит модель: лишние "
                f"токены, которых не было при обучении, либо потеря последних "
                f"ключей атрибутов — а именно различающиеся атрибуты и несут "
                f"решение. Уберите max_len из blend.json (он теперь едет "
                f"с каждым чекпойнтом) либо пересоберите архив")
        max_len_by_dir[d] = ckpt_len
        batch_by_dir[d] = batch_by_name.get(d.name, ce_batch)
        idf_path = _key_idf_source(d, art, cfg, log)
        groups.setdefault((cfg, idf_path), []).append(d)
    if len(set(max_len_by_dir.values())) > 1:
        log(f"max_len по моделям: "
            f"{ {d.name: v for d, v in max_len_by_dir.items()} }")
    if len(groups) > 1:
        log(f"групп сериализации: {len(groups)} — тексты считаются отдельно "
            f"для каждой конфигурации")

    parsed_cache: dict = {}
    release_raw = all(cfg.canon for cfg, _ in groups)

    idf_cache: dict[Path, dict] = {}
    scored: dict[Path, np.ndarray] = {}
    for (cfg, idf_path), dirs in groups.items():
        if left() <= 0:
            log(f"watchdog: пропускаю группу {[d.name for d in dirs]}, время вышло")
            break
        if idf_path not in idf_cache:
            idf_cache[idf_path] = _load_json(idf_path)
        key_idf = idf_cache[idf_path]
        log(f"группа {[d.name for d in dirs]}: key_idf "
            f"{len(key_idf) if key_idf else 0} ключей, до дедлайна {left():.0f}s")

        try:
            ta, tb = serialize_pairs(
                names, attrs, cats, p1, p2, key_idf, cfg, log,
                deadline=deadline,
                parsed=None if (workers > 1 and len(groups) == 1) else parsed_cache,
                release_raw=release_raw,
                workers=workers if len(groups) == 1 else 1)
        except BudgetExceeded:
            if not scored:
                raise
            log(f"watchdog: группа {[d.name for d in dirs]} не успела "
                f"сериализоваться, иду со смесью из уже посчитанных моделей")
            break
        log(f"сериализация готова, до дедлайна {left():.0f}s")

        for md, s in score_pairs(dirs, ta, tb, max_len=max_len_by_dir,
                                 batch=batch_by_dir, log=log, deadline=deadline):
            scored[md] = s
        del ta, tb

    parts: list[np.ndarray] = []
    used_w: list[float] = []
    for d, w in zip(ce_dirs, weights):
        s = scored.get(d)
        if s is None:
            log(f"{d.name}: результата нет, вес {w} в смесь не идёт")
            continue
        parts.append(s)
        used_w.append(w)
    if not parts:
        raise RuntimeError("кросс-энкодер не отдал ни одного результата")
    if ce_dirs[0] not in scored:
        raise RuntimeError(
            f"самая весомая модель {ce_dirs[0].name} (вес {weights[0]}) результата "
            f"не дала: не переехала на устройство, не влезла в память или была "
            f"оборвана сторожем. Смесь из оставшихся — это другая модель, "
            f"не та, что валидировалась; молча подменять её нельзя")

    if len(parts) == 1:
        scores = rank_within(parts[0], pair_cat)
    else:
        scores = blend(parts, used_w, pair_cat)
    log(f"смесь: источников {len(parts)} из {len(ce_dirs)}, веса {used_w}")

    out = np.zeros(n_pairs, dtype=np.float64)
    out[idx] = scores
    return out


In [ ]:
%%writefile src/ecup/testrate.json
{"Обувь": 0.04581, "Аптека": 0.1273, "Мебель": 0.04905, "Одежда": 0.05725, "Дом и сад": 0.09387, "Автотовары": 0.1178, "Электроника": 0.06474, "Спорт и отдых": 0.11153, "Бытовая химия": 0.2119, "Детские товары": 0.1468, "Бытовая техника": 0.20991, "Продукты питания": 0.11463, "Красота и гигиена": 0.21582, "Ювелирные изделия": 0.01571, "Хобби и творчество": 0.09738, "Товары для животных": 0.14515, "Канцелярские товары": 0.09494, "Строительство и ремонт": 0.12905, "Галантерея и аксессуары": 0.04304, "Музыкальные инструменты": 0.13473}


In [ ]:
%%writefile src/ecup/textprep.py
from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from dataclasses import dataclass
from typing import NamedTuple


WORD_RE = re.compile(r"[0-9a-zа-яё]+")


def normalize(s: str) -> str:
    return unicodedata.normalize("NFKC", s).lower().replace("ё", "е")


ID_KEYS: tuple[str, ...] = (
    "артикул", "партномер (артикул производителя)", "артикул производителя",
    "код товара", "oem-номер",
)
BRAND_KEYS: tuple[str, ...] = ("бренд", "производитель", "торговая марка")

VARIANT_KEYS: tuple[str, ...] = (
    "цвет товара", "цвет", "название цвета", "российский размер", "размер",
    "объем", "вес товара, г", "оптическая сила", "вкус", "аромат",
    "количество в упаковке", "единиц в одном товаре", "материал",
)

CANON_VARIANT_KEYS: tuple[str, ...] = (
    "цвет", "размер", "объем", "вес", "вкус", "аромат", "оптическая сила",
    "материал", "длина", "ширина", "высота", "диаметр", "количество в упаковке",
    "мощность", "ось",
)

PRIORITY_KEYS: tuple[str, ...] = tuple(ID_KEYS) + tuple(BRAND_KEYS) + ("тип",)

DROP_KEYS: frozenset[str] = frozenset({
    "валюта", "примечание", "гарантийный срок", "страна-изготовитель",
    "страна производства", "комплектация", "упаковка",
    "длина упаковки", "высота упаковки", "ширина упаковки",
    "вес брутто", "вес с упаковкой, г", "цена за",
})

DROP_KEYS_CANON: frozenset[str] = frozenset({
    "валюта", "примечание", "гарантийный срок", "комплектация", "упаковка",
    "длина упаковки", "высота упаковки", "ширина упаковки",
    "вес с упаковкой", "цена за",
})

KEY_SYNONYMS: dict[str, str] = {
    "название цвета": "цвет", "цвет товара": "цвет", "цвет изделия": "цвет",
    "бренд в одежде и обуви": "бренд", "производитель": "бренд",
    "торговая марка": "бренд",
    "страна-изготовитель": "страна", "страна производства": "страна",
    "страна бренда": "страна",
    "размер производителя": "размер", "российский размер": "размер",
    "размер одежды": "размер", "размер обуви": "размер",
    "вес нетто": "вес", "вес брутто": "вес", "вес товара": "вес",
    "объем товара": "объем",
}


UNITS: dict[str, tuple[str, float]] = {
    "кг": ("mass", 1000.0), "kg": ("mass", 1000.0),
    "г": ("mass", 1.0), "гр": ("mass", 1.0), "g": ("mass", 1.0),
    "мг": ("mass", 0.001), "mg": ("mass", 0.001),
    "л": ("vol", 1000.0), "l": ("vol", 1000.0),
    "мл": ("vol", 1.0), "ml": ("vol", 1.0),
    "м": ("len", 1000.0), "m": ("len", 1000.0),
    "см": ("len", 10.0), "cm": ("len", 10.0),
    "мм": ("len", 1.0), "mm": ("len", 1.0),
    "вт": ("power", 1.0), "w": ("power", 1.0),
    "квт": ("power", 1000.0), "kw": ("power", 1000.0),
    "мач": ("charge", 1.0), "mah": ("charge", 1.0),
    "гб": ("mem", 1024.0), "gb": ("mem", 1024.0),
    "тб": ("mem", 1048576.0), "tb": ("mem", 1048576.0),
    "мб": ("mem", 1.0), "mb": ("mem", 1.0),
    "шт": ("count", 1.0), "штук": ("count", 1.0), "pcs": ("count", 1.0),
    "уп": ("count", 1.0), "таб": ("count", 1.0), "капс": ("count", 1.0),
}

_KEY_UNIT: dict[str, tuple[str, float]] = {
    u: fm for u, fm in UNITS.items() if not u.isascii() and fm[0] != "count"
}

_UNIT_SUF_NO_CONV: frozenset[str] = frozenset({"шт", "в", "%"})

_UNIT_SUF_ORDER: tuple[str, ...] = (
    "мм", "см", "м", "кг", "г", "гр", "мг", "мл", "л", "шт", "вт", "квт",
    "в", "мач", "гб", "мб", "тб", "%",
)

_UNIT_SUF_RE = re.compile(rf"(?:\s|,)\s*({'|'.join(_UNIT_SUF_ORDER)})\.?$")

_PAREN_RE = re.compile(r"\s*\([^)]*\)")
_PAREN_UNIT_RE = re.compile(r"\s*\(\s*([а-яa-z%]{1,3})\s*\)\s*$")
_NUM_SUF_RE = re.compile(r"\s*\d+$")
_WS_RE = re.compile(r"\s+")
_LEAD_NUM_RE = re.compile(r"^\s*(\d+(?:[.,]\d+)?)")


def canon_key(k: str) -> tuple[str, str | None]:
    k = normalize(k).strip()
    unit = None
    mu = _UNIT_SUF_RE.search(k)
    if mu:
        unit = mu.group(1)
        k = k[:mu.start()]
    else:
        mp = _PAREN_UNIT_RE.search(k)
        if mp and mp.group(1) in _KEY_UNIT:
            unit = mp.group(1)
            k = k[:mp.start()]
    k = _PAREN_RE.sub("", k)
    k = _NUM_SUF_RE.sub("", k)
    k = _WS_RE.sub(" ", k).strip()
    return KEY_SYNONYMS.get(k, k), unit


def canon_value(v: str, unit: str | None) -> str:
    if unit is None:
        return v
    fam_mult = _KEY_UNIT.get(unit)
    if fam_mult is None:
        return v
    m = _LEAD_NUM_RE.match(v)
    if not m:
        return v
    try:
        x = float(m.group(1).replace(",", ".")) * fam_mult[1]
    except ValueError:
        return v
    return f"{x:g}"


def canonicalize(kv: dict[str, str]) -> dict[str, str]:
    out: dict[str, str] = {}
    for k, v in kv.items():
        ck, unit = canon_key(k)
        cv = canon_value(v, unit)
        if ck not in out or len(cv) > len(out[ck]):
            out[ck] = cv
    return out


PRIORITY_KEYS_CANON: frozenset[str] = frozenset(canon_key(k)[0] for k in PRIORITY_KEYS)

VARIANT_KEYS_CANON: frozenset[str] = (
    frozenset(CANON_VARIANT_KEYS) | frozenset(canon_key(k)[0] for k in VARIANT_KEYS)
)


class KeySpace(NamedTuple):
    canon: bool
    drop: frozenset[str]
    priority: frozenset[str]
    variant: frozenset[str]


RAW_SPACE = KeySpace(False, DROP_KEYS, frozenset(PRIORITY_KEYS), frozenset(VARIANT_KEYS))
CANON_SPACE = KeySpace(True, DROP_KEYS_CANON, PRIORITY_KEYS_CANON, VARIANT_KEYS_CANON)


def key_space(canon: bool) -> KeySpace:
    return CANON_SPACE if canon else RAW_SPACE


@dataclass(slots=True)
class ParsedItem:
    name_norm: str
    kv_raw: dict[str, str]
    _kv_canon: dict[str, str] | None = None
    attrs_failed: bool = False

    @property
    def kv_canon(self) -> dict[str, str]:
        if self._kv_canon is None:
            self._kv_canon = canonicalize(self.kv_raw)
        return self._kv_canon

    def kv(self, canon: bool) -> dict[str, str]:
        return self.kv_canon if canon else self.kv_raw

    def release_raw(self) -> None:
        self._kv_canon = self.kv_canon
        self.kv_raw = {}


def _pairs_from_list(seq) -> dict[str, str] | None:
    out: dict[str, str] = {}
    for el in seq:
        if isinstance(el, dict):
            k = el.get("name", el.get("key", el.get("attribute_name")))
            v = el.get("value", el.get("val", el.get("attribute_value")))
            if k is None:
                return None
        elif isinstance(el, (list, tuple)) and len(el) == 2:
            k, v = el
        else:
            return None
        if v is None:
            continue
        if isinstance(v, (list, tuple)):
            v = ", ".join(str(x) for x in v if x is not None)
        out[str(k)] = str(v)
    return out


def parse_attributes(attributes) -> tuple[dict, bool]:
    if attributes is None:
        return {}, False
    if isinstance(attributes, dict):
        return attributes, False
    if isinstance(attributes, (bytes, bytearray)):
        try:
            attributes = attributes.decode("utf-8")
        except Exception:
            return {}, True
    if isinstance(attributes, str):
        s = attributes.strip()
        if not s:
            return {}, False
        try:
            obj = json.loads(s)
        except Exception:
            return {}, True
        if isinstance(obj, str):
            return {}, True
        return parse_attributes(obj)
    if isinstance(attributes, (list, tuple)):
        d = _pairs_from_list(attributes)
        return ({}, True) if d is None else (d, False)
    return {}, True


def parse_item(name, attributes) -> ParsedItem:
    name_norm = normalize("" if name is None else str(name))
    attrs, failed = parse_attributes(attributes)

    kv: dict[str, str] = {}
    for k, v in attrs.items():
        if v is None:
            continue
        if isinstance(v, (list, tuple)):
            v = ", ".join(str(x) for x in v if x is not None)
        vs = normalize(str(v)).strip()
        if not vs:
            continue
        kv[normalize(str(k)).strip()] = vs
    return ParsedItem(name_norm, kv, None, failed)


def dictionaries_digest() -> str:
    payload = {
        "ID_KEYS": list(ID_KEYS),
        "BRAND_KEYS": list(BRAND_KEYS),
        "VARIANT_KEYS": list(VARIANT_KEYS),
        "CANON_VARIANT_KEYS": list(CANON_VARIANT_KEYS),
        "PRIORITY_KEYS": list(PRIORITY_KEYS),
        "DROP_KEYS": sorted(DROP_KEYS),
        "DROP_KEYS_CANON": sorted(DROP_KEYS_CANON),
        "KEY_SYNONYMS": dict(sorted(KEY_SYNONYMS.items())),
        "UNITS": {k: list(v) for k, v in sorted(UNITS.items())},
        "UNIT_SUF_NO_CONV": sorted(_UNIT_SUF_NO_CONV),
        "UNIT_SUF_ORDER": list(_UNIT_SUF_ORDER),
        "RE": [_UNIT_SUF_RE.pattern, _PAREN_RE.pattern, _PAREN_UNIT_RE.pattern,
               _NUM_SUF_RE.pattern, _WS_RE.pattern, _LEAD_NUM_RE.pattern,
               WORD_RE.pattern],
    }
    blob = json.dumps(payload, ensure_ascii=False, sort_keys=True).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()[:12]


def _check_constants() -> None:
    named = {
        "ID_KEYS": ID_KEYS, "BRAND_KEYS": BRAND_KEYS, "VARIANT_KEYS": VARIANT_KEYS,
        "CANON_VARIANT_KEYS": CANON_VARIANT_KEYS, "PRIORITY_KEYS": PRIORITY_KEYS,
        "DROP_KEYS": DROP_KEYS, "DROP_KEYS_CANON": DROP_KEYS_CANON,
        "KEY_SYNONYMS.keys": tuple(KEY_SYNONYMS), "KEY_SYNONYMS.values": tuple(KEY_SYNONYMS.values()),
    }
    for where, keys in named.items():
        for k in keys:
            if normalize(k).strip() != k:
                raise ValueError(
                    f"{where}: запись {k!r} не совпадает со своей нормализованной формой "
                    f"{normalize(k).strip()!r} и потому не сработает никогда")

    expected = set(_KEY_UNIT) | set(_UNIT_SUF_NO_CONV)
    if set(_UNIT_SUF_ORDER) != expected:
        raise ValueError(
            f"альтернация _UNIT_SUF_RE разошлась с таблицами единиц: "
            f"лишние {sorted(set(_UNIT_SUF_ORDER) - expected)}, "
            f"недостающие {sorted(expected - set(_UNIT_SUF_ORDER))}")
    if set(_KEY_UNIT) & set(_UNIT_SUF_NO_CONV):
        raise ValueError("единица одновременно и пересчитываемая, и нет")

    for space_name, space in (("сырое", RAW_SPACE), ("каноническое", CANON_SPACE)):
        clash = space.drop & (space.priority | space.variant)
        if clash:
            raise ValueError(
                f"{space_name} пространство: ключи {sorted(clash)} одновременно "
                f"мусорные и значимые")


_check_constants()


In [ ]:
%%writefile src/scripts/01_fetch_data.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from src.config import BASE_URL, RAW                          # noqa: E402
from src.utils import data                                    # noqa: E402


def main() -> int:
    p = argparse.ArgumentParser(
        prog="01_fetch_data.py",
        description=("Загрузка данных соревнования с хранилища организаторов. "
                     "Тот же код, что зовёт 02_bge.ipynb через bootstrap.setup()."))
    p.add_argument("--group", default="all", choices=sorted(data.GROUPS),
                   help="all — вместе с items.parquet (~4.5 ГБ), "
                        "light — без него (~340 МБ)")
    p.add_argument("--files", nargs="*", default=None,
                   help="только эти файлы вместо целой группы")
    p.add_argument("--parts", type=int, default=data.PARTS,
                   help=f"диапазонов на файл (по умолчанию: {data.PARTS})")
    a = p.parse_args()

    print(f"=== данные соревнования: {BASE_URL}")
    print(f"=== каталог: {RAW}")
    try:
        data.ensure_data(group=a.group, names=a.files, jobs=a.parts)
    except Exception as exc:
        print(f"=== ЗАГРУЗКА: ЕСТЬ ПРОБЛЕМЫ — {exc}", file=sys.stderr)
        return 1
    print("=== ДАННЫЕ НА МЕСТЕ")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/scripts/05_build_index.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import sys
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from src.ecup import oof
from src.ecup.config import ARTIFACTS, DATA
from src.ecup.folds import first_tokens, make_unseen_folds


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="05_build_index.py",
        description="Строит общий индекс OOF (метки, категории, фолды, отпечаток).")
    p.add_argument("--matches", type=Path, default=DATA / "matches.parquet")
    p.add_argument("--items", type=Path, default=DATA / "items_human.parquet")
    p.add_argument("--artifacts", type=Path, default=ARTIFACTS)
    p.add_argument("--n-folds", dest="n_folds", type=int, default=5)
    p.add_argument("--force", action="store_true",
                   help="перезаписать существующий индекс. Осторожно: все "
                        "накопленные OOF считались на старом разбиении")
    return p.parse_args()


def main() -> int:
    a = parse_args()
    out = oof.index_path(a.artifacts)
    if out.is_file() and not a.force:
        idx = oof.load_index(a.artifacts)
        print(f"индекс уже есть: {out}")
        print(f"  пар {len(idx):,}, фолдов {idx.n_folds}, отпечаток {idx.digest}")
        print("  перезаписать — флаг --force (все накопленные OOF станут "
              "несовместимы, если разбиение изменится)")
        return 0

    for p in (a.matches, a.items):
        if not p.is_file():
            print(f"нет файла {p}", file=sys.stderr)
            return 1

    m = pq.read_table(a.matches, columns=["id1", "id2", "target"])
    id1 = m.column("id1").to_numpy()
    id2 = m.column("id2").to_numpy()
    y = np.asarray(m.column("target").to_numpy(), dtype=np.float64)
    if not np.isin(y, (0.0, 1.0)).all():
        raise ValueError("в колонке target есть значения, кроме 0 и 1")
    y = y.astype(np.int8)

    it = pq.read_table(a.items, columns=["id", "name", "category"])
    ids = it.column("id").to_numpy()
    names = it.column("name").to_pylist()
    cats = it.column("category").to_pylist()
    pos = {int(v): i for i, v in enumerate(ids.tolist())}

    miss = [i for i, v in enumerate(id1.tolist()) if int(v) not in pos]
    if miss:
        raise RuntimeError(
            f"{len(miss):,} пар ссылаются на товар, которого нет в {a.items.name}. "
            f"Индекс строить нельзя: у этих строк не определены ни категория, "
            f"ни фолд")
    p1 = np.fromiter((pos[int(v)] for v in id1.tolist()), dtype=np.int64, count=len(id1))
    pair_cat = np.array([str(cats[i]) for i in p1.tolist()])
    pair_name = [names[i] for i in p1.tolist()]

    fold = make_unseen_folds(pair_cat, first_tokens(pair_name), n_folds=a.n_folds)
    digest = oof.pairs_digest(id1, id2)
    oof.save_index(a.artifacts, y, pair_cat, fold, digest)

    print(f"индекс записан: {out}")
    print(f"  пар               {len(y):,}")
    print(f"  доля позитивов    {y.mean():.4f}")
    print(f"  категорий         {len(np.unique(pair_cat))}")
    print(f"  отпечаток строк   {digest}")
    print()
    print(f"  {'фолд':>5} {'пар':>9} {'позитивов':>10} {'категорий':>10}")
    for f in range(a.n_folds):
        k = fold == f
        print(f"  {f:>5} {int(k.sum()):>9,} {y[k].mean():>10.4f} "
              f"{len(np.unique(pair_cat[k])):>10}")
    print()
    print("Доля позитивов по фолдам обязана быть близкой: разброс означает, что "
          "фолды несравнимы между собой и OOF собран из разных задач.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile src/scripts/07_noise_mask.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import re
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from src.ecup.config import ARTIFACTS, DATA

WORD_RE = re.compile(r"\w+")

GATE_DEFAULT = ("Красота и гигиена",)


def main() -> int:
    p = argparse.ArgumentParser(prog="07_noise_mask.py")
    p.add_argument("--text", default=None,
                   help="человеческий корпус (порядок строк = порядок маски)")
    p.add_argument("--hi", type=float, default=0.8,
                   help="негатив противоречив при жаккаре имён >= hi")
    p.add_argument("--lo", type=float, default=0.3,
                   help="позитив противоречив при жаккаре имён <= lo")
    p.add_argument("--ovl", type=float, default=0.0,
                   help="негатив противоречив ещё и при ВЛОЖЕННОСТИ имён "
                        "|A&B|/min(|A|,|B|) >= ovl. Ноль выключает ветку. "
                        "Жаккар не видит случай «одно имя усечённое, второе "
                        "полное»: общих токенов много, но объединение раздуто "
                        "длинным именем, и жаккар падает ниже порога. Замер по "
                        "fold0: всплывших негативов (балл выше медианы "
                        "позитивов) жаккар ловит 13%%, вложенность 41%% при той "
                        "же точности. Ветка идёт ТОЛЬКО в обучающую маску")
    p.add_argument("--agree", type=float, default=0.8,
                   help="доля совпавших значений среди ОБЩИХ ключей атрибутов, "
                        "при которой атрибуты объясняют метку и снимают "
                        "противоречие. 0 выключает поправку — получится маска "
                        "только по имени, как было до 24.08")
    p.add_argument("--gate", nargs="*", default=list(GATE_DEFAULT),
                   help="категории, глушимые целиком; пустой список выключает")
    p.add_argument("--canon", type=int, choices=(0, 1), default=0,
                   help="пространство ключей атрибутов; обязано совпадать с тем, "
                        "которым сериализован корпус")
    p.add_argument("--out", default=None)
    a = p.parse_args()

    import numpy as np
    import pyarrow.parquet as pq

    from src.ecup.textprep import parse_item

    text = Path(a.text) if a.text else ARTIFACTS / "ce_text_human_raw_k48.parquet"
    t = pq.read_table(text, columns=["id1", "id2", "category", "target"])
    id1 = t.column("id1").to_numpy(zero_copy_only=False)
    id2 = t.column("id2").to_numpy(zero_copy_only=False)
    cat = np.asarray(t.column("category").to_pylist(), dtype=object)
    y = t.column("target").to_numpy(zero_copy_only=False).astype(np.float32)
    pos = y > 0.5

    need = set(int(v) for v in id1) | set(int(v) for v in id2)
    it = pq.read_table(DATA / "items_human.parquet",
                       columns=["id", "name", "attributes"])
    ids = it.column("id").to_numpy(zero_copy_only=False)
    names = it.column("name").to_pylist()
    attrs_raw = it.column("attributes").to_pylist()

    tok: dict = {}
    kv: dict = {}
    for i, n, at in zip(ids, names, attrs_raw):
        i = int(i)
        if i not in need:
            continue
        tok[i] = set(WORD_RE.findall((n or "").lower()))
        kv[i] = parse_item(n, at).kv(bool(a.canon))
    del names, attrs_raw, it
    empty: set = set()
    empty_map: dict = {}

    def jac(u, v):
        A, B = tok.get(int(u), empty), tok.get(int(v), empty)
        s = len(A | B)
        return len(A & B) / s if s else 0.0

    def agr(u, v):
        da, db = kv.get(int(u), empty_map), kv.get(int(v), empty_map)
        common = da.keys() & db.keys()
        if not common:
            return float("nan")
        return sum(da[c] == db[c] for c in common) / len(common)

    def ovl(u, v):
        A, B = tok.get(int(u), empty), tok.get(int(v), empty)
        m = min(len(A), len(B))
        return len(A & B) / m if m else 0.0

    J = np.fromiter((jac(u, v) for u, v in zip(id1, id2)),
                    dtype=np.float32, count=len(y))
    O = (np.fromiter((ovl(u, v) for u, v in zip(id1, id2)),
                     dtype=np.float32, count=len(y))
         if a.ovl > 0 else np.zeros(len(y), dtype=np.float32))
    A = np.fromiter((agr(u, v) for u, v in zip(id1, id2)),
                    dtype=np.float32, count=len(y))

    noisy_name = (~pos & (J >= a.hi)) | (pos & (J <= a.lo))

    with np.errstate(invalid="ignore"):
        explained = np.where(pos, A >= a.agree, A < a.agree)
    explained &= ~np.isnan(A)
    noisy_train = noisy_name | (~pos & (O >= a.ovl)) if a.ovl > 0 else noisy_name
    noisy = noisy_train & ~explained

    gated = np.isin(cat, np.asarray(a.gate, dtype=object)) if a.gate \
        else np.zeros(len(y), dtype=bool)

    out = Path(a.out) if a.out else ARTIFACTS / (
        text.stem + f".noise_{a.hi}_{a.lo}_{a.agree}"
        + (f"_ovl{a.ovl}" if a.ovl > 0 else "") + ".npz")
    np.savez_compressed(out, noisy=noisy, noisy_name=noisy_name, gated=gated,
                        jaccard=J, overlap=O, agreement=A, hi=a.hi, lo=a.lo,
                        ovl=a.ovl,
                        agree=a.agree, gate=np.asarray(a.gate, dtype=object),
                        canon=a.canon, source=str(text))

    def share(m):
        return f"{int(m.sum()):>7,} ({100 * m.mean():>5.1f}%)"

    print(f"пар {len(y):,}")
    print(f"  противоречат имени          {share(noisy_name)}")
    if a.ovl > 0:
        print(f"  добавила вложенность        {share(noisy_train & ~noisy_name)}")
    print(f"    из них объяснены атрибутами {share(noisy_name & explained)}")
    print(f"  ГЛУШИТСЯ маской             {share(noisy)}")
    print(f"  ГЛУШИТСЯ гейтом {list(a.gate)}  {share(gated)}")
    print(f"  всего под весом             {share(noisy | gated)}")
    print(f"  без общих ключей атрибутов  {share(np.isnan(A))}")
    print("\nдоля глушения по категориям (маска -> маска+гейт):")
    for c in sorted(set(cat), key=lambda c: -noisy[cat == c].mean()):
        m = cat == c
        print(f"  {c:<26}{100 * noisy[m].mean():>6.1f}% ->"
              f"{100 * (noisy | gated)[m].mean():>6.1f}%"
              f"   было по имени {100 * noisy_name[m].mean():>5.1f}%")
    print(f"\n-> {out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile src/scripts/08_context_feats.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from src.ecup.config import ARTIFACTS, DATA


def main() -> int:
    p = argparse.ArgumentParser(prog="08_context_feats.py")
    p.add_argument("--text", required=True, help="корпус, к текстам которого дописать контекст")
    p.add_argument("--items", default=None)
    p.add_argument("--topk", type=int, default=50, help="сколько соседей смотреть")
    p.add_argument("--block", type=int, default=2000, help="запросов за раз")
    p.add_argument("--out", default=None)
    p.add_argument("--limit", type=int, default=0, help="только первые N пар (отладка)")
    a = p.parse_args()

    import numpy as np
    import pyarrow as pa
    import pyarrow.parquet as pq
    from sklearn.feature_extraction.text import TfidfVectorizer

    t0 = time.time()
    def log(m):
        print(f"[{time.time() - t0:7.1f}s] {m}", flush=True)

    src = Path(a.text)
    t = pq.read_table(src)
    if a.limit:
        t = t.slice(0, a.limit)
    id1 = t.column("id1").to_numpy(zero_copy_only=False)
    id2 = t.column("id2").to_numpy(zero_copy_only=False)
    log(f"пар: {len(id1):,}")

    items = Path(a.items) if a.items else DATA / "items_human.parquet"
    it = pq.read_table(items, columns=["id", "name"])
    ids = it.column("id").to_numpy(zero_copy_only=False)
    names = [(n or "") for n in it.column("name").to_pylist()]
    pos_s = {int(v): i for i, v in enumerate(ids)}
    log(f"каталог: {len(ids):,} товаров")

    vec = TfidfVectorizer(analyzer="word", token_pattern=r"\w+", lowercase=True,
                          min_df=2, sublinear_tf=True)
    X = vec.fit_transform(names)
    import sklearn.preprocessing as pp
    X = pp.normalize(X)
    log(f"tf-idf: {X.shape[0]:,} x {X.shape[1]:,}, ненулевых {X.nnz:,}")

    i1 = np.fromiter((pos_s.get(int(v), -1) for v in id1), dtype=np.int64, count=len(id1))
    i2 = np.fromiter((pos_s.get(int(v), -1) for v in id2), dtype=np.int64, count=len(id2))
    absent = (i1 < 0) | (i2 < 0)
    if absent.any():
        log(f"ВНИМАНИЕ: {int(absent.sum()):,} пар с товарами вне каталога — им контекст нулевой")

    sim = np.zeros(len(i1), dtype=np.float32)
    rank = np.full(len(i1), a.topk, dtype=np.int32)
    rka = np.zeros(len(i1), dtype=np.float32)
    rkb = np.zeros(len(i1), dtype=np.float32)
    KRK = 10
    XT = X.T.tocsr()
    for s in range(0, len(i1), a.block):
        e = min(s + a.block, len(i1))
        q = i1[s:e]
        ok = q >= 0
        if not ok.any():
            continue
        S = (X[q[ok]] @ XT).toarray()
        for j, (row, gi) in enumerate(zip(S, np.flatnonzero(ok))):
            tgt = i2[s + gi]
            if tgt < 0:
                continue
            row[q[ok][j]] = -1.0
            sim[s + gi] = row[tgt]
            above = int((row > row[tgt]).sum())
            rank[s + gi] = min(above, a.topk)
            top = np.partition(row, -KRK)[-KRK:]
            rka[s + gi] = float(top.mean())
        if (s // a.block) % 20 == 0:
            log(f"  {e:,}/{len(i1):,}")
    for s in range(0, len(i2), a.block):
        e = min(s + a.block, len(i2))
        q = i2[s:e]
        ok = q >= 0
        if not ok.any():
            continue
        S = (X[q[ok]] @ XT).toarray()
        for j, gi in enumerate(np.flatnonzero(ok)):
            row = S[j]
            row[q[ok][j]] = -1.0
            top = np.partition(row, -KRK)[-KRK:]
            rkb[s + gi] = float(top.mean())
        if (s // a.block) % 20 == 0:
            log(f"  r_K правой {e:,}/{len(i2):,}")
    csls = 2.0 * sim - rka - rkb
    log(f"близость медиана {np.median(sim):.3f}; ранг 0 у {100*(rank==0).mean():.1f}%; "
        f"CSLS медиана {np.median(csls):.3f}; r_K медиана {np.median(rka):.3f}")

    import re as _re
    ta_src = t.column("text_a").to_pylist()
    tb_src = t.column("text_b").to_pylist()

    def _parse_side(s, left_side):
        ch = s.split(" | ")
        head_ = ch[0]
        name_ = head_.split(" :: ", 1)[1] if left_side and " :: " in head_ else head_
        attr_ = {}
        for k in ch[1:]:
            if "=" in k:
                key_, val_ = k.split("=", 1)
                attr_[key_.strip()] = val_.strip().lower()
        return name_.lower(), attr_

    NA = [_parse_side(s, True) for s in ta_src]
    NB = [_parse_side(s, False) for s in tb_src]
    _ch = lambda s: set(_re.findall(r"\d+(?:[.,]\d+)?", s))

    def _conflict_nums(x, z):
        ca, cb = _ch(x), _ch(z)
        if not ca or not cb:
            return 0.0
        return 1.0 - len(ca & cb) / max(len(ca | cb), 1)

    def _conflict_attr(x, z):
        common_ = set(x) & set(z)
        if not common_:
            return 0.0
        return sum(1 for k in common_ if x[k] != z[k]) / len(common_)

    numc = np.array([_conflict_nums(a[0], b[0]) for a, b in zip(NA, NB)], dtype=np.float32)
    attrc = np.array([_conflict_attr(a[1], b[1]) for a, b in zip(NA, NB)], dtype=np.float32)
    log(f"конфликт чисел: медиана {np.median(numc):.2f}; "
        f"конфликт атрибутов: медиана {np.median(attrc):.2f}")

    ctx = [f"близость={s:.2f} ранг={r} csls={cs:.2f} густота={rk:.2f} "
           f"чисразн={nc:.2f} атрразн={ac:.2f} :: "
           for s, r, cs, rk, nc, ac in zip(sim, rank, csls, (rka + rkb) / 2, numc, attrc)]
    ta = [c + x for c, x in zip(ctx, t.column("text_a").to_pylist())]
    out = Path(a.out) if a.out else src.with_name(src.stem + "_ctx.parquet")
    cols = {n: t.column(n) for n in t.schema.names}
    cols["text_a"] = pa.array(ta, pa.large_string())
    pq.write_table(pa.table(cols), out, compression="zstd")

    meta_in = Path(str(src) + ".meta"); meta_out = Path(str(out) + ".meta")
    if meta_in.is_dir():
        import shutil, json
        if meta_out.exists():
            shutil.rmtree(meta_out)
        shutil.copytree(meta_in, meta_out)
        f = meta_out / "prep.json"
        if f.is_file():
            d = json.loads(f.read_text(encoding="utf-8"))
            d["context_feats"] = {"topk": a.topk, "source": str(src)}
            f.write_text(json.dumps(d, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    log(f"готово -> {out} ({out.stat().st_size / 2**20:.0f} МБ)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile src/scripts/10_prep_text.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from dataclasses import replace
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from src.ecup.config import ARTIFACTS, DATA, SerializeConfig
from src.ecup.serialize import (key_idf_from_parsed, serialize_pairs,
                            suggested_missing_idf)
from src.ecup.textprep import parse_item

_W: dict = {}

MIN_CHUNK = 2000


def meta_dir(out: Path) -> Path:
    return Path(str(out) + ".meta")


def key_idf_name(space: str, min_df: int) -> str:
    return f"key_idf_{space}_mindf{min_df}.json"


def _silent(*_args, **_kwargs) -> None:
    pass


def _init_worker(key_idf, cfg_dict) -> None:
    _W["key_idf"] = key_idf
    _W["cfg"] = SerializeConfig.from_dict(cfg_dict)


def _work(task):
    names, attrs, cats, loc1, loc2 = task
    return serialize_pairs(names, attrs, cats, loc1, loc2,
                           _W["key_idf"], _W["cfg"], log=_silent)


def resolve_n_jobs(value: int | None) -> int:
    cpu = os.cpu_count() or 1
    if value is None:
        value = int(os.environ.get("ECUP_N_JOBS", "0"))
    if value == 0:
        value = min(4, cpu)
    elif value < 0:
        value = max(1, cpu + value)
    if value > cpu:
        print(f"  n_jobs={value} больше числа ядер {cpu}, беру {cpu}")
        value = cpu
    return max(1, value)


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="10_prep_text.py",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        description=(
            "Готовит тексты пар для кросс-энкодера и сохраняет их в parquet.\n"
            "Колонки: id1, id2, target, category, brand, text_a, text_b.\n"
            "Рядом создаётся каталог <выход>.meta/ с serialize_config.json —\n"
            "обучение читает конфиг оттуда и кладёт его в каталог чекпойнта."),
        epilog=(
            "Примеры:\n"
            "  # человеческие пары, канонические ключи, 16 ключей, 4 процесса\n"
            "  src/scripts/10_prep_text.py --split human --canon 1 --max-keys 16 --n-jobs 4\n"
            "  # LLM-пары для стадии A на том же key_idf\n"
            "  src/scripts/10_prep_text.py --split llm --canon 1 --max-keys 16 --n-jobs 8\n"
            "  # крошечный прогон для проверки формата\n"
            "  src/scripts/10_prep_text.py --split human --limit 200 --n-jobs 1 --out /tmp/t.parquet"))
    p.add_argument("--split", choices=("human", "llm"), required=True,
                   help="human — размеченные людьми пары (обучение стадии B и вся "
                        "валидация); llm — пары с машинной разметкой для стадии A")
    p.add_argument("--canon", type=int, choices=(0, 1), default=1,
                   help="1 — канонические ключи атрибутов, 0 — сырые. "
                        "Боевые чекпойнты обучены на сырых "
                        "(по умолчанию: 1)")
    p.add_argument("--max-keys", dest="max_keys", type=int, default=16,
                   help="сколько ключей атрибутов попадает в текст пары "
                        "(по умолчанию: 16 — значение лучшей измеренной модели)")
    p.add_argument("--max-val-chars", dest="max_val_chars", type=int, default=60,
                   help="обрезка значения атрибута (по умолчанию: 60)")
    p.add_argument("--missing-key-idf", dest="missing_key_idf", default="1.0",
                   help="IDF ключа, которого нет в словаре: число либо auto. "
                        "Историческое 1.0 инвертировано по смыслу (соответствует "
                        "очень частому ключу), но именно на нём обучены боевые "
                        "чекпойнты (по умолчанию: 1.0)")
    p.add_argument("--n-jobs", dest="n_jobs", type=int, default=None,
                   help="число процессов: N — ровно N, 0 — min(4, ядер), "
                        "-1 — все ядра кроме одного. По умолчанию ECUP_N_JOBS "
                        "или min(4, ядер). Все ядра занимать нельзя: рядом идёт обучение")
    p.add_argument("--out", type=Path, default=None,
                   help="путь к выходному parquet (по умолчанию — в ARTIFACTS, "
                        "имя собирается из split/canon/max_keys)")
    p.add_argument("--key-idf", dest="key_idf", type=Path, default=None,
                   help="готовый словарь IDF ключей; иначе для human он считается "
                        "и сохраняется, а для llm читается из ARTIFACTS")
    p.add_argument("--min-df", dest="min_df", type=int, default=5,
                   help="отсечка частоты ключа при подсчёте IDF (по умолчанию: 5)")
    p.add_argument("--limit", type=int, default=0,
                   help="взять только первые N пар (для быстрой проверки формата)")
    p.add_argument("--llm-pairs", dest="llm_pairs", type=int, default=2_500_000,
                   help="сколько LLM-пар отобрать (по умолчанию: 2500000 — "
                        "столько было у стадии A лучшей модели)")
    p.add_argument("--share-grey", dest="share_grey", type=float, default=0.40,
                   help="доля спорных LLM-пар (0<target<1) в выборке")
    p.add_argument("--share-pos", dest="share_pos", type=float, default=0.20,
                   help="доля уверенных позитивов; остальное — уверенные негативы")
    p.add_argument("--seed", type=int, default=7, help="сид отбора LLM-пар")
    p.add_argument("--compression", default="zstd",
                   help="кодек parquet (по умолчанию: zstd)")
    return p.parse_args()


def read_items(path: Path):
    import pyarrow.parquet as pq
    t = pq.read_table(path, columns=["id", "name", "attributes", "category"])
    return (t.column("id").to_pylist(), t.column("name").to_pylist(),
            t.column("attributes").to_pylist(), t.column("category").to_pylist())


def read_matches(path: Path):
    import pyarrow.parquet as pq
    t = pq.read_table(path, columns=["id1", "id2", "target"])
    return (t.column("id1").to_pylist(), t.column("id2").to_pylist(),
            t.column("target").to_pylist())


def sample_llm(id1, id2, target, args, log):
    import numpy as np

    y = np.asarray(target, dtype=np.float32)
    grey = np.where((y > 0) & (y < 1))[0]
    pos = np.where(y == 1)[0]
    neg = np.where(y == 0)[0]
    log(f"  спорных {len(grey):,} | уверенно да {len(pos):,} | уверенно нет {len(neg):,}")

    n = args.llm_pairs
    n_grey = min(int(n * args.share_grey), len(grey))
    n_pos = min(int(n * args.share_pos), len(pos))
    n_neg = min(n - n_grey - n_pos, len(neg))
    rng = np.random.default_rng(args.seed)
    take = np.concatenate([
        rng.choice(grey, n_grey, replace=False),
        rng.choice(pos, n_pos, replace=False),
        rng.choice(neg, n_neg, replace=False)])
    rng.shuffle(take)
    log(f"  берём {len(take):,} = спорных {n_grey:,} + да {n_pos:,} + нет {n_neg:,}")
    a = [id1[i] for i in take.tolist()]
    b = [id2[i] for i in take.tolist()]
    return a, b, y[take]


def load_items_for(ids_needed: set, path: Path, log):
    import pyarrow.parquet as pq

    t0 = time.time()
    ids, names, attrs, cats = [], [], [], []
    pf = pq.ParquetFile(path)
    for i, batch in enumerate(pf.iter_batches(
            batch_size=200_000, columns=["id", "name", "attributes", "category"])):
        col = batch.column("id").to_pylist()
        keep = [j for j, v in enumerate(col) if v in ids_needed]
        if not keep:
            continue
        nm = batch.column("name").to_pylist()
        at = batch.column("attributes").to_pylist()
        ct = batch.column("category").to_pylist()
        ids.extend(col[j] for j in keep)
        names.extend(nm[j] for j in keep)
        attrs.extend(at[j] for j in keep)
        cats.extend(ct[j] for j in keep)
        if i % 10 == 0:
            log(f"    пачек {i}, набрано {len(ids):,} ({time.time() - t0:.0f}s)")
    log(f"  товаров прочитано: {len(ids):,} за {time.time() - t0:.0f}s")
    return ids, names, attrs, cats


def serialize(names, attrs, cats, p1, p2, key_idf, cfg, n_jobs, log):
    if n_jobs <= 1:
        return serialize_pairs(names, attrs, cats, p1, p2, key_idf, cfg, log=log)

    from concurrent.futures import ProcessPoolExecutor

    n = len(p1)
    n_chunks = min(max(n_jobs * 3, 1), max(1, n // MIN_CHUNK))
    chunk = n // n_chunks + 1
    tasks = []
    for lo in range(0, n, chunk):
        hi = min(lo + chunk, n)
        need = sorted(set(p1[lo:hi]) | set(p2[lo:hi]))
        remap = {q: j for j, q in enumerate(need)}
        tasks.append(([names[q] for q in need], [attrs[q] for q in need],
                      [cats[q] for q in need],
                      [remap[q] for q in p1[lo:hi]], [remap[q] for q in p2[lo:hi]]))
    log(f"  задач: {len(tasks)} на {n_jobs} процессов")

    t0 = time.time()
    ta: list[str] = []
    tb: list[str] = []
    with ProcessPoolExecutor(max_workers=n_jobs, initializer=_init_worker,
                             initargs=(key_idf, cfg.to_dict())) as ex:
        for k, (x, z) in enumerate(ex.map(_work, tasks)):
            ta.extend(x)
            tb.extend(z)
            if k % 10 == 0:
                log(f"    готово задач {k + 1}/{len(tasks)} ({time.time() - t0:.0f}s)")
    log(f"  сериализация: {len(ta):,} пар за {time.time() - t0:.1f}s "
        f"({len(ta) / max(time.time() - t0, 1e-9):,.0f} пар/с)")
    return ta, tb


def main() -> None:
    args = parse_args()
    t_start = time.time()

    def log(msg: str) -> None:
        print(f"[{time.time() - t_start:6.1f}s] {msg}", flush=True)

    canon = bool(args.canon)
    space = "canon" if canon else "raw"
    n_jobs = resolve_n_jobs(args.n_jobs)

    out = args.out
    if out is None:
        midf = ""
        if str(args.missing_key_idf).strip() not in ("1.0", "1"):
            midf = "_midf" + str(args.missing_key_idf).strip().replace(".", "p")
        if args.split == "human":
            out = ARTIFACTS / f"ce_text_human_{space}_k{args.max_keys}{midf}.parquet"
        else:
            out = ARTIFACTS / (f"ce_text_llm_{args.llm_pairs // 1000}k_"
                               f"{space}_k{args.max_keys}{midf}.parquet")
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)

    log(f"split={args.split} canon={canon} max_keys={args.max_keys} n_jobs={n_jobs}")
    log(f"выход: {out}")

    if args.split == "human":
        items_path, matches_path = DATA / "items_human.parquet", DATA / "matches.parquet"
    else:
        items_path, matches_path = DATA / "items.parquet", DATA / "matches_llm.parquet"
    for p in (items_path, matches_path):
        if not p.is_file():
            raise SystemExit(f"нет файла {p}: сначала src/scripts/01_fetch_data.py")

    import numpy as np
    import pyarrow as pa
    import pyarrow.parquet as pq

    id1, id2, target = read_matches(matches_path)
    log(f"пар в {matches_path.name}: {len(id1):,}")
    if args.split == "llm":
        id1, id2, target = sample_llm(id1, id2, target, args, log)
    if args.limit:
        id1, id2, target = id1[:args.limit], id2[:args.limit], target[:args.limit]
        log(f"ограничение --limit: осталось {len(id1):,} пар")

    if args.split == "human":
        ids, names, attrs, cats = read_items(items_path)
    else:
        ids, names, attrs, cats = load_items_for(
            set(id1) | set(id2), items_path, log)
    log(f"товаров в каталоге: {len(ids):,}")

    pos = {v: i for i, v in enumerate(ids)}
    keep = [i for i, (a, b) in enumerate(zip(id1, id2)) if a in pos and b in pos]
    if len(keep) != len(id1):
        log(f"ВНИМАНИЕ: {len(id1) - len(keep):,} пар без карточки товара отброшено")
    if not keep:
        raise SystemExit("не осталось ни одной пары с известными товарами")
    id1 = [id1[i] for i in keep]
    id2 = [id2[i] for i in keep]
    target = np.asarray(target, dtype=np.float32)[keep]
    p1 = [pos[v] for v in id1]
    p2 = [pos[v] for v in id2]

    cfg = SerializeConfig(canon=canon, max_keys=args.max_keys,
                          max_val_chars=args.max_val_chars)

    key_idf_path = args.key_idf
    if key_idf_path is None and args.split == "llm":
        key_idf_path = ARTIFACTS / key_idf_name(space, args.min_df)
        if not key_idf_path.is_file():
            raise SystemExit(
                f"нет {key_idf_path}. Словарь IDF ключей считается по человеческому "
                f"каталогу; сначала выполните тот же вызов с --split human "
                f"(и теми же --canon и --min-df), затем повторите для llm")

    if key_idf_path is not None:
        key_idf = json.loads(Path(key_idf_path).read_text(encoding="utf-8"))
        log(f"key_idf прочитан из {key_idf_path}: {len(key_idf):,} ключей")
    else:
        t0 = time.time()
        parsed = (parse_item(names[i], attrs[i]) for i in range(len(names)))
        key_idf = key_idf_from_parsed(parsed, cfg, min_df=args.min_df)
        key_idf_path = ARTIFACTS / key_idf_name(space, args.min_df)
        key_idf_path.parent.mkdir(parents=True, exist_ok=True)
        if key_idf_path.is_file():
            was = json.loads(key_idf_path.read_text(encoding="utf-8"))
            if was != key_idf:
                raise SystemExit(
                    f"{key_idf_path} уже существует и отличается от только что "
                    f"посчитанного ({len(was):,} ключей против {len(key_idf):,}). "
                    f"Этим словарём, возможно, отранжированы ключи в уже обученных "
                    f"чекпойнтах. Уберите файл руками, если он больше не нужен, "
                    f"либо укажите другой каталог через --key-idf")
            log(f"key_idf совпал с уже лежащим {key_idf_path}")
        key_idf_path.write_text(json.dumps(key_idf, ensure_ascii=False),
                                encoding="utf-8")
        log(f"key_idf посчитан ({space}, min_df={args.min_df}): {len(key_idf):,} "
            f"ключей за {time.time() - t0:.0f}s -> {key_idf_path}")

    sug = suggested_missing_idf(key_idf)
    if args.missing_key_idf == "auto":
        cfg = replace(cfg, missing_key_idf=sug)
        log(f"missing_key_idf=auto -> {sug:.3f}")
    else:
        cfg = replace(cfg, missing_key_idf=float(args.missing_key_idf))
        if abs(cfg.missing_key_idf - sug) > 1.0:
            log(f"ЗАМЕТЬТЕ: missing_key_idf={cfg.missing_key_idf:g}, а по словарю "
                f"уместнее {sug:.3f}. Значение 1.0 историческое: редкий ключ "
                f"ранжируется как очень частый и вылетает из среза max_keys. "
                f"Менять можно только вместе с переобучением")

    ta, tb = serialize(names, attrs, cats, p1, p2, key_idf, cfg, n_jobs, log)

    from src.ecup.folds import first_tokens
    pair_cat = [cats[q] for q in p1]
    brand = first_tokens(names[q] for q in p1)

    table = pa.table({
        "id1": pa.array(id1, type=pa.int64()),
        "id2": pa.array(id2, type=pa.int64()),
        "target": pa.array(target.tolist(), type=pa.float32()),
        "category": pa.array(pair_cat, type=pa.string()),
        "brand": pa.array(brand, type=pa.string()),
        "text_a": pa.array(ta, type=pa.string()),
        "text_b": pa.array(tb, type=pa.string()),
    })
    pq.write_table(table, out, compression=args.compression)

    md = meta_dir(out)
    cfg.save(md)
    lens = np.fromiter((len(x) + len(z) for x, z in zip(ta, tb)),
                       dtype=np.int64, count=len(ta))
    stats = {
        "split": args.split,
        "source_items": str(items_path),
        "source_matches": str(matches_path),
        "key_idf": str(key_idf_path),
        "key_idf_name": Path(key_idf_path).name,
        "key_idf_keys": len(key_idf),
        "key_idf_min_df": args.min_df,
        "suggested_missing_key_idf": sug,
        "pairs": len(ta),
        "target_mean": float(target.mean()),
        "chars_p50": int(np.percentile(lens, 50)),
        "chars_p90": int(np.percentile(lens, 90)),
        "chars_p99": int(np.percentile(lens, 99)),
        "n_jobs": n_jobs,
        "argv": sys.argv,
    }
    (md / "prep.json").write_text(
        json.dumps(stats, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    log(f"записано {len(ta):,} пар в {out} ({out.stat().st_size / 1e6:.0f} МБ)")
    log(f"конфигурация сериализации: {md / 'serialize_config.json'}")
    log(f"символов на пару: p50={stats['chars_p50']} p90={stats['chars_p90']} "
        f"p99={stats['chars_p99']} | средний таргет {stats['target_mean']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/scripts/13_prep_llm_chunked.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import importlib.util
import json
import sys
import time
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from src.ecup.config import ARTIFACTS, DATA, SerializeConfig
from src.ecup.folds import first_tokens
from src.ecup.serialize import serialize_pairs


def take_from_10():
    p = ROOT / "src" / "scripts" / "10_prep_text.py"
    spec = importlib.util.spec_from_file_location("prep10", p)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(prog="13_prep_llm_chunked.py")
    p.add_argument("--chunks", type=int, default=4,
                   help="на сколько частей резать пары (по умолчанию: 4)")
    p.add_argument("--llm-pairs", dest="llm_pairs", type=int, default=11_200_000)
    p.add_argument("--canon", type=int, choices=(0, 1), default=0)
    p.add_argument("--max-keys", dest="max_keys", type=int, default=16)
    p.add_argument("--max-val-chars", dest="max_val_chars", type=int, default=60)
    p.add_argument("--min-df", dest="min_df", type=int, default=5)
    p.add_argument("--share-grey", dest="share_grey", type=float, default=0.40)
    p.add_argument("--share-pos", dest="share_pos", type=float, default=0.20)
    p.add_argument("--seed", type=int, default=7)
    p.add_argument("--out", type=Path, default=None)
    p.add_argument("--compression", default="zstd")
    return p.parse_args()


def main() -> int:
    args = parse_args()
    import numpy as np
    import pyarrow as pa
    import pyarrow.parquet as pq

    M = take_from_10()
    t_start = time.time()

    def log(msg: str) -> None:
        print(f"[{time.time() - t_start:7.1f}s] {msg}", flush=True)

    space = "canon" if args.canon else "raw"
    cfg = SerializeConfig(canon=bool(args.canon), max_keys=args.max_keys,
                          max_val_chars=args.max_val_chars)
    kidf = ARTIFACTS / M.key_idf_name(space, args.min_df)
    if not kidf.is_file():
        raise SystemExit(
            f"нет словаря {kidf}. Сначала соберите человеческий корпус тем же "
            f"пространством ключей: 10_prep_text.py --split human --canon {args.canon}")
    key_idf = json.loads(kidf.read_text(encoding="utf-8"))
    log(f"словарь IDF: {len(key_idf):,} ключей из {kidf.name}")

    m = pq.read_table(DATA / "matches_llm.parquet", columns=["id1", "id2", "target"])
    id1 = m.column("id1").to_numpy(zero_copy_only=False)
    id2 = m.column("id2").to_numpy(zero_copy_only=False)
    y = m.column("target").to_numpy(zero_copy_only=False).astype(np.float32)
    del m
    log(f"пар в matches_llm: {len(y):,}")

    id1, id2, y = M.sample_llm(id1, id2, y, args, log)
    log(f"отобрано пар: {len(y):,}, средний таргет {y.mean():.4f}")

    out = args.out or ARTIFACTS / f"ce_text_llm_{len(y) // 1000}k_{space}_k{args.max_keys}.parquet"
    out = Path(out)
    parts = []
    bounds = np.linspace(0, len(y), args.chunks + 1).astype(int)

    for ch in range(args.chunks):
        a, b = bounds[ch], bounds[ch + 1]
        p1, p2, yy = id1[a:b], id2[a:b], y[a:b]
        wanted = set(int(v) for v in p1) | set(int(v) for v in p2)
        log(f"--- часть {ch + 1}/{args.chunks}: {len(yy):,} пар, "
            f"{len(wanted):,} карточек ({100 * len(wanted) / 12_384_610:.0f}% каталога)")
        ids, names, attrs, cats = M.load_items_for(wanted, DATA / "items.parquet", log)
        pos_s = {int(v): i for i, v in enumerate(ids)}
        del ids
        i1 = np.fromiter((pos_s[int(v)] for v in p1), dtype=np.int64, count=len(p1))
        i2 = np.fromiter((pos_s[int(v)] for v in p2), dtype=np.int64, count=len(p2))
        del pos_s, wanted

        cat_ = [cats[i] for i in i1]
        brand = first_tokens(names[q] for q in i1)
        ta, tb = serialize_pairs(names, attrs, cats, i1, i2, key_idf, cfg,
                                 log=log, release_raw=True)
        del names, attrs, cats

        path_ = out.with_name(out.stem + f".part{ch}.parquet")
        pq.write_table(pa.table({
            "id1": pa.array(p1, pa.int64()), "id2": pa.array(p2, pa.int64()),
            "target": pa.array(yy, pa.float32()),
            "category": pa.array(cat_, pa.large_string()),
            "brand": pa.array(brand, pa.large_string()),
            "text_a": pa.array(ta, pa.large_string()),
            "text_b": pa.array(tb, pa.large_string()),
        }), path_, compression=args.compression)
        parts.append(path_)
        del ta, tb, brand, cat_, i1, i2
        log(f"    часть {ch + 1} записана: {path_.stat().st_size / 2**20:.0f} МБ")

    log("склеиваю части")
    w = None
    for path_ in parts:
        t = pq.read_table(path_)
        if w is None:
            w = pq.ParquetWriter(out, t.schema, compression=args.compression)
        w.write_table(t)
        del t
    w.close()
    for path_ in parts:
        path_.unlink()

    meta_ = Path(str(out) + ".meta")
    meta_.mkdir(parents=True, exist_ok=True)
    cfg.save(meta_)
    (meta_ / "prep.json").write_text(json.dumps({
        "split": "llm", "n_pairs": int(len(y)), "chunks": args.chunks,
        "key_idf": str(kidf), "key_idf_name": kidf.name,
        "share_grey": args.share_grey, "share_pos": args.share_pos,
        "seed": args.seed, "made_by": "13_prep_llm_chunked.py",
    }, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    n = pq.ParquetFile(out).metadata.num_rows
    log(f"готово: {n:,} пар -> {out} ({out.stat().st_size / 2**20:.0f} МБ)")
    log(f"спутник: {meta_}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile src/scripts/20_pretrain.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import os
import random
import shutil
import sys
import time
from contextlib import nullcontext
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from src.ecup.config import ARTIFACTS, SerializeConfig


def meta_dir(text_path: Path) -> Path:
    return Path(str(text_path) + ".meta")


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="20_pretrain.py",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        description=(
            "Предобучает кросс-энкодер на мягких LLM-метках (стадия A).\n"
            "Валидация — на человеческих парах одного фолда невиданных брендов;\n"
            "печатаются macro-AP и ожидаемый лидерборд.\n"
            "Рядом с чекпойнтом сохраняются serialize_config.json (формат текста)\n"
            "и train_meta.json (полная команда запуска и метрики)."),
        epilog=(
            "Пример (рецепт лучшей измеренной модели). Пути даны от ECUP_ROOT:\n"
            "artifacts/ лежит рядом с каталогом v2, а не внутри него.\n"
            "  src/scripts/20_pretrain.py \\\n"
            '      --train-text "$ECUP_ROOT"/artifacts/ce_text_llm_2500k_canon_k16.parquet \\\n'
            '      --val-text   "$ECUP_ROOT"/artifacts/ce_text_human_canon_k16.parquet \\\n'
            '      --max-len 256 --batch 48 --lr 3e-5 --max-steps 15000 \\\n'
            '      --out "$ECUP_ROOT"/artifacts/ce_pre'))
    p.add_argument("--train-text", dest="train_text", type=Path, required=True,
                   help="parquet с LLM-текстами из 10_prep_text.py --split llm")
    p.add_argument("--val-text", dest="val_text", type=Path, required=True,
                   help="parquet с человеческими текстами: только по ним считается "
                        "метрика, сопоставимая с лидербордом")
    p.add_argument("--val-fold", dest="val_fold", type=int, default=0,
                   help="номер фолда невиданных брендов для валидации (по умолчанию: 0)")
    p.add_argument("--n-folds", dest="n_folds", type=int, default=5,
                   help="число фолдов в разбиении (по умолчанию: 5)")
    p.add_argument("--model", default="deepvk/RuModernBERT-base",
                   help="исходные веса (по умолчанию: deepvk/RuModernBERT-base)")
    p.add_argument("--out", type=Path, default=None,
                   help="каталог чекпойнта (по умолчанию: ARTIFACTS/ce_pre)")
    p.add_argument("--key-idf", dest="key_idf", type=Path, default=None,
                   help="словарь IDF ключей; по умолчанию берётся из спутника "
                        "обучающего текста (<текст>.meta/prep.json) и кладётся "
                        "в каталог чекпойнта")
    p.add_argument("--max-len", dest="max_len", type=int, default=256,
                   help="длина в токенах (по умолчанию: 256). Значение едет "
                        "в train_meta.json и сверяется при сборке архива и на "
                        "инференсе: обрезка меняет то, что видит модель")
    p.add_argument("--batch", type=int, default=48, help="размер батча (по умолчанию: 48)")
    p.add_argument("--epochs", type=int, default=1, help="эпох (по умолчанию: 1)")
    p.add_argument("--lr", type=float, default=3e-5, help="скорость обучения (по умолчанию: 3e-5)")
    p.add_argument("--warmup", type=float, default=0.03,
                   help="доля шагов на прогрев (по умолчанию: 0.03)")
    p.add_argument("--weight-decay", dest="weight_decay", type=float, default=0.01)
    p.add_argument("--clip", type=float, default=1.0, help="обрезка нормы градиента")
    p.add_argument("--max-steps", dest="max_steps", type=int, default=0,
                   help="остановиться после N шагов оптимизатора: воспроизводимая "
                        "замена принудительному обрыву процесса (0 — до конца)")
    p.add_argument("--eval-every", dest="eval_every", type=int, default=2000,
                   help="как часто валидироваться, в шагах (по умолчанию: 2000)")
    p.add_argument("--eval-max-pairs", dest="eval_max_pairs", type=int, default=0,
                   help="ограничить валидацию N парами (0 — весь фолд)")
    p.add_argument("--workers", type=int, default=6,
                   help="процессов загрузки данных (по умолчанию: 6)")
    p.add_argument("--swap-aug", dest="swap_aug", type=int, choices=(0, 1), default=1,
                   help="перестановка сторон пары: стороны статистически симметричны "
                        "(12.61 против 12.59 ключей), перестановка законна (по умолчанию: 1)")
    p.add_argument("--label-smooth", dest="label_smooth", type=float, default=0.0,
                   help="сглаживание меток: LLM-разметка шумная (по умолчанию: 0)")
    p.add_argument("--grad-ckpt", dest="grad_ckpt", action="store_true",
                   help="контрольные точки градиента: медленнее, но экономит память")
    p.add_argument("--device", default="auto", choices=("auto", "cuda", "cpu"),
                   help="cpu пригоден только для крошечной проверки формата")
    p.add_argument("--seed", type=int, default=99)
    p.add_argument("--limit", type=int, default=0,
                   help="взять только первые N обучающих пар (проверка запуска)")
    p.add_argument("--bucket", type=int, choices=(0, 1), default=1,
                   help="батчи, однородные по длине (по умолчанию: 1). 39%% "
                        "вычислений уходило в паддинг; см. src/ecup/bucket.py")
    p.add_argument("--bucket-window", dest="bucket_window", type=int, default=64,
                   help="ширина окна сортировки в батчах (по умолчанию: 64)")
    p.add_argument("--save-every", dest="save_every", type=int, default=2000,
                   help="сохранять состояние для продолжения каждые N шагов "
                        "(по умолчанию: 2000). Сервер исчезает без предупреждения, "
                        "и прогон может прерваться в любой момент")
    p.add_argument("--resume", action="store_true",
                   help="продолжить с последнего состояния в <out>/resume: веса, "
                        "оптимизатор, схедулер и номер шага")
    p.add_argument("--on-save", dest="on_save", default="",
                   help="команда, запускаемая после каждого сохранения "
                        "(бэкап чекпойнта на сторону); падение команды только "
                        "печатается, обучение не останавливает")
    return p.parse_args()


def read_text_table(path: Path, need_brand: bool = False) -> dict:
    import numpy as np
    import pyarrow.parquet as pq

    from src.ecup.arrowstr import ArrowStrings, dedup_strings, load_texts

    if not path.is_file():
        raise SystemExit(f"нет файла с текстами: {path} (сделайте src/scripts/10_prep_text.py)")
    cols = set(pq.ParquetFile(path).schema_arrow.names)
    required = {"target", "category", "text_a", "text_b"} | ({"brand"} if need_brand else set())
    missing = sorted(required - cols)
    if missing:
        raise SystemExit(
            f"в {path} нет колонок {missing}. Файл сделан старой версией "
            f"10_prep_text.py — пересоберите его, а не подставляйте замену: "
            f"brand задаёт разбиение по невиданным брендам")
    needed = ["target", "category", "text_a", "text_b"] + (["brand"] if need_brand else [])
    t = load_texts(path, columns=needed)
    out = {
        "_table": t,
        "target": t.column("target").to_numpy(zero_copy_only=False).astype(np.float32),
        "category": dedup_strings(t.column("category")),
        "text_a": ArrowStrings(t.column("text_a")),
        "text_b": ArrowStrings(t.column("text_b")),
    }
    if need_brand:
        out["brand"] = dedup_strings(t.column("brand"))
    return out


def load_matching_cfg(train_text: Path, val_text: Path) -> SerializeConfig:
    from src.ecup.serialize import check_cfg_matches_code

    cfg_tr = SerializeConfig.load(meta_dir(train_text))
    cfg_va = SerializeConfig.load(meta_dir(val_text))
    check_cfg_matches_code(cfg_tr)
    if cfg_tr != cfg_va:
        raise SystemExit(
            f"конфигурации сериализации разошлись:\n"
            f"  обучение   {train_text}: {cfg_tr.to_dict()}\n"
            f"  валидация  {val_text}: {cfg_va.to_dict()}\n"
            f"Пересоберите оба корпуса одним вызовом 10_prep_text.py с одними "
            f"и теми же --canon/--max-keys/--max-val-chars")
    return cfg_tr


def build_dataset_class():
    import torch
    from torch.utils.data import Dataset

    class Pairs(Dataset):

        def __init__(self, ta, tb, y, tok, max_len, swap_aug=False, seed=0):
            self.ta, self.tb, self.y = ta, tb, y
            self.tok, self.max_len = tok, max_len
            self.swap_aug = swap_aug
            self.seed = seed
            self._rng = None

        def __len__(self):
            return len(self.ta)

        def _rng_here(self):
            if self._rng is None:
                info = torch.utils.data.get_worker_info()
                self._rng = random.Random(self.seed * 1_000_003 + (info.id if info else 0))
            return self._rng

        def __getitem__(self, i):
            a, b = self.ta[i], self.tb[i]
            if self.swap_aug and self._rng_here().random() < 0.5:
                a, b = b, a
            return a, b, self.y[i]

        def collate(self, batch):
            a, b, y = zip(*batch)
            enc = self.tok(list(a), list(b), padding=True, truncation=True,
                           max_length=self.max_len, return_tensors="pt")
            enc["labels"] = torch.tensor(y, dtype=torch.float32)
            return enc

    return Pairs


def evaluate(model, dl, accepted, device, amp):
    import numpy as np
    import torch

    model.eval()
    preds = []
    with torch.inference_mode():
        for batch in dl:
            batch.pop("labels")
            batch = {k: v.to(device, non_blocking=True)
                     for k, v in batch.items() if k in accepted}
            with amp():
                preds.append(model(**batch).logits.squeeze(-1).float().cpu())
    model.train()
    return np.concatenate([p.numpy() for p in preds])


def key_idf_of(text_path: Path, override: Path | None = None) -> Path:
    if override is not None:
        if not Path(override).is_file():
            raise SystemExit(f"нет словаря IDF {override} (--key-idf)")
        return Path(override)
    p = meta_dir(text_path) / "prep.json"
    if not p.is_file():
        raise SystemExit(
            f"нет {p}: неизвестно, каким словарём IDF отранжированы ключи "
            f"в {text_path}. Пересоберите тексты src/scripts/10_prep_text.py "
            f"или укажите словарь флагом --key-idf")
    prep = json.loads(p.read_text(encoding="utf-8"))
    cands = [prep.get("key_idf"), prep.get("key_idf_name")]
    for v in cands:
        if not v:
            continue
        for c in (Path(v), ARTIFACTS / Path(v).name):
            if c.is_file():
                return c
    raise SystemExit(
        f"словарь IDF из {p} не найден: пробовал {[str(v) for v in cands if v]} "
        f"и те же имена в {ARTIFACTS}. Без него чекпойнт уедет без key_idf.json, "
        f"а сборщик подставит чужой словарь — ключи отранжируются иначе, чем "
        f"при обучении, и текст разойдётся молча. Укажите файл флагом --key-idf")


def save_checkpoint(out: Path, model, tok, cfg: SerializeConfig, meta: dict,
                    key_idf: Path | None = None) -> None:
    out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out), safe_serialization=True)
    tok.save_pretrained(str(out))
    cfg.save(out)
    if key_idf is None or not Path(key_idf).is_file():
        raise SystemExit(
            f"нет словаря IDF {key_idf}: чекпойнт без key_idf.json неполон")
    shutil.copy2(key_idf, out / "key_idf.json")
    (out / "train_meta.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8")


def bind_padding(model, tok, log=print) -> None:
    if getattr(model.config, "pad_token_id", None) is not None:
        return
    pid = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    if pid is None:
        raise SystemExit(
            f"у {type(model).__name__} нет pad_token_id, и токенизатор его не "
            f"знает — батч больше единицы собрать нельзя")
    model.config.pad_token_id = int(pid)
    log(f"токен паддинга не был задан, беру из токенизатора: {pid}")


RESUME_NAME = "resume"


def save_resume(out: Path, model, opt, sched, step: int, epoch: int,
                best: float, best_step: int, log) -> None:
    import torch

    d = out / RESUME_NAME
    d.mkdir(parents=True, exist_ok=True)
    tmp = d / "state.pt.tmp"
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                "sched": sched.state_dict(), "step": step, "epoch": epoch,
                "best": best, "best_step": best_step}, tmp)
    tmp.replace(d / "state.pt")
    log(f"     состояние для продолжения сохранено (шаг {step:,})")


def load_resume(out: Path, model, opt, sched, log):
    import torch

    p = out / RESUME_NAME / "state.pt"
    if not p.is_file():
        log(f"--resume задан, но {p} нет — начинаю с нуля")
        return 0, 0, -1.0, -1
    st = torch.load(p, map_location="cpu", weights_only=False)
    model.load_state_dict(st["model"])
    opt.load_state_dict(st["opt"])
    sched.load_state_dict(st["sched"])
    log(f"продолжаю с шага {st['step']:,} (эпоха {st['epoch']}, "
        f"лучший macro-AP {st['best']:.4f} на шаге {st['best_step']:,})")
    return st["step"], st["epoch"], st["best"], st["best_step"]


def run_on_save(cmd: str, log) -> None:
    if not cmd:
        return
    import subprocess

    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=1800)
        if r.returncode != 0:
            log(f"     --on-save вернул {r.returncode}: {(r.stderr or r.stdout)[-400:]}")
        else:
            log("     --on-save выполнен")
    except Exception as e:
        log(f"     --on-save не выполнен: {type(e).__name__}: {e}")


def main() -> None:
    args = parse_args()
    t_start = time.time()

    def log(msg: str) -> None:
        if int(os.environ.get("RANK", 0)) == 0:
            print(f"[{time.time() - t_start:7.1f}s] {msg}", flush=True)

    os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
    os.environ.setdefault("HF_HOME", str(ARTIFACTS.parent / ".hf_home"))

    out = Path(args.out) if args.out else ARTIFACTS / "ce_pre"
    cfg = load_matching_cfg(args.train_text, args.val_text)
    log(f"формат текста: {cfg.to_dict()}")
    kidf = key_idf_of(args.train_text, args.key_idf)
    log(f"словарь IDF: {kidf}")
    log(f"чекпойнт: {out}")

    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              get_cosine_schedule_with_warmup)

    from src.ecup import lb
    from src.ecup.folds import make_unseen_folds
    from src.ecup.metric import macro_pr_auc

    rank = int(os.environ.get("RANK", 0))
    world = int(os.environ.get("WORLD_SIZE", 1))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    ddp = world > 1
    main_ = rank == 0
    if ddp:
        import torch.distributed as dist
        dist.init_process_group(backend="nccl")
        torch.cuda.set_device(local_rank)
        log(f"распределённый режим: карта {rank + 1} из {world}, "
            f"эффективный батч {args.batch * world}")

    device = args.device
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if ddp:
        device = f"cuda:{local_rank}"
    if device == "cpu":
        log("ВНИМАНИЕ: обучение на cpu. Это пригодно только для проверки запуска")
    log(f"устройство: {device}")
    torch.manual_seed(args.seed)
    if device == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    tr = read_text_table(args.train_text)
    ytr = tr["target"]
    if ytr.min() < 0.0 or ytr.max() > 1.0:
        raise SystemExit(f"таргеты вне [0,1]: min={ytr.min()} max={ytr.max()}")
    if args.label_smooth > 0:
        ytr = ytr * (1 - args.label_smooth) + 0.5 * args.label_smooth
    if args.limit:
        ytr = ytr[:args.limit]
        tr["text_a"] = tr["text_a"][:args.limit]
        tr["text_b"] = tr["text_b"][:args.limit]
    soft = int(np.sum((ytr > 0) & (ytr < 1)))
    log(f"обучение: {len(ytr):,} пар, средний таргет {ytr.mean():.4f}, "
        f"мягких (0<y<1) {soft:,} — они и есть смысл стадии A")

    va = read_text_table(args.val_text, need_brand=True)
    fold = make_unseen_folds(va["category"], va["brand"], n_folds=args.n_folds)
    vm = np.where(fold == args.val_fold)[0]
    if args.eval_max_pairs and len(vm) > args.eval_max_pairs:
        vm = vm[:args.eval_max_pairs]
    yva = va["target"][vm]
    if not np.all((yva == 0) | (yva == 1)):
        raise SystemExit("в валидационном корпусе таргеты не 0/1: это не человеческая разметка")
    cva = va["category"][vm]
    log(f"валидация: {len(vm):,} человеческих пар, фолд {args.val_fold} "
        f"из {args.n_folds} (невиданные бренды), доля позитивов {yva.mean():.4f}")

    tok = AutoTokenizer.from_pretrained(args.model)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            args.model, num_labels=1, dtype=torch.float32,
            attn_implementation="sdpa").to(device)
    except Exception as e:
        log(f"sdpa не подошёл ({type(e).__name__}: {e}), беру eager")
        model = AutoModelForSequenceClassification.from_pretrained(
            args.model, num_labels=1, dtype=torch.float32,
            attn_implementation="eager").to(device)
    bind_padding(model, tok, log)
    if args.grad_ckpt:
        model.gradient_checkpointing_enable()
    core = model
    if ddp:
        from torch.distributed.algorithms.ddp_comm_hooks import default_hooks
        from torch.nn.parallel import DistributedDataParallel
        model = DistributedDataParallel(model, device_ids=[local_rank],
                                        output_device=local_rank,
                                        gradient_as_bucket_view=True)
        model.register_comm_hook(None, default_hooks.bf16_compress_hook)
        log("градиенты пересылаются в bf16 (обмен вдвое дешевле)")

    import inspect
    accepted = set(inspect.signature(core.forward).parameters)
    log(f"модель принимает: "
        f"{sorted(accepted & {'input_ids', 'attention_mask', 'token_type_ids', 'position_ids'})}")

    Pairs = build_dataset_class()
    ds_tr = Pairs(tr["text_a"], tr["text_b"], ytr, tok, args.max_len,
                  swap_aug=bool(args.swap_aug), seed=args.seed)
    ds_va = Pairs([va["text_a"][i] for i in vm], [va["text_b"][i] for i in vm],
                  yva.astype(np.float32), tok, args.max_len)
    sampler = None
    if args.bucket:
        from src.ecup.bucket import build_sampler_class, padding_share, pair_lengths

        lens = pair_lengths(tr["text_a"], tr["text_b"])
        sampler = build_sampler_class()(
            lens, args.batch, seed=args.seed,
            window_batches=args.bucket_window, drop_last=True,
            rank=rank, world_size=world)
        share = padding_share(lens, list(sampler))
        log(f"батчи по длине: {len(sampler):,} батчей, паддинг {share:.1%} "
            f"(при случайном порядке было бы около 39%)")
        dl_tr = DataLoader(ds_tr, batch_sampler=sampler,
                           num_workers=args.workers, collate_fn=ds_tr.collate,
                           pin_memory=(device == "cuda"),
                           persistent_workers=args.workers > 0)
    else:
        dl_tr = DataLoader(ds_tr, batch_size=args.batch, shuffle=True,
                           num_workers=args.workers, collate_fn=ds_tr.collate,
                           pin_memory=(device == "cuda"), drop_last=True,
                           persistent_workers=args.workers > 0)
    dl_va = DataLoader(ds_va, batch_size=args.batch * 2, shuffle=False,
                       num_workers=min(args.workers, 4), collate_fn=ds_va.collate,
                       pin_memory=(device == "cuda"))

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                            weight_decay=args.weight_decay)
    total = len(dl_tr) * args.epochs
    if args.max_steps:
        total = min(total, args.max_steps)
    sched = get_cosine_schedule_with_warmup(opt, int(args.warmup * total), total)
    lossf = torch.nn.BCEWithLogitsLoss()
    log(f"шагов всего: {total:,} (прогрев {int(args.warmup * total):,})")

    def amp():
        return (torch.autocast("cuda", dtype=torch.bfloat16) if device == "cuda"
                else nullcontext())

    def validate(step: int, tag: str) -> float:
        p = evaluate(model, dl_va, accepted, device, amp)
        ap = macro_pr_auc(yva.astype(int), p, cva)
        log(f"  {tag}: macro-AP={ap:.4f} | отношение к Jaccard фолда "
            f"{args.val_fold} = {ap / lb.JACCARD_UNSEEN_FOLDS[args.val_fold]:.4f} "
            f"(шаг {step:,})")
        return ap

    best = -1.0
    best_step = -1
    step = 0
    ep0 = 0
    if args.resume:
        step, ep0, best, best_step = load_resume(out, model, opt, sched, log)
    stop = False
    skip = step - ep0 * len(dl_tr)
    for ep in range(ep0, args.epochs):
        model.train()
        if sampler is not None:
            sampler.set_epoch(ep)
            if skip > 0:
                log(f"пропускаю {skip:,} уже пройденных батчей эпохи {ep}")
                sampler._batches = sampler._batches[skip:]
                skip = 0
        run = 0.0
        seen = 0
        for batch in dl_tr:
            if sampler is None and skip > 0:
                skip -= 1
                continue
            labels = batch.pop("labels").to(device, non_blocking=True)
            batch = {k: v.to(device, non_blocking=True)
                     for k, v in batch.items() if k in accepted}
            with amp():
                logits = model(**batch).logits.squeeze(-1)
                loss = lossf(logits.float(), labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            run += float(loss.item())
            seen += 1
            step += 1
            if step % 500 == 0:
                log(f"  эпоха {ep} шаг {step:,}/{total:,} loss={run / seen:.4f}")
            if args.eval_every and step % args.eval_every == 0:
                ap = validate(step, "промежуточная проверка") if main_ else -1.0
                if ddp:
                    import torch.distributed as dist
                    dist.barrier()
                if main_ and ap > best:
                    best, best_step = ap, step
                    save_checkpoint(out, core, tok, cfg, {
                        "stage": "A", "argv": sys.argv, "best_macro_ap": best,
                        "ratio_to_jaccard": best / lb.JACCARD_UNSEEN_FOLDS[args.val_fold],
                        "jaccard_of_fold": lb.JACCARD_UNSEEN_FOLDS[args.val_fold], "step": step, "epoch": ep,
                        "train_text": str(args.train_text), "val_text": str(args.val_text),
                        "val_fold": args.val_fold, "serialize_config": cfg.to_dict(),
                        "max_len": args.max_len, "batch": args.batch, "lr": args.lr,
                        "key_idf": str(kidf)}, key_idf=kidf)
                    log(f"     сохранён как лучший -> {out}")
                    run_on_save(args.on_save, log)
            if args.save_every and step % args.save_every == 0:
                if main_:
                    save_resume(out, core, opt, sched, step, ep, best, best_step, log)
                if ddp:
                    import torch.distributed as dist
                    dist.barrier()
            if args.max_steps and step >= args.max_steps:
                log(f"достигнут --max-steps={args.max_steps:,}, останавливаюсь")
                stop = True
                break
        if stop:
            break

    ap = validate(step, "итоговая проверка") if main_ else -1.0
    meta_final = {
        "stage": "A", "argv": sys.argv,
        "ratio_to_jaccard": ap / lb.JACCARD_UNSEEN_FOLDS[args.val_fold],
        "jaccard_of_fold": lb.JACCARD_UNSEEN_FOLDS[args.val_fold],
        "step": step, "epoch": args.epochs - 1,
        "train_text": str(args.train_text), "val_text": str(args.val_text),
        "val_fold": args.val_fold, "serialize_config": cfg.to_dict(),
        "max_len": args.max_len, "batch": args.batch, "lr": args.lr,
        "key_idf": str(kidf)}
    if main_:
        tail_rows = out.with_name(out.name + "_last")
        save_checkpoint(tail_rows, core, tok, cfg,
                        {**meta_final, "selection": "last", "macro_ap": ap},
                        key_idf=kidf)
        log(f"последние веса (шаг {step:,}, macro-AP {ap:.4f}) -> {tail_rows}")
    if main_ and ap > best:
        best, best_step = ap, step
        save_checkpoint(out, core, tok, cfg,
                        {**meta_final, "selection": "best", "best_macro_ap": best,
                         "ratio_to_jaccard": best / lb.JACCARD_UNSEEN_FOLDS[args.val_fold]},
                        key_idf=kidf)
    if ddp:
        import torch.distributed as dist
        dist.barrier()
        dist.destroy_process_group()
    if not main_:
        return
    if best < 0:
        raise SystemExit("ни одной валидации не выполнено: чекпойнт не сохранён")

    log(f"ЛУЧШИЙ macro-AP стадии A: {best:.4f} на шаге {best_step:,} "
        f"(отношение к Jaccard фолда {args.val_fold}: "
        f"{best / lb.JACCARD_UNSEEN_FOLDS[args.val_fold]:.4f})")
    log(f"чекпойнт со всем сопровождением: {out}")
    log("дальше: src/scripts/30_finetune.py --init-from " + str(out))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/scripts/30_finetune.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import os
import random
import shutil
import sys
import time
from contextlib import nullcontext
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from src.ecup import losses
from src.ecup.config import ARTIFACTS, SerializeConfig


def meta_dir(text_path: Path) -> Path:
    return Path(str(text_path) + ".meta")


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="30_finetune.py",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        description=(
            "Дообучает кросс-энкодер на человеческих метках (стадия B).\n"
            "Разбиение — по невиданным брендам (категория + первый токен имени).\n"
            "После каждой эпохи печатаются macro-AP и ожидаемый лидерборд.\n"
            "Чекпойнт сохраняется вместе с serialize_config.json и командой запуска."),
        epilog=(
            "Пример (рецепт лучшей измеренной модели). Пути даны от ECUP_ROOT:\n"
            "artifacts/ лежит рядом с каталогом v2, а не внутри него.\n"
            "  src/scripts/30_finetune.py \\\n"
            '      --text "$ECUP_ROOT"/artifacts/ce_text_human_canon_k16.parquet \\\n'
            '      --init-from "$ECUP_ROOT"/artifacts/ce_pre \\\n'
            "      --folds 0 --epochs 2 --lr 2e-5 --batch 48 --max-len 320 --tag ce"))
    p.add_argument("--text", type=Path, required=True,
                   help="parquet с человеческими текстами из 10_prep_text.py --split human")
    p.add_argument("--init-from", dest="init_from", type=Path, default=None,
                   help="каталог чекпойнта стадии A; без него старт с --model")
    p.add_argument("--model", default="rmb-base",
                   help="бэкбон: алиас из src.ecup.backbones или любой id Hugging "
                        "Face. Используется, если не задан --init-from. "
                        "Список алиасов: --list-backbones")
    p.add_argument("--allow-text-change", dest="allow_text_change",
                   action="store_true",
                   help="разрешить дообучение на формате текста, отличном от "
                        "того, на котором обучена стадия A (например k24 поверх "
                        "k16). Это законно и является отдельной гипотезой; флаг "
                        "нужен, чтобы смена не прошла незамеченной")
    p.add_argument("--list-backbones", action="store_true",
                   help="показать реестр бэкбонов и выйти")
    p.add_argument("--tag", default="ce_ft",
                   help="префикс имён чекпойнтов: ARTIFACTS/<tag>_f<fold> (по умолчанию: ce_ft)")
    p.add_argument("--recipe", default=None,
                   help="имя рецепта в хранилище OOF (по умолчанию совпадает "
                        "с --tag). Рецепт — это НАБОР чекпойнтов по фолдам: "
                        "полное покрытие даёт только весь набор")
    p.add_argument("--folds", default="0",
                   help="какие фолды обучать, через запятую (по умолчанию: 0). "
                        "Полный OOF считается, когда обучены все")
    p.add_argument("--n-folds", dest="n_folds", type=int, default=5,
                   help="число фолдов разбиения (по умолчанию: 5)")
    p.add_argument("--key-idf", dest="key_idf", type=Path, default=None,
                   help="словарь IDF ключей; по умолчанию берётся из спутника "
                        "текста (<текст>.meta/prep.json) и кладётся в каталог "
                        "чекпойнта")
    p.add_argument("--max-len", dest="max_len", type=int, default=320,
                   help="длина в токенах (по умолчанию: 320 — под max_keys=16). "
                        "Значение едет в train_meta.json и сверяется при сборке "
                        "архива и на инференсе: обрезка меняет то, что видит модель")
    p.add_argument("--batch", type=int, default=48)
    p.add_argument("--accum", type=int, default=1, help="накопление градиента")
    p.add_argument("--epochs", type=int, default=2)
    p.add_argument("--lr", type=float, default=2e-5)
    p.add_argument("--warmup", type=float, default=0.06)
    p.add_argument("--weight-decay", dest="weight_decay", type=float, default=0.01)
    p.add_argument("--clip", type=float, default=1.0)
    p.add_argument("--workers", type=int, default=6)
    p.add_argument("--swap-aug", dest="swap_aug", type=int, choices=(0, 1), default=1,
                   help="перестановка сторон пары (по умолчанию: 1)")
    p.add_argument("--cat-heads", dest="cat_heads", type=int, choices=(0, 1), default=0,
                   help="СВОЯ ГОЛОВА НА КАЖДУЮ КАТЕГОРИЮ поверх общего энкодера. "
                        "Технически это num_labels=20 и выбор столбца по категории "
                        "пары: один проход энкодера, время инференса не растёт, "
                        "а решающее правило у каждой категории своё. Нужно потому, "
                        "что знак сигнала между категориями противоположен: при "
                        "дословно совпавшем имени доля позитивов в «Электронике» "
                        "0.776, а в «Обуви» 0.072 при фоне 0.099")
    p.add_argument("--cat-batches", dest="cat_batches", type=int, choices=(0, 1), default=0,
                   help="собирать батч из пар ОДНОЙ категории. Нужно вместе с "
                        "--rank-loss: ранжирующая добавка имеет смысл только "
                        "внутри той группы, по которой считается метрика")
    p.add_argument("--rank-loss", dest="rank_loss", type=float, default=0.0,
                   help="вес попарной ранжирующей добавки внутри батча. Метрика — "
                        "average precision ВНУТРИ категории, то есть чистое "
                        "ранжирование; BCE же тянет логиты к абсолютным 0 и 1 "
                        "и порядок внутри категории оптимизирует лишь косвенно")
    p.add_argument("--ap-loss", dest="ap_loss", type=float, default=0.0,
                   help="вес ПРЯМОГО суррогата average precision (Smooth-AP) "
                        "внутри категории. В отличие от --rank-loss, который "
                        "весит все инверсии одинаково, здесь ошибка в голове "
                        "списка стоит дороже — ровно как в самой метрике. "
                        "Проверено тестом: при малой температуре совпадает с "
                        "sklearn.average_precision_score")
    p.add_argument("--ap-tau", dest="ap_tau", type=float, default=0.05,
                   help="температура сигмоиды в Smooth-AP. Меньше — ближе к "
                        "настоящему AP, но градиент вырождается в ступеньку; "
                        "0.01..0.1 рабочий диапазон (по умолчанию: 0.05)")
    p.add_argument("--noise-mask", dest="noise_mask", default="",
                   help="npz из 07_noise_mask.py: какие пары текстово "
                        "невыучиваемы. Их вес в лоссе умножается на "
                        "--noise-weight, а не выбрасывается совсем — метка "
                        "всё-таки несёт часть сигнала")
    p.add_argument("--noise-weight", dest="noise_weight", type=float, default=0.1,
                   help="множитель веса шумных пар (по умолчанию: 0.1)")
    p.add_argument("--replay", dest="replay", default="",
                   help="корпус МАШИННЫХ пар для подмешивания в стадию B. "
                        "Тест размечен процессом, более близким к машинному "
                        "(доля позитивов 0.111 против 0.257 у ручной), поэтому "
                        "чистое дообучение на ручных метках частично уводит "
                        "ОТ цели. Реплей держит модель на месте")
    p.add_argument("--replay-share", dest="replay_share", type=float, default=0.2,
                   help="доля машинных пар в обучающей выборке стадии B")
    p.add_argument("--trust-remote-code", dest="trust_remote_code", action="store_true",
                   help="разрешить модели принести СВОЙ код (EuroBERT, gte и "
                        "прочие с нестандартной архитектурой). В контейнере "
                        "сети нет, поэтому такой код обязан лежать В АРХИВЕ "
                        "рядом с весами; 40_build_submission кладёт содержимое "
                        "каталога чекпойнта целиком, так что при обучении из "
                        "локального каталога он туда попадает сам")
    p.add_argument("--local-attention", dest="local_attention", type=int, default=0,
                   help="ПЕРЕОПРЕДЕЛИТЬ окно локального внимания у ModernBERT. "
                        "В конфиге 256, а полуокно = 256/2, то есть токен видит "
                        "+-128 позиций; глобальны лишь 8 слоёв из 22. При медиане "
                        "пары 178 токенов и p95 330 локальные слои вторую "
                        "карточку видят, но на длинных парах — уже нет. Значение "
                        "8192 делает все слои эффективно глобальными: счёт "
                        "внимания это ~6%% работы, так что цена около +3.5%% "
                        "к инференсу. 0 — не трогать (по умолчанию)")
    p.add_argument("--cat-balance", dest="cat_balance", type=int, choices=(0, 1), default=0,
                   help="взвешивать примеры обратно доле их КАТЕГОРИИ. Метрика "
                        "усредняет 20 категорий поровну, а в выборке «Обувь» это "
                        "5.7%% пар против 13%% у «Детских товаров» — без веса "
                        "обучение оптимизирует средний по парам результат, то есть "
                        "не ту величину, которую меряют")
    p.add_argument("--grad-ckpt", dest="grad_ckpt", action="store_true")
    p.add_argument("--no-save", dest="save", action="store_false",
                   help="не сохранять веса: только измерить качество")
    p.add_argument("--device", default="auto", choices=("auto", "cuda", "cpu"),
                   help="cpu пригоден только для крошечной проверки формата")
    p.add_argument("--bootstrap", action="store_true",
                   help="печатать интервал ожидаемого ЛБ по подвыборкам: без него "
                        "разница в третьем знаке ничего не значит")
    p.add_argument("--seed", type=int, default=1000)
    p.add_argument("--limit", type=int, default=0,
                   help="взять только первые N пар. НЕ работает на стадии B: "
                        "разбиение и предсказания берутся из общего индекса на "
                        "все пары. Оставлен для отладки загрузчика")
    p.add_argument("--bucket", type=int, choices=(0, 1), default=1,
                   help="батчи, однородные по длине (по умолчанию: 1); "
                        "игнорируется при --cat-batches, там своя раскладка")
    p.add_argument("--bucket-window", dest="bucket_window", type=int, default=64,
                   help="ширина окна сортировки в батчах (по умолчанию: 64)")
    return p.parse_args()


def read_text_table(path: Path) -> dict:
    import numpy as np
    import pyarrow.parquet as pq

    if not path.is_file():
        raise SystemExit(f"нет файла с текстами: {path} (сделайте src/scripts/10_prep_text.py)")
    t = pq.read_table(path)
    missing = sorted({"target", "category", "brand", "text_a", "text_b",
                      "id1", "id2"} - set(t.column_names))
    if missing:
        raise SystemExit(
            f"в {path} нет колонок {missing}. Пересоберите файл: колонка brand "
            f"задаёт разбиение по невиданным брендам и заменять её нечем")
    return {
        "target": t.column("target").to_numpy(zero_copy_only=False).astype(np.float32),
        "category": np.asarray(t.column("category").to_pylist(), dtype=object),
        "brand": t.column("brand").to_pylist(),
        "text_a": t.column("text_a").to_pylist(),
        "text_b": t.column("text_b").to_pylist(),
        "id1": t.column("id1").to_numpy(zero_copy_only=False),
        "id2": t.column("id2").to_numpy(zero_copy_only=False),
    }


def resolve_cfg(text: Path, init_from: Path | None,
                allow_change: bool = False, log=print) -> SerializeConfig:
    from src.ecup.serialize import check_cfg_matches_code

    cfg = SerializeConfig.load(meta_dir(text))
    check_cfg_matches_code(cfg)
    if init_from is not None:
        cfg_a = SerializeConfig.load(init_from)
        if cfg_a != cfg:
            if not allow_change:
                raise SystemExit(
                    f"формат текста стадии A и стадии B различается:\n"
                    f"  {init_from}: {cfg_a.to_dict()}\n"
                    f"  {text}: {cfg.to_dict()}\n"
                    f"Пересоберите тексты одним набором параметров, повторите "
                    f"стадию A на нужном формате — либо, если смена формата "
                    f"и есть проверяемая гипотеза, задайте --allow-text-change")
            log("ВНИМАНИЕ: формат текста намеренно меняется между стадиями "
                "(--allow-text-change)")
            log(f"  стадия A {init_from}: {cfg_a.to_dict()}")
            log(f"  стадия B {text}: {cfg.to_dict()}")
            log("  на инференс поедет формат стадии B — он записан в чекпойнт")
    return cfg


def build_cat_sampler(categories, batch_size, seed, lengths=None):
    import numpy as _np
    from torch.utils.data import Sampler

    class ByCategory(Sampler):
        def __init__(self):
            self.by_cat = {}
            cats = _np.asarray(categories).astype(str)
            for c in _np.unique(cats):
                self.by_cat[c] = _np.flatnonzero(cats == c)
            self.n = len(cats)
            self.seed = seed
            self.ep_ = 0

        def __len__(self):
            return sum(int(_np.ceil(len(v) / batch_size)) for v in self.by_cat.values())

        def __iter__(self):
            rng = _np.random.default_rng(self.seed + self.ep_)
            self.ep_ += 1
            batches = []
            window = 64 * batch_size
            for c, idxs in self.by_cat.items():
                d = idxs.copy()
                rng.shuffle(d)
                if lengths is not None:
                    L = _np.asarray(lengths)
                    d = _np.concatenate([
                        chunk[_np.argsort(L[chunk], kind="stable")]
                        for chunk in (d[s:s + window] for s in range(0, len(d), window))
                    ]) if len(d) else d
                for s in range(0, len(d), batch_size):
                    batches.append(d[s:s + batch_size].tolist())
            rng.shuffle(batches)
            return iter(batches)

    return ByCategory()


def build_dataset_class():
    import torch
    from torch.utils.data import Dataset

    class Pairs(Dataset):

        def __init__(self, ta, tb, y, tok, max_len, swap_aug=False, seed=0, w=None,
                     ci=None):
            self.ta, self.tb, self.y = ta, tb, y
            self.w = w
            self.ci = ci
            self.tok, self.max_len = tok, max_len
            self.swap_aug = swap_aug
            self.seed = seed
            self._rng = None

        def __len__(self):
            return len(self.ta)

        def _rng_here(self):
            if self._rng is None:
                info = torch.utils.data.get_worker_info()
                self._rng = random.Random(self.seed * 1_000_003 + (info.id if info else 0))
            return self._rng

        def __getitem__(self, i):
            a, b = self.ta[i], self.tb[i]
            if self.swap_aug and self._rng_here().random() < 0.5:
                a, b = b, a
            return (a, b, self.y[i], (1.0 if self.w is None else self.w[i]),
                    (0 if self.ci is None else self.ci[i]))

        def collate(self, batch):
            a, b, y, w, ci = zip(*batch)
            enc = self.tok(list(a), list(b), padding=True, truncation=True,
                           max_length=self.max_len, return_tensors="pt")
            enc["_w"] = torch.tensor(w, dtype=torch.float32)
            enc["_ci"] = torch.tensor(ci, dtype=torch.long)
            enc["labels"] = torch.tensor(y, dtype=torch.float32)
            return enc

    return Pairs


def key_idf_of(text_path: Path, override: Path | None = None) -> Path:
    if override is not None:
        if not Path(override).is_file():
            raise SystemExit(f"нет словаря IDF {override} (--key-idf)")
        return Path(override)
    p = meta_dir(text_path) / "prep.json"
    if not p.is_file():
        raise SystemExit(
            f"нет {p}: неизвестно, каким словарём IDF отранжированы ключи "
            f"в {text_path}. Пересоберите тексты src/scripts/10_prep_text.py "
            f"или укажите словарь флагом --key-idf")
    prep = json.loads(p.read_text(encoding="utf-8"))
    cands = [prep.get("key_idf"), prep.get("key_idf_name")]
    for v in cands:
        if not v:
            continue
        for c in (Path(v), ARTIFACTS / Path(v).name):
            if c.is_file():
                return c
    raise SystemExit(
        f"словарь IDF из {p} не найден: пробовал {[str(v) for v in cands if v]} "
        f"и те же имена в {ARTIFACTS}. Без него чекпойнт уедет без key_idf.json, "
        f"а сборщик подставит чужой словарь — ключи отранжируются иначе, чем "
        f"при обучении, и текст разойдётся молча. Укажите файл флагом --key-idf")


def save_cat_index(out: Path, cat_list: list) -> None:
    (out / "cat_index.json").write_text(
        json.dumps({"categories": cat_list}, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8")


def save_checkpoint(out: Path, model, tok, cfg: SerializeConfig, meta: dict,
                    key_idf: Path | None = None) -> None:
    out.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(out), safe_serialization=True)
    tok.save_pretrained(str(out))
    cfg.save(out)
    if key_idf is None or not Path(key_idf).is_file():
        raise SystemExit(
            f"нет словаря IDF {key_idf}: чекпойнт без key_idf.json неполон")
    shutil.copy2(key_idf, out / "key_idf.json")
    (out / "train_meta.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2, default=str) + "\n",
        encoding="utf-8")


def run_fold(args, data, fold_arr, fold: int, cfg: SerializeConfig,
             kidf: Path | None, log):
    import numpy as np
    import torch
    from torch.utils.data import DataLoader
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                              get_cosine_schedule_with_warmup)

    from src.ecup import lb
    from src.ecup.folds import bootstrap_lb
    from src.ecup.metric import macro_pr_auc

    device = args.device
    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    torch.manual_seed(args.seed + fold)
    random.seed(args.seed + fold)

    y = data["target"]
    is_va = fold_arr == fold
    tr_i = np.where(~is_va)[0]
    va_i = np.where(is_va)[0]
    log(f"фолд {fold}: обучение {len(tr_i):,} пар, валидация {len(va_i):,} "
        f"(доля позитивов {y[va_i].mean():.4f})")

    cat_list = sorted({str(c) for c in data["category"]})
    cat_index = {c: i for i, c in enumerate(cat_list)}
    n_labels = len(cat_list) if args.cat_heads else 1
    if args.cat_heads:
        log(f"  головы по категориям: {n_labels} штук поверх общего энкодера")

    init = str(args.init_from) if args.init_from else getattr(args, "model_id", args.model)
    tok = AutoTokenizer.from_pretrained(init, trust_remote_code=args.trust_remote_code)
    extra = {}
    if args.trust_remote_code:
        extra["trust_remote_code"] = True
    if args.local_attention:
        from transformers import AutoConfig
        cfg_ = AutoConfig.from_pretrained(init, trust_remote_code=args.trust_remote_code)
        was = getattr(cfg_, "local_attention", None)
        if was is None:
            raise SystemExit(f"у {init} нет local_attention — модель не ModernBERT, "
                             f"флаг --local-attention неприменим")
        cfg_.local_attention = args.local_attention
        cfg_.num_labels = n_labels
        extra["config"] = cfg_
        log(f"  ОКНО ВНИМАНИЯ ПЕРЕОПРЕДЕЛЕНО: {was} -> {args.local_attention} "
            f"(полуокно {args.local_attention // 2} позиций)")
    try:
        model = AutoModelForSequenceClassification.from_pretrained(
            init, dtype=torch.float32, ignore_mismatched_sizes=True,
            **(extra if "config" in extra else {**extra, "num_labels": n_labels}),
            attn_implementation="sdpa").to(device)
    except Exception as e:
        log(f"  sdpa не подошёл ({type(e).__name__}: {e}), беру eager")
        model = AutoModelForSequenceClassification.from_pretrained(
            init, dtype=torch.float32, ignore_mismatched_sizes=True,
            **(extra if "config" in extra else {**extra, "num_labels": n_labels}),
            attn_implementation="eager").to(device)
    if args.grad_ckpt:
        model.gradient_checkpointing_enable()

    import inspect
    accepted = set(inspect.signature(model.forward).parameters)

    Pairs = build_dataset_class()
    ta, tb = data["text_a"], data["text_b"]
    w_train = None
    if args.cat_balance:
        import numpy as _np
        cat_train = _np.asarray(data["category"])[tr_i].astype(str)
        uniq, score_ = _np.unique(cat_train, return_counts=True)
        share = dict(zip(uniq.tolist(), (score_ / score_.sum()).tolist()))
        raw_w = _np.array([1.0 / share[c] for c in cat_train.tolist()], dtype=_np.float32)
        w_train = (raw_w / raw_w.mean()).tolist()
        log(f"  категорийная балансировка: {len(uniq)} категорий, "
            f"вес от {min(w_train):.2f} до {max(w_train):.2f}")
    if args.noise_mask:
        import numpy as _np
        z = _np.load(args.noise_mask, allow_pickle=True)
        noisy = z["noisy"]
        gated = z["gated"] if "gated" in z.files else _np.zeros(len(noisy), dtype=bool)
        if len(noisy) != len(y):
            raise SystemExit(
                f"маска на {len(noisy):,} пар, а корпус на {len(y):,}: маска "
                f"строилась на другом корпусе, порядок строк не совпадает")
        m_ = (noisy | gated)[tr_i]
        if gated.any():
            log(f"  гейт категорий {list(z['gate']) if 'gate' in z.files else '?'}: "
                f"{int(gated[tr_i].sum()):,} пар")
        if w_train is None:
            w_train = _np.ones(len(tr_i), dtype=_np.float32)
        else:
            w_train = _np.asarray(w_train, dtype=_np.float32)
        w_train = w_train * _np.where(m_, args.noise_weight, 1.0)
        w_train = (w_train / w_train.mean()).tolist()
        log(f"  глушение шума: {int(m_.sum()):,} пар из {len(tr_i):,} "
            f"({100 * m_.mean():.1f}%) с весом {args.noise_weight}")

    ci_all = np.array([cat_index[str(c)] for c in data["category"]], dtype=np.int64)
    ta_tr = [ta[i] for i in tr_i]
    tb_tr = [tb[i] for i in tr_i]
    y_tr = y[tr_i]
    ci_tr = ci_all[tr_i]
    w_tr = w_train

    if args.replay:
        import numpy as _np
        import pyarrow.parquet as _pq
        need = int(len(tr_i) * args.replay_share / max(1 - args.replay_share, 1e-6))
        rt = _pq.read_table(args.replay, columns=["text_a", "text_b", "target", "category"])
        rng2 = _np.random.default_rng(args.seed + fold)
        taken = rng2.choice(rt.num_rows, size=min(need, rt.num_rows), replace=False)
        taken.sort()
        rt = rt.take(taken)
        rcat = [str(c) for c in rt.column("category").to_pylist()]
        known = [i for i, c in enumerate(rcat) if c in cat_index]
        if len(known) < len(rcat):
            log(f"  реплей: {len(rcat) - len(known)} пар с неизвестной категорией пропущено")
        ta_tr = ta_tr + [rt.column("text_a")[i].as_py() for i in known]
        tb_tr = tb_tr + [rt.column("text_b")[i].as_py() for i in known]
        y_tr = _np.concatenate([y_tr,
                                rt.column("target").to_numpy(zero_copy_only=False)[known].astype(y_tr.dtype)])
        ci_tr = _np.concatenate([ci_tr, _np.array([cat_index[rcat[i]] for i in known], dtype=_np.int64)])
        if w_tr is not None:
            w_tr = list(w_tr) + [1.0] * len(known)
        log(f"  реплей: +{len(known):,} машинных пар "
            f"({100 * len(known) / len(y_tr):.1f}% обучающей выборки)")

    ds_tr = Pairs(ta_tr, tb_tr, y_tr, tok,
                  args.max_len, swap_aug=bool(args.swap_aug), seed=args.seed + fold,
                  w=w_tr, ci=ci_tr)
    ds_va = Pairs([ta[i] for i in va_i], [tb[i] for i in va_i], y[va_i], tok, args.max_len,
                  ci=ci_all[va_i])
    g = torch.Generator()
    g.manual_seed(args.seed + fold)
    sampler_ = None
    if args.cat_batches:
        from src.ecup.bucket import pair_lengths, padding_share
        lens_ = pair_lengths([ta[i] for i in tr_i], [tb[i] for i in tr_i])
        sampler_ = build_cat_sampler(np.asarray(data["category"])[tr_i],
                                    args.batch, args.seed + fold, lengths=lens_)
        log(f"  батчи однородны по категории: {len(sampler_)} батчей на эпоху, "
            f"доля паддинга {padding_share(lens_, list(iter(sampler_))):.1%}")
        dl_tr = DataLoader(ds_tr, batch_sampler=sampler_,
                           num_workers=args.workers, collate_fn=ds_tr.collate,
                           pin_memory=(device == "cuda"),
                           persistent_workers=args.workers > 0)
    elif args.bucket:
        from src.ecup.bucket import build_sampler_class, padding_share, pair_lengths

        lens = pair_lengths([ta[i] for i in tr_i], [tb[i] for i in tr_i])
        sampler_ = build_sampler_class()(
            lens, args.batch, seed=args.seed + fold,
            window_batches=args.bucket_window, drop_last=True)
        log(f"  батчи по длине: {len(sampler_):,} батчей, "
            f"паддинг {padding_share(lens, list(sampler_)):.1%}")
        dl_tr = DataLoader(ds_tr, batch_sampler=sampler_,
                           num_workers=args.workers, collate_fn=ds_tr.collate,
                           pin_memory=(device == "cuda"),
                           persistent_workers=args.workers > 0)
    else:
        dl_tr = DataLoader(ds_tr, batch_size=args.batch, shuffle=True,
                           num_workers=args.workers, collate_fn=ds_tr.collate,
                           pin_memory=(device == "cuda"), drop_last=True,
                           persistent_workers=args.workers > 0, generator=g)
    dl_va = DataLoader(ds_va, batch_size=args.batch * 2, shuffle=False,
                       num_workers=min(args.workers, 4), collate_fn=ds_va.collate,
                       pin_memory=(device == "cuda"))

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr,
                            weight_decay=args.weight_decay)
    steps = max(1, len(dl_tr) * args.epochs // args.accum)
    sched = get_cosine_schedule_with_warmup(opt, int(args.warmup * steps), steps)
    lossf = torch.nn.BCEWithLogitsLoss(reduction='none')

    def amp():
        return (torch.autocast("cuda", dtype=torch.bfloat16) if device == "cuda"
                else nullcontext())

    cat_va = data["category"][va_i]
    yva_int = y[va_i].astype(int)
    best, best_pred, best_ep = -1.0, None, -1
    out = ARTIFACTS / f"{args.tag}_f{fold}"

    for ep in range(args.epochs):
        model.train()
        if sampler_ is not None and hasattr(sampler_, "set_epoch"):
            sampler_.set_epoch(ep)
        t0 = time.time()
        run = 0.0
        opt.zero_grad(set_to_none=True)
        for step, batch in enumerate(dl_tr):
            labels = batch.pop("labels").to(device, non_blocking=True)
            ws = batch.pop("_w").to(device, non_blocking=True)
            ci = batch.pop("_ci").to(device, non_blocking=True)
            batch = {k: v.to(device, non_blocking=True)
                     for k, v in batch.items() if k in accepted}
            with amp():
                raw_ = model(**batch).logits
                logits = (raw_.gather(1, ci.unsqueeze(1)).squeeze(1)
                          if args.cat_heads else raw_.squeeze(-1))
                elementwise = lossf(logits.float(), labels)
                loss = (elementwise * ws).sum() / ws.sum().clamp(min=1e-6)
                if args.rank_loss > 0 or args.ap_loss > 0:
                    log_ = logits.float()
                    if args.rank_loss > 0:
                        loss = loss + args.rank_loss * losses.pairwise_logistic(
                            log_, labels, groups=ci)
                    if args.ap_loss > 0:
                        loss = loss + args.ap_loss * losses.smooth_ap(
                            log_, labels, groups=ci, tau=args.ap_tau)
                loss = loss / args.accum
            loss.backward()
            run += float(loss.item()) * args.accum
            if (step + 1) % args.accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), args.clip)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
            if step % 200 == 0:
                log(f"  f{fold} ep{ep} шаг {step}/{len(dl_tr)} "
                    f"loss={run / (step + 1):.4f} {time.time() - t0:.0f}s")

        model.eval()
        preds = []
        with torch.inference_mode():
            for batch in dl_va:
                batch.pop("labels")
                ci_v = batch.pop("_ci")
                batch = {k: v.to(device, non_blocking=True)
                         for k, v in batch.items() if k in accepted}
                with amp():
                    raw2 = model(**batch).logits.float().cpu()
                preds.append(raw2.gather(1, ci_v.unsqueeze(1)).squeeze(1)
                             if args.cat_heads else raw2.squeeze(-1))
        p = torch.cat(preds).numpy()
        ap = macro_pr_auc(yva_int, p, cat_va)
        log(f"  f{fold} ep{ep}: macro-AP={ap:.4f} | ожидаемый ЛБ "
            f"{lb.expected(ap, fold):.4f} (÷Jaccard фолда {fold} = "
            f"{lb.JACCARD_UNSEEN_FOLDS[fold]:.4f}) ({time.time() - t0:.0f}s)")
        if args.bootstrap:
            m, lo, hi = bootstrap_lb(yva_int, p, cat_va, fold)
            log(f"       интервал ожидаемого ЛБ: {m:.4f} [{lo:.4f}; {hi:.4f}]")
        if ap > best:
            best, best_pred, best_ep = ap, p, ep
            if args.save:
                save_checkpoint(out, model, tok, cfg, {
                    "stage": "B", "argv": sys.argv, "fold": fold, "epoch": ep,
                    "macro_ap_unseen_brands": ap,
                    "expected_lb": lb.expected(ap, fold),
                    "jaccard_of_fold": lb.JACCARD_UNSEEN_FOLDS[fold],
                    "ratio_to_jaccard": ap / lb.JACCARD_UNSEEN_FOLDS[fold],
                    "n_folds": args.n_folds, "text": str(args.text),
                    "init_from": str(args.init_from) if args.init_from else None,
                    "max_len": args.max_len, "batch": args.batch, "lr": args.lr,
                    "epochs": args.epochs, "swap_aug": bool(args.swap_aug),
                    "cat_heads": bool(args.cat_heads),
                    "serialize_config": cfg.to_dict(),
                    "key_idf": str(kidf) if kidf else None}, key_idf=kidf)
                if args.cat_heads:
                    save_cat_index(out, cat_list)
                log(f"       сохранён как лучший -> {out}")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return va_i, best_pred, best, best_ep


def main() -> None:
    args = parse_args()
    t_start = time.time()

    from src.ecup import backbones as _bb
    if args.list_backbones:
        print(_bb.table())
        print("\nЛюбой id Hugging Face тоже принимается — реестр только хранит "
              "проверенные настройки, а не ограничивает выбор.")
        return

    def log(msg: str) -> None:
        print(f"[{time.time() - t_start:7.1f}s] {msg}", flush=True)

    bb = _bb.get(args.model)
    log(f"бэкбон: {_bb.describe(args.model)}")
    if bb is None:
        log("ВНИМАНИЕ: бэкбона нет в реестре. Гиперпараметры берутся из флагов, "
            "а поведение архитектуры в контейнере проверяющей системы НЕ "
            "проверено: известно только, что образ тянет ModernBERT. Прогоните "
            "архив через src/scripts/50_selftest.py.")
    elif not bb.container_verified:
        log("ВНИМАНИЕ: архив с этим бэкбоном ещё ни разу не проходил стенд. "
            "Обучение на сервере ничего не говорит о контейнере.")
    args.model_id = _bb.resolve(args.model)

    os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
    os.environ.setdefault("HF_HOME", str(ARTIFACTS.parent / ".hf_home"))

    cfg = resolve_cfg(args.text, args.init_from,
                      allow_change=args.allow_text_change, log=log)
    log(f"формат текста: {cfg.to_dict()}")
    kidf = key_idf_of(args.text, args.key_idf)
    log(f"словарь IDF: {kidf}")
    if args.max_len < 20 * cfg.max_keys:
        log(f"ЗАМЕТЬТЕ: max_len={args.max_len} мал для max_keys={cfg.max_keys}; "
            f"у измеренной модели было 320 токенов на 16 ключей")

    import numpy as np
    import torch

    from src.ecup import lb

    from src.ecup import backbones
    from src.ecup import oof as oof_store
    from src.ecup.folds import refit_scale
    from src.ecup.metric import macro_pr_auc, per_category_ap

    data = read_text_table(args.text)
    if args.limit:
        for k in ("text_a", "text_b", "brand"):
            data[k] = data[k][:args.limit]
        data["target"] = data["target"][:args.limit]
        data["category"] = data["category"][:args.limit]
    n = len(data["target"])
    if not np.all((data["target"] == 0) | (data["target"] == 1)):
        raise SystemExit("таргеты не 0/1: стадия B работает на человеческой разметке")
    log(f"пар: {n:,}, доля позитивов {data['target'].mean():.4f}")

    index = oof_store.load_index(ARTIFACTS)
    digest = oof_store.pairs_digest(data["id1"], data["id2"])
    if digest != index.digest:
        raise SystemExit(
            f"порядок строк в {args.text.name} ({digest}) не совпадает с общим "
            f"индексом OOF ({index.digest}). Складывать такие предсказания "
            f"нельзя: метрика получилась бы правдоподобной и неверной. "
            f"Либо пересоберите тексты из того же matches.parquet, либо "
            f"перестройте индекс: src/scripts/05_build_index.py --force")
    if len(index) != n:
        why = (
            f" Похоже, тексты урезаны флагом --limit {args.limit}. Со стадией B "
            f"он несовместим: разбиение и предсказания берутся из общего индекса "
            f"на все пары, и на куске таблицы их складывать не с чем. Боевые "
            f"прогоны его не использовали. Чтобы проверить запуск, уменьшайте "
            f"--batch или прервите обучение после первых шагов."
            if args.limit else "")
        raise SystemExit(
            f"в индексе {len(index):,} пар, в текстах {n:,}: это разные таблицы."
            + why)
    fold_arr = index.fold
    sizes = [int((fold_arr == f).sum()) for f in range(index.n_folds)]
    log(f"разбиение из общего индекса: {sizes} (фолдов {index.n_folds})")
    if args.n_folds != index.n_folds:
        log(f"ЗАМЕТЬТЕ: --n-folds={args.n_folds} игнорируется, в индексе "
            f"{index.n_folds} фолдов")

    recipe = args.recipe or args.tag
    scores: dict[int, float] = {}
    for f in [int(x) for x in args.folds.split(",") if x.strip() != ""]:
        va_i, pred, best, best_ep = run_fold(args, data, fold_arr, f, cfg, kidf, log)
        scores[f] = best
        full_pred = np.full(n, np.nan, dtype=np.float32)
        full_pred[va_i] = pred
        oof_store.save_fold(ARTIFACTS, recipe, f, full_pred[fold_arr == f], index, meta={
            "backbone": getattr(args, "model_id", backbones.resolve(args.model)),
            "backbone_alias": args.model,
            "init_from": str(args.init_from) if args.init_from else None,
            "max_len": args.max_len, "batch": args.batch, "lr": args.lr,
            "epochs": args.epochs, "seed": args.seed, "swap_aug": args.swap_aug,
            "serialize": cfg.to_dict(),
            "serialize_group": f"canon{int(cfg.canon)}_k{cfg.max_keys}",
            "text": str(args.text),
            f"fold{f}_macro_ap": best, f"fold{f}_best_epoch": best_ep,
        })
        log(f"[фолд {f}] лучший macro-AP={best:.4f} (эпоха {best_ep}) | "
            f"ожидаемый ЛБ {lb.expected(best, f):.4f} | "
            f"отношение к Jaccard {best / lb.JACCARD_UNSEEN_FOLDS[f]:.4f}")
        log(f"[фолд {f}] OOF записан в хранилище: рецепт «{recipe}»")

    oof_all, meta = oof_store.load(ARTIFACTS, recipe, index)
    done = np.isfinite(oof_all)
    if done.all():
        y = index.y.astype(int)
        full = macro_pr_auc(y, oof_all, index.category)
        log(f"ПОЛНЫЙ OOF macro-AP = {full:.4f} | ЛБ грубо ~{lb.predict(full, sum(lb.JACCARD_UNSEEN_FOLDS.values()) / len(lb.JACCARD_UNSEEN_FOLDS)):.4f} "
            f"(множитель врёт на приростах впятеро — сравнивайте "
            f"через 36_clean_metric.py)")
        for c, v in sorted(per_category_ap(y, oof_all, index.category).items(),
                           key=lambda kv: kv[1]):
            log(f"    {c:30s} {v:.4f}")
    else:
        log(f"OOF рецепта «{recipe}» заполнен на {int(done.sum()):,}/{n:,} "
            f"(фолды {meta['folds_present']}): полная оценка будет после всех фолдов")
    log(f"чистая метрика фолда 0: src/scripts/36_clean_metric.py")

    mult, spread = refit_scale()
    log(f"множитель ЛБ сейчас {mult:.3f} (разброс {spread:.1f}%). После замера "
        f"добавьте точку в src.ecup.folds.LB_POINTS и повторите refit_scale: "
        f"рост разброса означает, что неверна модель связи, а не константа")
    log(f"результаты фолдов: {scores}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/scripts/36_clean_metric.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import json
import re
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))

from src.ecup.config import ARTIFACTS, DATA

WORD_RE = re.compile(r"\w+")
HI, LO = 0.8, 0.3
SLOPE, INTERCEPT = 1.0457, -0.1919
RATE_NAMES = ("testrate.json",)


def find_rate() -> Path:
    for base in (ROOT / "src" / "ecup", ROOT / "ecup", ROOT, ARTIFACTS, Path(ARTIFACTS).parent):
        for n in RATE_NAMES:
            p = Path(base) / n
            if p.is_file():
                return p
    raise SystemExit(
        "нет testrate.json (доли позитивов теста по категориям). Без него "
        "выравнивание невозможно, а без выравнивания локальное число и доска "
        "в разных шкалах")


def noise_mask(id1, id2, y, items: Path, log=print):
    import numpy as np
    import pyarrow.parquet as pq

    it = pq.read_table(items, columns=["id", "name"])
    need = set(int(v) for v in id1) | set(int(v) for v in id2)
    tok = {int(i): set(WORD_RE.findall((n or "").lower()))
           for i, n in zip(it.column("id").to_numpy(zero_copy_only=False),
                           it.column("name").to_pylist()) if int(i) in need}
    empty: set = set()

    def jac(u, v):
        A, B = tok.get(int(u), empty), tok.get(int(v), empty)
        s = len(A | B)
        return len(A & B) / s if s else 0.0

    J = np.fromiter((jac(a, b) for a, b in zip(id1, id2)),
                    dtype=np.float32, count=len(y))
    pos = y > 0.5
    m = (~pos & (J >= HI)) | (pos & (J <= LO))
    log(f"шумных по имени: {int(m.sum()):,} ({100 * m.mean():.1f}%)")
    return m


def main() -> int:
    p = argparse.ArgumentParser(prog="36_clean_metric.py")
    p.add_argument("--oof", type=Path, default=None, help="каталог OOF")
    p.add_argument("--text", type=Path, default=None,
                   help="человеческий корпус: из него берутся id пар в порядке индекса")
    p.add_argument("--items", type=Path, default=None)
    p.add_argument("--only", nargs="*", default=None, help="только эти модели")
    p.add_argument("--seeds", type=int, default=12,
                   help="прореживаний под тестовые доли; разброс падает как 1/sqrt")
    a = p.parse_args()

    import numpy as np
    import pyarrow.parquet as pq
    from sklearn.metrics import average_precision_score

    oof = a.oof or ARTIFACTS / "oof"
    idx_p = oof / "_index.npz"
    if not idx_p.is_file():
        raise SystemExit(f"нет {idx_p}: сначала src/scripts/05_build_index.py")
    idx = np.load(idx_p, allow_pickle=True)
    y, cat, fold = idx["y"].astype(float), idx["category"], idx["fold"].astype(int)

    text = a.text or ARTIFACTS / "ce_text_human_raw_k48.parquet"
    t = pq.read_table(text, columns=["id1", "id2", "target"])
    if t.num_rows != len(y):
        raise SystemExit(
            f"в корпусе {t.num_rows:,} строк, в индексе {len(y):,}: это разные "
            f"наборы пар, маску не на что накладывать")
    yt = t.column("target").to_numpy(zero_copy_only=False)
    if not np.array_equal((yt > 0.5).astype(np.int8), idx["y"].astype(np.int8)):
        raise SystemExit(
            f"порядок строк {text} не совпадает с индексом OOF: метки разошлись")
    id1 = t.column("id1").to_numpy(zero_copy_only=False)
    id2 = t.column("id2").to_numpy(zero_copy_only=False)

    memo = oof / "_noisy_name_fold0.npy"
    va = fold == 0
    if memo.is_file():
        noisy = np.load(memo)
        print(f"маска шума из кэша: {int(noisy.sum()):,} ({100 * noisy.mean():.1f}%)")
    else:
        full_ = noise_mask(id1, id2, y, a.items or DATA / "items_human.parquet")
        noisy = full_[va]
        np.save(memo, noisy)

    ys, cs = y[va], cat[va]
    RATE = json.loads(find_rate().read_text(encoding="utf-8"))
    KAT = [(c, cs == c) for c in np.unique(cs)]

    def macro(s, keep=None):
        out = []
        for _, k in KAT:
            m = k if keep is None else (k & keep)
            if m.sum() < 10 or ys[m].max() == ys[m].min():
                continue
            out.append(average_precision_score(ys[m] > 0.5, s[m]))
        return float(np.mean(out)) if out else float("nan")

    def aligned(s, seed, keep):
        rng = np.random.default_rng(seed)
        out = []
        for c, k in KAT:
            m = k if keep is None else (k & keep)
            yy, ss = ys[m], s[m]
            p_ = np.flatnonzero(yy > 0.5)
            n_ = np.flatnonzero(yy <= 0.5)
            r = RATE.get(str(c))
            if r is None or not len(p_) or not len(n_):
                continue
            keep_n = min(int(round(r * len(n_) / max(1 - r, 1e-9))), len(p_))
            if keep_n < 5:
                continue
            take = np.concatenate([rng.choice(p_, keep_n, replace=False), n_])
            out.append(average_precision_score(yy[take] > 0.5, ss[take]))
        return float(np.mean(out)) if out else float("nan")

    def aligned_(s, keep):
        v = np.array([aligned(s, k, keep) for k in range(a.seeds)])
        return v.mean(), v.std()

    names_ = sorted(d.name for d in oof.iterdir()
                   if d.is_dir() and (d / "fold0.npy").is_file())
    if a.only:
        names_ = [n for n in names_ if n in set(a.only)]
    if not names_:
        raise SystemExit(f"в {oof} нет ни одной модели с fold0.npy")

    print(f"\n{'модель':18s} {'сырая':>7s} {'clean':>7s} {'ВЫРОВН_cl':>9s} {'±':>6s} "
          f"{'прогноз ЛБ':>10s}")
    rows = []
    for n in names_:
        s = np.load(oof / n / "fold0.npy")
        if len(s) != va.sum():
            print(f"{n:18s} пропуск: {len(s):,} значений против {va.sum():,} в фолде")
            continue
        m, sd = aligned_(s, ~noisy)
        rows.append((n, macro(s), macro(s, ~noisy), m, sd, SLOPE * m + INTERCEPT))
    for n, raw, cl, v, sd, lb in sorted(rows, key=lambda r: -r[3]):
        print(f"{n:18s} {raw:7.4f} {cl:7.4f} {v:9.4f} {sd:6.4f} {lb:10.4f}")

    print(f"\nпрогноз = {SLOPE}*ВЫРОВН_clean {INTERCEPT:+}; честная ошибка прямой "
          f"0.0017 средняя, 0.0044 худшая.")
    print("Разрывы прогноза меньше 0.003 за сигнал не считать: на замере из семи "
          "моделей метрика ошиблась ровно на паре с разрывом 0.0014.")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile src/scripts/40_build_submission.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import ast
import hashlib
import json
import shutil
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

REPO = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(REPO))

from src.ecup.config import ARTIFACTS, ROOT, SerializeConfig

INFER_MODULES = ["__init__.py", "config.py", "textprep.py", "serialize.py",
                 "infer.py", "solve.py"]

ALLOWED_THIRD_PARTY = frozenset({
    "numpy", "torch", "transformers", "pyarrow", "tokenizers", "safetensors", "src",
})

CKPT_SUFFIXES = (".json", ".txt", ".model", ".safetensors")

CKPT_REQUIRED = ("config.json", "model.safetensors", "serialize_config.json")

TRAIN_META_NAME = "train_meta.json"

IMAGE = "odsai/ecup26-matching-baseline:1.0"


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="40_build_submission.py",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        description=(
            "Собирает zip решения: run.py, пакет src.ecup (только путь инференса),\n"
            "metadata.json, artifacts/key_idf.json, artifacts/blend.json и каталоги\n"
            "моделей ce_f0..ce_fN, в каждом — serialize_config.json.\n"
            "Перед упаковкой проверяет комплектность чекпойнтов, совпадение формата\n"
            "текста у всех моделей и чистоту импортов."),
        epilog=(
            "Пример (пути даны от ECUP_ROOT: artifacts/ лежит в корне дерева,\n"
            "рядом с src/):\n"
            "  src/scripts/40_build_submission.py --tag k48b1 \\\n"
            "      --note 'канонические ключи, k16, один кросс-энкодер' \\\n"
            '      --ce "$ECUP_ROOT"/artifacts/ce_f0 --ce-batch 256\n'
            "Затем обязательно: src/scripts/50_selftest.py --zip <путь к архиву>"))
    p.add_argument("--tag", required=True, help="метка сабмита: имя архива и запись в реестре")
    p.add_argument("--note", required=True, help="что изменилось по сравнению с прошлым разом")
    p.add_argument("--ce", nargs="+", type=Path, required=True,
                   help="каталоги чекпойнтов кросс-энкодера, по одному на модель")
    p.add_argument("--weights", nargs="*", type=float, default=None,
                   help="веса моделей в смеси (по умолчанию равные). Смешивание идёт "
                        "по рангам внутри категории: метрика инвариантна к монотонным "
                        "преобразованиям, усреднять логиты нельзя")
    p.add_argument("--max-len", dest="max_len", type=int, default=None,
                   help="длина в токенах на инференсе. По умолчанию берётся из "
                        "train_meta.json чекпойнта: обрезка меняет то, что видит "
                        "модель, и обязана совпадать с обучением. Явное значение "
                        "принимается, только если совпадает с обучением")
    p.add_argument("--ce-batch", dest="ce_batch", type=int, default=256)
    p.add_argument("--key-idf", dest="key_idf", type=Path, default=None,
                   help="словарь IDF ключей. По умолчанию берётся key_idf.json из "
                        "каталога чекпойнта — тот самый, которым отранжированы "
                        "ключи при обучении. Явный файл обязан совпасть с ним")
    p.add_argument("--image", default=IMAGE,
                   help=f"образ в metadata.json (по умолчанию официальный: {IMAGE})")
    p.add_argument("--out", type=Path, default=None,
                   help="путь к архиву (по умолчанию: ROOT/submissions/submission_<tag>.zip)")
    p.add_argument("--fp16", dest="fp16", action="store_true", default=True,
                   help="сохранять веса в fp16: 299 МБ вместо 600, инференс всё равно "
                        "идёт в bf16 (по умолчанию включено)")
    p.add_argument("--no-fp16", dest="fp16", action="store_false",
                   help="копировать веса как есть (нужно, когда torch недоступен)")
    p.add_argument("--oof", type=float, default=None,
                   help="macro-AP на фолде невиданных брендов — уедет в реестр")
    p.add_argument("--max-size-mb", dest="max_size_mb", type=float, default=5000.0)
    return p.parse_args()


def check_code_tree() -> list[Path]:
    files = [REPO / "run.py"] + [REPO / "src" / "ecup" / m for m in INFER_MODULES]
    missing = [str(f) for f in files if not f.is_file()]
    if missing:
        raise SystemExit(
            "не хватает файлов кода:\n  " + "\n  ".join(missing) +
            "\nАрхив без них соберётся, а решение упадёт на сервере проверки, "
            "где лога не будет")
    return files


def resolve_key_idf(d: Path, explicit: Path | None) -> Path:
    local = d / "key_idf.json"
    if explicit is None:
        if not local.is_file():
            raise SystemExit(
                f"в чекпойнте {d} нет key_idf.json. Его кладут туда 20_pretrain.py "
                f"и 30_finetune.py при каждом сохранении; если чекпойнт приехал "
                f"со стороны, укажите словарь флагом --key-idf, подтвердив тем "
                f"самым, что это именно тот словарь, которым отранжированы ключи "
                f"при обучении. Подставлять общий молча нельзя")
        return local
    explicit = Path(explicit)
    if not explicit.is_file():
        raise SystemExit(f"нет словаря IDF ключей {explicit}")
    if local.is_file():
        a = json.loads(local.read_text(encoding="utf-8"))
        b = json.loads(explicit.read_text(encoding="utf-8"))
        if a != b:
            raise SystemExit(
                f"--key-idf {explicit} ({len(b):,} ключей) не совпадает со словарём "
                f"из чекпойнта {local} ({len(a):,} ключей). Именно словарём из "
                f"чекпойнта отранжированы ключи в обучающих текстах; подмена "
                f"не даёт ошибки, но меняет набор атрибутов в тексте пары")
    return explicit


def resolve_max_len(dirs: list[Path], explicit: int | None) -> int:
    from_ckpts: dict[str, int] = {}
    no_meta: list[str] = []
    for d in dirs:
        p = d / TRAIN_META_NAME
        v = json.loads(p.read_text(encoding="utf-8")).get("max_len") if p.is_file() else None
        if v is None:
            no_meta.append(d.name)
        else:
            from_ckpts[d.name] = int(v)
    if len(set(from_ckpts.values())) > 1:
        raise SystemExit(
            f"чекпойнты обучены с разной длиной обрезки: {from_ckpts}. "
            f"Одно значение max_len на всю смесь означает, что часть моделей "
            f"получит не тот текст, на котором обучалась")
    if from_ckpts:
        train_part = next(iter(from_ckpts.values()))
        if no_meta:
            raise SystemExit(
                f"в чекпойнтах {no_meta} нет {TRAIN_META_NAME}, а в остальных "
                f"max_len={train_part}. Неизвестную длину нельзя приравнять к "
                f"известной догадкой: пересоберите чекпойнт или соберите смесь "
                f"только из тех моделей, про которые известно, как они обучались")
        if explicit is not None and explicit != train_part:
            raise SystemExit(
                f"--max-len {explicit} не совпадает с обучением ({train_part}, "
                f"из {TRAIN_META_NAME}). Обрезка меняет то, что видит модель: "
                f"либо соберите без --max-len, либо переобучите модель")
        return train_part
    if explicit is None:
        raise SystemExit(
            f"ни в одном чекпойнте нет {TRAIN_META_NAME} с max_len, и флаг "
            f"--max-len не задан. Длину обрезки нельзя брать по умолчанию: "
            f"она определяет, какие ключи атрибутов доедут до модели")
    print(f"ЗАМЕТЬТЕ: {TRAIN_META_NAME} нет ни в одном чекпойнте, беру --max-len "
          f"{explicit} под вашу ответственность")
    return explicit


def load_ckpt_cfgs(dirs: list[Path]) -> SerializeConfig:
    from src.ecup.serialize import check_cfg_matches_code

    cfgs = []
    for d in dirs:
        if not d.is_dir():
            raise SystemExit(f"нет каталога чекпойнта: {d}")
        missing = [f for f in CKPT_REQUIRED if not (d / f).is_file()]
        if missing:
            hint = ("\n  serialize_config.json пишется скриптами 20_pretrain.py и "
                    "30_finetune.py при каждом сохранении. Если его нет, формат "
                    "текста этого чекпойнта неизвестен, и восстанавливать его "
                    "догадками нельзя." if "serialize_config.json" in missing else "")
            raise SystemExit(f"в чекпойнте {d} не хватает: {missing}{hint}")
        cfgs.append(SerializeConfig.load(d))
    first = cfgs[0]
    check_cfg_matches_code(first)
    for d, c in zip(dirs, cfgs):
        if c != first:
            raise SystemExit(
                f"чекпойнты обучены на разном тексте и в одну смесь не годятся:\n"
                f"  {dirs[0]}: {first.to_dict()}\n  {d}: {c.to_dict()}")
    return first


def scan_imports(stage: Path) -> None:
    allowed = set(sys.stdlib_module_names) | ALLOWED_THIRD_PARTY
    bad: list[str] = []
    for f in sorted(stage.rglob("*.py")):
        tree = ast.parse(f.read_text(encoding="utf-8"), filename=str(f))
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                names = [a.name.split(".")[0] for a in node.names]
            elif isinstance(node, ast.ImportFrom):
                if node.level:
                    continue
                names = [(node.module or "").split(".")[0]]
            else:
                continue
            for nm in names:
                if nm and nm not in allowed:
                    bad.append(f"{f.relative_to(stage)}:{node.lineno} -> {nm}")
    if bad:
        raise SystemExit(
            "в архиве импорты вне списка разрешённых:\n  " + "\n  ".join(bad) +
            "\nВ официальном образе есть только torch, transformers, tokenizers, "
            "sentence-transformers, safetensors, numpy, pandas, pyarrow, "
            "scikit-learn, scipy, joblib, networkx, regex, tqdm. Путь "
            "инференса ограничен ещё жёстче: стандартная библиотека, numpy, torch, "
            "transformers, pyarrow.")


def copy_checkpoint(src: Path, dst: Path, fp16: bool) -> float:
    dst.mkdir(parents=True, exist_ok=True)
    for f in sorted(src.iterdir()):
        if not f.is_file() or f.suffix not in CKPT_SUFFIXES:
            continue
        if f.name == "model.safetensors" and fp16:
            continue
        if f.name == "key_idf.json":
            continue
        shutil.copy2(f, dst / f.name)
    if fp16:
        try:
            import torch
            from safetensors.torch import load_file, save_file
        except Exception as e:
            raise SystemExit(
                f"для --fp16 нужен torch ({type(e).__name__}: {e}). "
                f"Соберите с --no-fp16 либо на машине с окружением обучения")
        sd = load_file(str(src / "model.safetensors"))
        half = {k: (v.half() if v.is_floating_point() else v) for k, v in sd.items()}
        save_file(half, str(dst / "model.safetensors"), metadata={"format": "pt"})
    else:
        shutil.copy2(src / "model.safetensors", dst / "model.safetensors")
    return sum(f.stat().st_size for f in dst.iterdir()) / 1e6


def lb_scale() -> float:
    from src.ecup.folds import LB_SCALE
    return LB_SCALE


def register(tag: str, entry: dict) -> Path:
    reg = ROOT / "submissions" / "registry.json"
    reg.parent.mkdir(parents=True, exist_ok=True)
    items = json.loads(reg.read_text(encoding="utf-8")) if reg.is_file() else []
    items = [x for x in items if x.get("tag") != tag]
    items.append(entry)
    reg.write_text(json.dumps(items, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    return reg


def main() -> None:
    args = parse_args()
    code_files = check_code_tree()
    ce_dirs = [Path(d) for d in args.ce]
    cfg = load_ckpt_cfgs(ce_dirs)
    print(f"формат текста чекпойнтов: {cfg.to_dict()}")

    weights = args.weights if args.weights else [1.0] * len(ce_dirs)
    if len(weights) != len(ce_dirs):
        raise SystemExit(f"весов {len(weights)}, а моделей {len(ce_dirs)}")

    space = "canon" if cfg.canon else "raw"
    key_idfs = [resolve_key_idf(d, args.key_idf) for d in ce_dirs]
    max_len = resolve_max_len(ce_dirs, args.max_len)
    print(f"пространство ключей: {space} | max_len из обучения: {max_len}")

    stage = ROOT / "submissions" / f"stage_{args.tag}"
    if stage.exists():
        shutil.rmtree(stage)
    (stage / "src" / "ecup").mkdir(parents=True)
    (stage / "artifacts").mkdir(parents=True)

    shutil.copy2(code_files[0], stage / "run.py")
    for f in code_files[1:]:
        shutil.copy2(f, stage / "src" / "ecup" / f.name)
    (stage / "src" / "__init__.py").write_text("", encoding="utf-8")
    (stage / "metadata.json").write_text(
        json.dumps({"image": args.image, "entry_point": "python -u run.py"},
                   indent=4) + "\n", encoding="utf-8")
    scan_imports(stage)
    print(f"импорты проверены: только стандартная библиотека, "
          f"{', '.join(sorted(ALLOWED_THIRD_PARTY - {'src'}))}")

    if "ECUP_ROOT" not in (stage / "run.py").read_text(encoding="utf-8"):
        print("ЗАМЕТЬТЕ: run.py не упоминает ECUP_ROOT. Внутри архива artifacts/ "
              "лежит рядом с run.py, а src.ecup.config по умолчанию берёт корнем "
              "родительский каталог пакета. Прогоните 50_selftest.py и убедитесь, "
              "что решение находит artifacts/")

    shutil.copy2(key_idfs[0], stage / "artifacts" / "key_idf.json")
    print(f"key_idf: {key_idfs[0]} -> artifacts/key_idf.json "
          f"({key_idfs[0].stat().st_size / 1e6:.1f} МБ)")

    for i, (d, w, kidf) in enumerate(zip(ce_dirs, weights, key_idfs)):
        dst = stage / "artifacts" / f"ce_f{i}"
        sz = copy_checkpoint(d, dst, args.fp16)
        shutil.copy2(kidf, dst / "key_idf.json")
        print(f"модель {d.name} -> ce_f{i} (вес {w}, {sz:.0f} МБ, "
              f"{'fp16' if args.fp16 else 'как есть'}, key_idf {kidf})")

    (stage / "artifacts" / "blend.json").write_text(json.dumps({
        "w_ce_list": weights,
        "max_len": max_len,
        "ce_batch": args.ce_batch,
    }, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"blend.json: w_ce_list/max_len={max_len}/ce_batch. Формат текста "
          f"(max_keys={cfg.max_keys}, canon={cfg.canon}, version={cfg.version}) "
          "едет в serialize_config.json внутри каталогов моделей")
    print("режим ГРОМКИЙ: аварийного резерва в архиве нет, при любой аварии "
          "будет traceback и ненулевой код возврата")

    out = Path(args.out) if args.out else ROOT / "submissions" / f"submission_{args.tag}.zip"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.unlink(missing_ok=True)
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in sorted(stage.rglob("*")):
            if p.is_file():
                z.write(p, p.relative_to(stage))
    size = out.stat().st_size
    digest = hashlib.sha256(out.read_bytes()).hexdigest()

    entry = {
        "tag": args.tag,
        "note": args.note,
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "archive": out.name,
        "size_mb": round(size / 1e6, 1),
        "sha256": digest,
        "image": args.image,
        "ce_dirs": [str(d) for d in ce_dirs],
        "weights": weights,
        "max_len": max_len,
        "ce_batch": args.ce_batch,
        "serialize_config": cfg.to_dict(),
        "key_idf": [str(p) for p in key_idfs],
        "oof_unseen_brands": args.oof,
        "expected_lb": (args.oof / lb_scale()) if args.oof else None,
    }
    reg = register(args.tag, entry)

    print(f"\nсобрано: {out}")
    print(f"размер: {size / 1e6:.1f} МБ (лимит {args.max_size_mb:.0f} МБ)")
    print(f"sha256: {digest}")
    print(f"реестр: {reg}")
    if size > args.max_size_mb * 1e6:
        raise SystemExit("АРХИВ БОЛЬШЕ ЛИМИТА")
    print("\nдальше:")
    print(f"  1) src/scripts/50_selftest.py --zip {out}")
    print("  2) после замера добавить точку "
          "в src.ecup.folds.LB_POINTS")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/scripts/50_selftest.py
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import ast
import csv
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

REPO = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(REPO))

from src.ecup.config import DATA, ROOT

LIMITS = {"check": (1000, 60), "public": (115_000, 360), "private": (275_000, 780)}
TOTAL_LIMIT = 1200.0

ALLOWED_THIRD_PARTY = frozenset({
    "numpy", "torch", "transformers", "pyarrow", "tokenizers", "safetensors", "src",
})


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        prog="50_selftest.py",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        description=(
            "Проверяет, что архив решения работает: распаковывает его, делает\n"
            "каталог решения доступным только для чтения, глушит сеть и запускает\n"
            "run.py на тестовых наборах с лимитами проверочной системы.\n"
            "Ненулевой код возврата — при любой обнаруженной проблеме."),
        epilog=(
            "Примеры:\n"
            "  # полный прогон на реальных данных\n"
            "  src/scripts/50_selftest.py --zip submissions/submission_k48b1.zip\n"
            "  # быстрая проверка формата на синтетике, без data/\n"
            "  src/scripts/50_selftest.py --zip submissions/submission_k48b1.zip \\\n"
            "      --synthetic --pairs 200 --budget 600"))
    p.add_argument("--zip", dest="zip_path", type=Path, default=None,
                   help="архив, собранный 40_build_submission.py "
                        "(обязателен, кроме режима --make-tiny)")
    p.add_argument("--make-tiny", dest="make_tiny", type=Path, default=None,
                   help="только сгенерировать крошечные items_tiny.parquet и "
                        "matches_tiny.parquet в указанном каталоге и выйти. "
                        "Ими проверяют, что run.py запускается целиком")
    p.add_argument("--stages", default="check,public,private",
                   help="какие этапы гонять (по умолчанию: все три)")
    p.add_argument("--runs", type=int, default=1, help="повторов каждого этапа")
    p.add_argument("--python", default=sys.executable,
                   help="интерпретатор для запуска решения; для проверки совместимости "
                        "указывайте python из реплики официального образа")
    p.add_argument("--synthetic", action="store_true",
                   help="сгенерировать крошечные наборы вместо чтения data/: "
                        "проверка формата и импортов там, где данных нет")
    p.add_argument("--pairs", type=int, default=0,
                   help="переопределить число пар на этапе (полезно с --synthetic)")
    p.add_argument("--budget", type=float, default=0.0,
                   help="значение ECUP_BUDGET_S для решения (0 — не задавать)")
    p.add_argument("--work", type=Path, default=None,
                   help="рабочий каталог (по умолчанию: ROOT/submissions/selftest)")
    p.add_argument("--keep", action="store_true", help="не удалять распакованное решение")
    p.add_argument("--skip-run", dest="skip_run", action="store_true",
                   help="только статические проверки архива, без запуска")
    return p.parse_args()


def check_archive(app: Path) -> list[str]:
    problems: list[str] = []
    for f in ("metadata.json", "run.py"):
        if not (app / f).is_file():
            problems.append(f"нет {f}")
    if (app / "metadata.json").is_file():
        md = json.loads((app / "metadata.json").read_text(encoding="utf-8"))
        if md.get("entry_point") != "python -u run.py":
            problems.append(f"entry_point не тот: {md.get('entry_point')!r}")
        print(f"    образ: {md.get('image')}")

    ce = sorted(d for d in (app / "artifacts").glob("ce_f*") if d.is_dir()) \
        if (app / "artifacts").is_dir() else []
    if not ce:
        problems.append("в artifacts нет ни одного каталога ce_f*")
    cfgs = []
    for d in ce:
        for need in ("config.json", "model.safetensors", "serialize_config.json"):
            if not (d / need).is_file():
                problems.append(f"в {d.name} нет {need}")
        p = d / "serialize_config.json"
        if p.is_file():
            cfgs.append((d.name, json.loads(p.read_text(encoding="utf-8"))))
    if cfgs:
        print(f"    формат текста: {cfgs[0][1]}")
        for name, c in cfgs[1:]:
            if c != cfgs[0][1]:
                problems.append(f"{name}: формат текста отличается от {cfgs[0][0]}")
    if ce and not all((d / "key_idf.json").is_file() for d in ce) \
            and not (app / "artifacts" / "key_idf.json").is_file():
        problems.append("нет key_idf.json ни в каталогах моделей, ни в artifacts")
    for name, c in cfgs:
        if c.get("canon") and not (app / "artifacts" / name / "key_idf.json").is_file():
            problems.append(f"{name}: канонические ключи, но нет своего key_idf.json")

    blend_p = app / "artifacts" / "blend.json"
    if not blend_p.is_file():
        problems.append("нет artifacts/blend.json")
    else:
        blend = json.loads(blend_p.read_text(encoding="utf-8"))
        if "max_len" not in blend:
            problems.append("в blend.json нет max_len")
        for d in ce:
            tm = d / "train_meta.json"
            if not tm.is_file():
                print(f"    ЗАМЕТЬТЕ: в {d.name} нет train_meta.json, сверить "
                      f"max_len не с чем")
                continue
            want = json.loads(tm.read_text(encoding="utf-8")).get("max_len")
            if want is not None and want != blend.get("max_len"):
                problems.append(
                    f"{d.name} обучен с max_len={want}, а blend.json задаёт "
                    f"{blend.get('max_len')}: обрезка меняет то, что видит модель")
    if (app / "artifacts" / "allow_fallback").is_file():
        print("    ЗАМЕТЬТЕ: в архиве лежит artifacts/allow_fallback, но run.py "
              "этой версии его не читает — резерва нет ни в каком виде. "
              "Файл вводит в заблуждение, уберите его")
    if (app / "vendor").is_dir():
        problems.append("в архиве есть vendor/ — вендоренные колёса уже роняли решение")

    allowed = set(sys.stdlib_module_names) | ALLOWED_THIRD_PARTY
    for f in sorted(app.rglob("*.py")):
        try:
            tree = ast.parse(f.read_text(encoding="utf-8"), filename=str(f))
        except SyntaxError as e:
            problems.append(f"{f.relative_to(app)}: синтаксическая ошибка {e}")
            continue
        # Строки импортов, лежащие внутри функций: они выполняются только при
        # вызове. Импорт на уровне модуля выполняется всегда, поэтому чужой
        # модуль там — отказ, а в теле функции — предупреждение.
        lazy = set()
        for fn in ast.walk(tree):
            if isinstance(fn, (ast.FunctionDef, ast.AsyncFunctionDef)):
                lazy.update(range(fn.lineno, (fn.end_lineno or fn.lineno) + 1))
        for node in ast.walk(tree):
            if isinstance(node, ast.Import):
                names = [a.name.split(".")[0] for a in node.names]
            elif isinstance(node, ast.ImportFrom) and not node.level:
                names = [(node.module or "").split(".")[0]]
            else:
                continue
            for nm in names:
                if not nm or nm in allowed:
                    continue
                where = f"{f.relative_to(app)}:{node.lineno}"
                if node.lineno in lazy:
                    print(f"    ~~ {where} импортирует {nm} внутри функции — "
                          f"выполнится только при её вызове, на пути инференса "
                          f"она не вызывается")
                else:
                    problems.append(
                        f"{where} импортирует {nm} на уровне модуля — "
                        f"этого модуля в образе может не быть")
    return problems


def make_synthetic(dest: Path, n_pairs: int, seed: int = 7, name: str = "synth"):
    import random

    import pyarrow as pa
    import pyarrow.parquet as pq

    rng = random.Random(seed)
    cats = [f"категория {i}" for i in range(20)]
    brands = ["альфа", "бета", "гамма", "дельта", "омега"]
    n_items = max(4, n_pairs)
    ids, names, attrs, item_cats = [], [], [], []
    for i in range(n_items):
        b = brands[i % len(brands)]
        ids.append(1000 + i)
        names.append(f"{b} модель {i} набор {rng.randint(1, 9)} шт")
        attrs.append(json.dumps({
            "бренд": b,
            "артикул": f"{100000 + i}",
            "цвет": rng.choice(["черный", "белый", "синий"]),
            "вес, кг": f"{rng.randint(1, 5)}",
            "материал": rng.choice(["хлопок", "сталь", "пластик"]),
        }, ensure_ascii=False))
        item_cats.append(cats[i % len(cats)])

    id1, id2 = [], []
    for _ in range(n_pairs):
        a = rng.randrange(n_items)
        b = rng.randrange(n_items)
        b = (a + rng.choice([0, 20, 40])) % n_items if rng.random() < 0.8 else b
        id1.append(ids[a])
        id2.append(ids[b])

    dest.mkdir(parents=True, exist_ok=True)
    ip, mp = dest / f"items_{name}.parquet", dest / f"matches_{name}.parquet"
    pq.write_table(pa.table({
        "id": pa.array(ids, pa.int64()), "name": pa.array(names, pa.string()),
        "attributes": pa.array(attrs, pa.string()),
        "category": pa.array(item_cats, pa.string())}), ip)
    pq.write_table(pa.table({
        "id1": pa.array(id1, pa.int64()), "id2": pa.array(id2, pa.int64())}), mp)
    return ip, mp, n_pairs


def make_from_data(dest: Path, n_pairs: int, name: str, seed: int = 7):
    import numpy as np
    import pyarrow as pa
    import pyarrow.parquet as pq

    dest.mkdir(parents=True, exist_ok=True)
    ip, mp = dest / f"items_{name}.parquet", dest / f"matches_{name}.parquet"
    if ip.is_file() and mp.is_file():
        return ip, mp, pq.ParquetFile(mp).metadata.num_rows

    m = pq.read_table(DATA / "matches.parquet", columns=["id1", "id2"])
    total = m.num_rows
    rng = np.random.default_rng(seed)
    if n_pairs <= total:
        idx = rng.choice(total, size=n_pairs, replace=False)
        sub = m.take(pa.array(idx))
    else:
        reps = -(-n_pairs // total)
        sub = pa.concat_tables([m] * reps).slice(0, n_pairs)
    ids = set(sub.column("id1").to_pylist()) | set(sub.column("id2").to_pylist())

    it = pq.read_table(DATA / "items_human.parquet",
                       columns=["id", "name", "attributes", "category"])
    keep = [i for i, v in enumerate(it.column("id").to_pylist()) if v in ids]
    pq.write_table(it.take(pa.array(keep)), ip)
    pq.write_table(sub, mp)
    print(f"    {name}: {sub.num_rows:,} пар, {len(keep):,} товаров")
    return ip, mp, sub.num_rows


def check_output(path: Path, expected_rows: int) -> list[str]:
    problems: list[str] = []
    if not path.is_file():
        return [f"файл результата не создан: {path}"]
    with path.open(encoding="utf-8", newline="") as f:
        rd = csv.reader(f)
        header = next(rd, None)
        if header != ["id1", "id2", "predict"]:
            problems.append(f"заголовок не тот: {header}")
        vals: list[float] = []
        rows = 0
        bad = 0
        for row in rd:
            rows += 1
            if len(row) != 3 or any(c == "" for c in row):
                bad += 1
                continue
            try:
                vals.append(float(row[2]))
            except ValueError:
                bad += 1
    if rows != expected_rows:
        problems.append(f"строк {rows:,}, ожидалось {expected_rows:,}")
    if bad:
        problems.append(f"битых или пустых строк: {bad}")
    if vals:
        uniq = len(set(vals))
        zeros = sum(1 for v in vals if v == 0.0) / len(vals)
        print(f"    predict: min={min(vals):.6g} max={max(vals):.6g} "
              f"среднее={sum(vals) / len(vals):.6g} нулей={100 * zeros:.2f}% "
              f"различных значений={uniq:,}")
        if any(v != v for v in vals):
            problems.append("в predict есть NaN")
        if uniq < min(10, max(2, len(vals) // 10)):
            problems.append(f"слишком мало различных значений predict ({uniq}): "
                            f"похоже, модель не отработала")
    else:
        problems.append("не разобрано ни одного значения predict")
    return problems


def chmod_tree(root: Path, mode: int) -> None:
    for p in sorted(root.rglob("*"), reverse=True):
        if p.is_dir():
            os.chmod(p, mode)
    os.chmod(root, mode)


def main() -> None:
    args = parse_args()

    if args.make_tiny is not None:
        ip, mp, rows = make_synthetic(Path(args.make_tiny), args.pairs or 200,
                                      name="tiny")
        print(f"крошечный набор готов: {rows:,} пар\n  {ip}\n  {mp}")
        raise SystemExit(0)

    if args.zip_path is None:
        raise SystemExit("не задан --zip (архив для проверки)")
    if not args.zip_path.is_file():
        raise SystemExit(f"нет архива: {args.zip_path}")

    work = Path(args.work) if args.work else ROOT / "submissions" / "selftest"
    app = work / "app"
    if app.exists():
        chmod_tree(app, 0o755)
        shutil.rmtree(app)
    app.mkdir(parents=True)
    with zipfile.ZipFile(args.zip_path) as z:
        z.extractall(app)
    print(f"=== архив распакован: {args.zip_path.name} -> {app}")

    ok = True
    print("=== статические проверки")
    problems = check_archive(app)
    for p in problems:
        print(f"    !! {p}")
    if problems:
        ok = False
    else:
        print("    комплектность и импорты в порядке")

    if args.skip_run:
        print("=== запуск пропущен по --skip-run")
        raise SystemExit(0 if ok else 1)

    want = [s.strip() for s in args.stages.split(",") if s.strip()]
    unknown = [s for s in want if s not in LIMITS]
    if unknown:
        raise SystemExit(f"неизвестные этапы: {unknown}; доступны {sorted(LIMITS)}")
    sets_dir = work / "sets"
    print("=== тестовые наборы")
    sets: dict[str, tuple[Path, Path, int, float]] = {}
    for stage in want:
        n_pairs, limit = LIMITS[stage]
        if args.pairs:
            n_pairs = args.pairs
        if args.synthetic:
            ip, mp, rows = make_synthetic(sets_dir / stage, n_pairs)
        else:
            need = [DATA / "matches.parquet", DATA / "items_human.parquet"]
            missing = [str(p) for p in need if not p.is_file()]
            if missing:
                raise SystemExit(
                    f"нет данных {missing}. Либо src/scripts/01_fetch_data.py, "
                    f"либо запуск с --synthetic (тогда проверяется только формат)")
            ip, mp, rows = make_from_data(sets_dir, n_pairs, stage)
        sets[stage] = (ip, mp, rows, limit)

    home = work / "home"
    home.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ)
    for k in ("ECUP_ROOT", "PYTHONPATH"):
        env.pop(k, None)
    env.update({
        "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
        "http_proxy": "http://127.0.0.1:9", "https_proxy": "http://127.0.0.1:9",
        "HOME": str(home),
    })
    if args.budget:
        env["ECUP_BUDGET_S"] = str(args.budget)

    chmod_tree(app, 0o555)
    print(f"=== запуск: {args.python} -u run.py (каталог решения только на чтение, "
          f"сеть заглушена)")

    total = 0.0
    try:
        for stage in want:
            ip, mp, rows, limit = sets[stage]
            for r in range(args.runs):
                outp = work / f"out_{stage}_{r}.csv"
                outp.unlink(missing_ok=True)
                t0 = time.time()
                proc = subprocess.run(
                    [args.python, "-u", "run.py",
                     "--items_path", str(ip), "--matches_path", str(mp),
                     "--output_path", str(outp)],
                    cwd=str(app), env=env, capture_output=True, text=True)
                dt = time.time() - t0
                total += dt
                fit = "в лимите" if dt <= limit else "ПРЕВЫШЕН"
                print(f"\n--- {stage} ({rows:,} пар, лимит {limit}s): {dt:.1f}s {fit}, "
                      f"код {proc.returncode}")
                if proc.returncode != 0:
                    ok = False
                    print(proc.stdout[-4000:])
                    print(proc.stderr[-4000:])
                    continue
                if dt > limit:
                    ok = False
                elif dt > 0.85 * limit:
                    ok = False
                    print(f"    !! запас меньше 15% ({dt:.1f}s из {limit}s): "
                          f"на проверочной машине такой запас не повторится")
                for pb in check_output(outp, rows):
                    ok = False
                    print(f"    !! {pb}")
                print("    последние строки лога решения:")
                for line in proc.stdout.strip().splitlines()[-6:]:
                    print(f"      {line}")
    finally:
        chmod_tree(app, 0o755)
        if not args.keep:
            shutil.rmtree(app, ignore_errors=True)

    print(f"\n=== суммарно {total:.1f}s из {TOTAL_LIMIT:.0f}s "
          f"({total / TOTAL_LIMIT:.0%})")
    if total > TOTAL_LIMIT:
        ok = False
        print("!!! ПРЕВЫШЕН СУММАРНЫЙ ЛИМИТ")
    print("ИТОГ:", "ЗЕЛЁНЫЙ" if ok else "ЕСТЬ ПРОБЛЕМЫ")
    raise SystemExit(0 if ok else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/solution/__init__.py
from .cross_encoder import CrossEncoder

__all__ = ["CrossEncoder"]


In [ ]:
%%writefile src/solution/batch.py
from __future__ import annotations

import json
import re
from collections.abc import Callable, Iterator
from dataclasses import dataclass

import numpy as np

from .progress import track

SPACE_RE = re.compile(r"\s+")
SLASH_RE = re.compile(r"^\s*([^/]{1,40})\s+/\s+")

KEY_JUNK = frozenset(("валюта", "страна-изготовитель", "страна производства",
                      "гарантийный срок", "ндс", "тип упаковки"))


def norm_text(s) -> str:
    return SPACE_RE.sub(" ", (s or "").lower().strip())


def strip_seller(s: str) -> str:
    m = SLASH_RE.match(s)
    return s[m.end():] if m else s


def parse_attrs(s) -> dict:
    if not s:
        return {}
    try:
        d = json.loads(s)
    except Exception:
        return {}
    if not isinstance(d, dict):
        return {}
    return {str(k).strip().lower(): str(v).strip().lower() for k, v in d.items()}


@dataclass
class Batch:
    id1: np.ndarray
    id2: np.ndarray
    name: dict
    attrs: dict
    category: dict
    target: np.ndarray | None = None
    weight: np.ndarray | None = None
    prefix: np.ndarray | None = None

    @property
    def n(self) -> int:
        return len(self.id1)

    @property
    def n_items(self) -> int:
        return len(self.name)

    def pair_category(self) -> np.ndarray:
        cat = self.category
        return np.array([cat.get(a) or cat.get(b) or ""
                         for a, b in zip(self.id1.tolist(), self.id2.tolist())])

    def __repr__(self) -> str:
        t = "—" if self.target is None else f"{(self.target > 0.5).mean():.1%}"
        return f"<Batch {self.n:,} пар / {self.n_items:,} товаров, позитивов {t}>"


@dataclass
class Stream:

    parts: Callable[[], Iterator[Batch]]
    n: int
    n_parts: int

    def __iter__(self) -> Iterator[Batch]:
        return self.parts()

    def __repr__(self) -> str:
        return f"<Stream {self.n:,} пар, {self.n_parts} кусков>"


def as_stream(source) -> Stream:
    if isinstance(source, Stream):
        return source
    if not isinstance(source, Batch):
        raise TypeError(f"обучать можно на Batch или Stream, а не на {type(source).__name__}")
    return Stream(parts=lambda: iter((source,)), n=source.n, n_parts=1)


def from_columns(ids, names, cats, attrs, id1, id2, target=None,
                 prefix=None) -> Batch:
    name = {i: norm_text(n) for i, n in zip(ids, names)}
    at = {i: {k: v for k, v in parse_attrs(a).items() if k not in KEY_JUNK}
          for i, a in track(zip(ids, attrs), "разбор атрибутов", total=len(ids),
                            unit="товар")}
    return Batch(id1=np.asarray(id1), id2=np.asarray(id2), name=name, attrs=at,
                 category=dict(zip(ids, cats)),
                 target=None if target is None else np.asarray(target),
                 prefix=None if prefix is None else np.asarray(prefix, dtype=object))


def read_batch(items_path: str, matches_path: str) -> Batch:
    import pyarrow.parquet as pq

    it = pq.read_table(items_path).to_pydict()
    pr = pq.read_table(matches_path).to_pydict()
    return from_columns(it["id"], it["name"], it["category"], it["attributes"],
                        pr["id1"], pr["id2"], pr.get("target"))


def from_frames(items, pairs) -> Batch:
    tgt = pairs["target"].to_numpy() if "target" in pairs.columns else None
    ctx = (pairs["ctx"].fill_null("").to_list() if "ctx" in pairs.columns
           else None)
    return from_columns(items["id"].to_list(), items["name"].to_list(),
                        items["category"].to_list(), items["attributes"].to_list(),
                        pairs["id1"].to_numpy(), pairs["id2"].to_numpy(), tgt,
                        prefix=ctx)


def merge(*batches: Batch) -> Batch:
    if not batches:
        raise ValueError("склеивать нечего")
    if any(b.target is None for b in batches):
        raise ValueError("склеиваются только размеченные батчи — у одного нет target")

    name, attrs, category = {}, {}, {}
    for b in batches:
        name.update(b.name)
        attrs.update(b.attrs)
        category.update(b.category)
    weight = (None if all(b.weight is None for b in batches) else
              np.concatenate([np.ones(b.n, dtype=np.float32) if b.weight is None
                              else np.asarray(b.weight, dtype=np.float32)
                              for b in batches]))
    prefix = (None if all(b.prefix is None for b in batches) else
              np.concatenate([np.full(b.n, "", dtype=object) if b.prefix is None
                              else np.asarray(b.prefix, dtype=object)
                              for b in batches]))
    return Batch(id1=np.concatenate([b.id1 for b in batches]),
                 id2=np.concatenate([b.id2 for b in batches]),
                 name=name, attrs=attrs, category=category,
                 target=np.concatenate([np.asarray(b.target, dtype=np.float32)
                                        for b in batches]),
                 weight=weight, prefix=prefix)


def slice_folds(items, pairs, mask) -> Batch:
    import polars as pl

    sub = pairs.filter(pl.Series(np.asarray(mask)))
    ids = pl.concat([sub["id1"], sub["id2"]]).unique()
    return from_frames(items.filter(pl.col("id").is_in(ids)), sub)


In [ ]:
%%writefile src/solution/blend.py
from __future__ import annotations

import argparse
import json
import os
import sys
import time
from pathlib import Path

_T0 = time.time()

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

ROOT = Path(__file__).resolve().parents[2]

_CPU = str(min(os.cpu_count() or 20, 20))
os.environ.setdefault("OMP_NUM_THREADS", _CPU)
os.environ.setdefault("MKL_NUM_THREADS", _CPU)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")
os.environ.setdefault("ECUP_ROOT", str(ROOT))
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")

BGE_RATE_REF = 430.0
E2_RATE_REF = 278.0
E2_SETUP_REF = 110.0
SPEED_FLOOR = 1.3
STAGE_BUDGET = ((5_000, 50.0), (150_000, 320.0), (None, 730.0))


def log(msg: str) -> None:
    print(f"[{time.time() - _T0:7.1f}s] {msg}", flush=True)


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--items_path", "--items-path", "-i", dest="items_path")
    p.add_argument("--matches_path", "--matches-path", "-m", dest="matches_path")
    p.add_argument("--output_path", "--output-path", "-o", dest="output_path")
    args, unknown = p.parse_known_args()
    if unknown:
        log(f"игнорирую неизвестные аргументы: {unknown}")
    for name in ("items_path", "matches_path", "output_path"):
        if not getattr(args, name):
            p.error(f"не задан --{name}")
    return args


def canary() -> None:
    log(f"python {sys.version.split()[0]} | ROOT={ROOT}")
    try:
        log(f"artifacts: {sorted(os.listdir(ROOT / 'artifacts'))}")
    except OSError as exc:
        log(f"artifacts не читается: {exc}")
    for m in ("numpy", "pyarrow", "torch", "transformers"):
        try:
            log(f"  импорт {m:14s} OK {getattr(__import__(m), '__version__', '')}")
        except Exception as exc:
            log(f"  импорт {m:14s} СБОЙ {type(exc).__name__}: {exc}")
    try:
        import torch
        log(f"  cuda={torch.cuda.is_available()} "
            f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''}")
    except Exception as exc:
        log(f"  torch.cuda недоступен: {exc}")


def stage_budget(n_pairs: int) -> float:
    if override := os.environ.get("ECUP_BUDGET_S"):
        return float(override)
    for limit, budget in STAGE_BUDGET:
        if limit is None or n_pairs <= limit:
            return budget
    raise AssertionError("последний ярус STAGE_BUDGET обязан быть None")


def rank_within(scores, category):
    import numpy as np

    category = np.asarray(category)
    if category.dtype == object:
        category = category.astype(str)
    out = np.zeros(len(scores), dtype=np.float64)
    for c in np.unique(category):
        k = np.where(category == c)[0]
        s = scores[k]
        r = np.empty(len(s), dtype=np.float64)
        r[np.argsort(s, kind="stable")] = np.arange(len(s), dtype=np.float64)
        out[k] = r / max(len(s) - 1, 1)
    return out


def _read(items_path: str, matches_path: str):
    import pyarrow.parquet as pq

    mt = pq.read_table(matches_path, columns=["id1", "id2"])
    id1 = mt.column("id1").to_numpy(zero_copy_only=False).tolist()
    id2 = mt.column("id2").to_numpy(zero_copy_only=False).tolist()
    it = pq.read_table(items_path,
                       columns=["id", "name", "attributes", "category"])
    return (id1, id2, it.column("id").to_numpy(zero_copy_only=False).tolist(),
            it.column("name").to_pylist(), it.column("attributes").to_pylist(),
            it.column("category").to_pylist())


def _score_bge(approach: str, ids, names, cats, attrs, id1, id2, log):
    import numpy as np

    from .batch import from_columns
    from .cross_encoder import CrossEncoder

    t0 = time.time()
    batch = from_columns(ids, names, cats, attrs, id1, id2)
    log(f"bge: атрибуты разобраны за {time.time() - t0:.0f}s")
    model = CrossEncoder.load(ROOT / "artifacts" / approach)

    t0 = time.time()
    scores = np.asarray(model.predict(batch), dtype=np.float64)
    dt = max(time.time() - t0, 1e-6)
    speed = max((len(id1) / dt) / BGE_RATE_REF, SPEED_FLOOR)
    log(f"bge: {len(id1):,} пар за {dt:.0f}s ({len(id1) / dt:,.0f} пар/с) — "
        f"машина быстрее опорной карты в {speed:.2f}x")

    del model, batch
    try:
        import torch
        torch.cuda.empty_cache()
    except Exception:
        pass
    return scores, speed


def _score_e2(order, n_of, pair_cat, ids, names, cats, attrs, pos, id1, id2,
              deadline, budget, speed, log):
    import numpy as np

    n = len(id1)
    out = np.full(n, np.nan)
    setup = E2_SETUP_REF / speed * (len(ids) / 700_000.0)
    rate = E2_RATE_REF * speed * 0.85
    left = deadline - time.time()
    need_all = setup + n / rate
    log(f"mmb: на все {n:,} пар нужно ~{need_all:.0f}s (подготовка "
        f"{setup:.0f}s + {rate:,.0f} пар/с), в запасе {left:.0f}s")

    one_shot = need_all <= (left - setup) * 0.65
    log(f"mmb: {'один заход на всё' if one_shot else 'дроблю на два захода'}")

    os.environ["ECUP_BUDGET_S"] = str(budget)
    from src.ecup.solve import predict as e2_predict

    pending, done = list(order), []
    for step in (1, 2):
        if not pending:
            break
        left = deadline - time.time()
        if left <= setup + 20:
            log(f"mmb: заход {step} отменён — в запасе {left:.0f}s, "
                f"одна подготовка стоит {setup:.0f}s")
            break
        room = (left - setup) * ((1.0 if one_shot else 0.5) if step == 1 else 0.92)
        chunk, acc = [], 0
        for c in pending:
            if (acc + n_of[c]) / rate > room:
                continue
            chunk.append(c)
            acc += n_of[c]
        if not chunk:
            log(f"mmb: заход {step} — ни одна категория не влезает "
                f"({rate:,.0f} пар/с, комната {room:.0f}s)")
            break

        idx = np.where(np.isin(pair_cat, chunk))[0]
        log(f"mmb: заход {step} — {len(chunk)} кат., {len(idx):,} пар "
            f"(оценка {setup + acc / rate:.0f}s из {left:.0f}s)")
        try:
            t0 = time.time()
            out[idx] = e2_predict(names, attrs, cats, pos,
                                  [id1[i] for i in idx], [id2[i] for i in idx],
                                  str(ROOT), _T0, log)
            dt = max(time.time() - t0, 1e-6)
            done += chunk
            pending = [c for c in pending if c not in chunk]
            rate = max(len(idx) / max(dt - setup, 1.0), 1.0)
            log(f"mmb: заход {step} готов за {dt:.0f}s ({len(idx) / dt:,.0f} "
                f"пар/с сквозных, {rate:,.0f} пар/с чистых)")
        except Exception as exc:
            import traceback
            log(f"mmb: заход {step} СБОЙ {type(exc).__name__}: {exc}")
            traceback.print_exc()
            out[idx] = np.nan
            break
    return out, done


def _mix(test_cats, pair_cat, s_bge, s_e2, w_by_cat, log):
    import numpy as np

    have_bge, have_e2 = np.isfinite(s_bge), np.isfinite(s_e2)
    if not have_bge.any() and not have_e2.any():
        raise RuntimeError("ни одна модель не отдала результата")

    out = np.zeros(len(pair_cat), dtype=np.float64)
    stat = []
    for c in test_cats:
        m = pair_cat == c
        full_bge = (have_bge & m).sum() == m.sum()
        full_e2 = (have_e2 & m).sum() == m.sum()
        w = w_by_cat[c]
        if full_bge and full_e2:
            out[m] = (w * rank_within(s_bge[m], pair_cat[m])
                      + (1 - w) * rank_within(s_e2[m], pair_cat[m]))
            stat.append(f"{c}:смесь w={w:.2f}")
        elif full_bge:
            out[m] = rank_within(s_bge[m], pair_cat[m])
            stat.append(f"{c}:только bge")
        elif full_e2:
            out[m] = rank_within(s_e2[m], pair_cat[m])
            stat.append(f"{c}:только mmb")
        else:
            src = s_bge if (have_bge & m).sum() >= (have_e2 & m).sum() else s_e2
            ok = m & np.isfinite(src)
            if ok.any():
                out[ok] = rank_within(src[ok], pair_cat[ok])
            stat.append(f"{c}:частично ({int(ok.sum())}/{int(m.sum())})")
    log("смесь: " + " | ".join(stat))
    log(f"категорий со смесью: {sum('смесь' in s for s in stat)}/{len(test_cats)}")
    return out


def main() -> None:
    args = parse_args()
    canary()
    cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8-sig"))
    w_cfg = {str(k): float(v) for k, v in cfg["w_bge"].items()}
    w_default = float(cfg.get("w_default", 0.5))
    min_pairs = int(cfg.get("min_pairs_for_bge", 5000))

    import numpy as np

    id1, id2, ids, names, attrs, cats = _read(args.items_path, args.matches_path)
    n = len(id1)
    pos = {v: i for i, v in enumerate(ids)}
    log(f"пар: {n:,}   товаров: {len(ids):,}")

    cat_of = {v: ("" if cats[i] is None else str(cats[i])) for i, v in enumerate(ids)}
    pair_cat = np.array([cat_of.get(a) or cat_of.get(b) or ""
                         for a, b in zip(id1, id2)])
    test_cats = sorted(set(pair_cat.tolist()))
    n_of = {c: int((pair_cat == c).sum()) for c in test_cats}
    w_by_cat = {c: w_cfg.get(c, w_default) for c in test_cats}

    budget = stage_budget(n)
    reserve = max(4.0, min(30.0, budget * 0.06))
    deadline = _T0 + budget - reserve
    log(f"бюджет этапа {budget:.0f}s (резерв {reserve:.0f}s)")

    s_bge, speed = np.full(n, np.nan), SPEED_FLOOR
    if n >= min_pairs:
        try:
            s_bge, speed = _score_bge(cfg["bge_dir"], ids, names, cats, attrs,
                                      id1, id2, log)
        except Exception as exc:
            import traceback
            log(f"bge: СБОЙ {type(exc).__name__}: {exc}")
            traceback.print_exc()
            log("bge: откат — всё считает mmb")
            s_bge = np.full(n, np.nan)
    else:
        log(f"пар {n:,} < {min_pairs:,}: bge не поднимаем, считает только mmb")

    gain = {c: 0.02 + max(0.0, 1.0 - w_by_cat[c]) * 0.05 for c in test_cats}
    order = sorted(test_cats, key=lambda c: -gain[c] / max(n_of[c], 1))
    s_e2, done = _score_e2(order, n_of, pair_cat, ids, names, cats, attrs, pos,
                           id1, id2, deadline, budget, speed, log)
    log(f"mmb: посчитано {len(done)}/{len(test_cats)} категорий")

    out = np.nan_to_num(_mix(test_cats, pair_cat, s_bge, s_e2, w_by_cat, log),
                        nan=0.0, posinf=1.0, neginf=0.0)
    log(f"итог: min={out.min():.4f} max={out.max():.4f} mean={out.mean():.4f}")

    with open(args.output_path, "w", encoding="utf-8") as f:
        f.write("id1,id2,predict\n")
        for a, b, s in zip(id1, id2, out.tolist()):
            f.write(f"{a},{b},{s:.6g}\n")
    log(f"записано {n:,} строк -> {args.output_path}")


In [ ]:
%%writefile src/solution/cross_encoder.py
from __future__ import annotations

import json
import math
import time
from pathlib import Path

import numpy as np

from . import text as textlib
from .batch import Batch, as_stream
from .progress import bar

BASE_MODEL = "BAAI/bge-reranker-v2-m3"
MAX_LEN = 224
HEARTBEAT = 100_000
INFER_HEARTBEAT = 50_000
ATTR_CHARS = 256
SEED = 42
SCHEDULES = ("linear", "cosine")


def _check_schedule(schedule: str) -> str:
    if schedule not in SCHEDULES:
        raise ValueError(f"schedule должен быть 'linear' или 'cosine', "
                         f"получено {schedule!r}")
    return schedule


def item_text(name: str, attrs: dict, attr_chars: int = ATTR_CHARS) -> str:
    return textlib.side_text(name, attrs, attr_chars=attr_chars)


class CrossEncoder:

    PARAMS = ("base_model", "max_len", "attr_chars", "val_chars", "epochs",
              "batch_size", "grad_accum", "lr", "warmup", "seed", "flip",
              "infer_batch", "use_category", "rank_attrs", "grad_checkpoint",
              "init_from", "total_steps", "schedule", "bucket", "ctx_drop")

    def __init__(self, base_model: str = BASE_MODEL, max_len: int = MAX_LEN,
                 attr_chars: int = ATTR_CHARS, val_chars: int = 40, epochs: int = 1,
                 batch_size: int = 32, grad_accum: int = 4, lr: float = 1e-5,
                 warmup: float = 0.05, seed: int = SEED, flip: bool = False,
                 infer_batch: int = 128, use_category: bool = True,
                 rank_attrs: bool = True, grad_checkpoint: bool = False,
                 init_from: str | None = None, total_steps: int | None = None,
                 schedule: str = "linear", bucket: int = 0, ctx_drop: float = 0.0):
        self.base_model = base_model
        self.max_len = int(max_len)
        self.attr_chars = int(attr_chars)
        self.val_chars = int(val_chars)
        self.epochs = int(epochs)
        self.batch_size = int(batch_size)
        self.grad_accum = max(int(grad_accum), 1)
        self.lr = float(lr)
        self.warmup = float(warmup)
        self.schedule = _check_schedule(schedule)
        self.seed = int(seed)
        self.flip = bool(flip)
        self.infer_batch = int(infer_batch)
        self.use_category = bool(use_category)
        self.rank_attrs = bool(rank_attrs)
        self.grad_checkpoint = bool(grad_checkpoint)
        self.bucket = max(int(bucket), 0)
        self.ctx_drop = min(max(float(ctx_drop), 0.0), 1.0)
        self.total_steps = None if total_steps is None else max(int(total_steps), 1)
        self.init_from = None if init_from is None else str(init_from)
        self.key_order: dict = {}
        self.tok = None
        self.model = None
        self._head_hook = False
        self._optimizer = None
        self._optim_steps = 0

    def retune(self, lr: float | None = None, epochs: int | None = None,
               schedule: str | None = None) -> "CrossEncoder":
        if lr is not None:
            self.lr = float(lr)
        if epochs is not None:
            self.epochs = int(epochs)
        if schedule is not None:
            self.schedule = _check_schedule(schedule)
        self.reset_optimizer()
        return self

    def reset_optimizer(self, total_steps: int | None = None) -> None:
        self.total_steps = None if total_steps is None else max(int(total_steps), 1)
        self._optimizer = None
        self._optim_steps = 0

    def _horizon(self, n_pairs: int, per_step: int | None = None) -> int:
        per_step = per_step or self.batch_size * self.grad_accum
        return max(math.ceil(n_pairs / per_step), 1)

    def _ensure_optimizer(self) -> None:
        if self._optimizer is None:
            import torch
            self._optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.lr,
                                                 weight_decay=0.01)

    def _optimizer_step(self) -> float:
        self._ensure_optimizer()

        horizon = self.total_steps
        warm = max(int((horizon or 1) * self.warmup), 1)
        step = self._optim_steps
        if step < warm:
            scale = (step + 1) / warm
        elif horizon is None:
            scale = 1.0
        elif self.schedule == "cosine":
            progress = min((step - warm) / max(horizon - warm, 1), 1.0)
            scale = 0.5 * (1.0 + math.cos(math.pi * progress))
        else:
            scale = max((horizon - step) / max(horizon - warm, 1), 0.0)
        lr = self.lr * scale
        for group in self._optimizer.param_groups:
            group["lr"] = lr
        self._optimizer.step()
        self._optim_steps += 1
        return lr

    @property
    def device(self):
        import torch
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _to_device(self, train: bool):
        import torch

        cuda = torch.cuda.is_available()
        if train or not cuda:
            self.model.to(self.device, dtype=torch.float32)
        else:
            self.model.to(self.device, dtype=self._eval_dtype())
            self._fp32_head()
        self.model.train(train)

    def _eval_dtype(self):
        import torch

        mt = getattr(getattr(self.model, "config", None), "model_type", "")
        if mt == "t5" and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16

    def _fp32_head(self) -> None:
        import torch

        head = next((h for name in ("classifier", "classification_head")
                     if (h := getattr(self.model, name, None)) is not None), None)
        if head is None:
            return
        head.float()
        if not self._head_hook:
            head.register_forward_pre_hook(lambda _m, args: tuple(
                a.float() if isinstance(a, torch.Tensor) else a for a in args))
            self._head_hook = True

    def _texts(self, batch: Batch) -> tuple[dict, dict]:
        order = self.key_order if self.rank_attrs else None

        def build(show_cat: bool) -> dict:
            return {i: textlib.side_text(
                batch.name.get(i, ""), batch.attrs.get(i, {}),
                batch.category.get(i, ""), order, self.attr_chars,
                self.val_chars, show_cat) for i in batch.name}

        plain = build(False)
        return (build(True) if self.use_category else plain), plain

    @staticmethod
    def _pick(texts: dict, ids: list) -> list[str]:
        return [texts.get(i, "") for i in ids]

    def _encode(self, left: list[str], right: list[str]):
        return self.tok(left, right, padding=True, truncation=True,
                        max_length=self.max_len, return_tensors="pt").to(self.device)

    def _by_length(self, perm: np.ndarray, key: np.ndarray, rng) -> np.ndarray:
        bs = self.batch_size
        n_win = max(-(-len(perm) // (bs * self.bucket)), 1)
        perm = np.concatenate([w[np.argsort(key[w], kind="stable")]
                               for w in np.array_split(perm, n_win)])
        full = len(perm) // bs * bs
        heads = perm[:full].reshape(-1, bs)[rng.permutation(full // bs)]
        return np.concatenate([heads.reshape(-1), perm[full:]])

    def _new_model(self) -> None:
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        self.tok = AutoTokenizer.from_pretrained(self.base_model)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.base_model, num_labels=1)

    def _build(self, batch: Batch, verbose: bool):
        if self.init_from:
            prev = type(self).load(self.init_from)
            same = ("use_category", "rank_attrs", "attr_chars", "val_chars", "max_len")
            if bad := [k for k in same if getattr(prev, k) != getattr(self, k)]:
                raise SystemExit(
                    f"дообучение с {self.init_from}: не совпадают {', '.join(bad)}. "
                    f"Вход обязан быть тем же, иначе модель дообучается на тексте, "
                    f"которого не видела: "
                    + ", ".join(f"{k} {getattr(prev, k)}→{getattr(self, k)}"
                                for k in bad))
            self.key_order = prev.key_order
            self.tok, self.model = prev.tok, prev.model
            if verbose:
                print(f"  старт с весов {self.init_from}", flush=True)
            return

        if self.rank_attrs:
            self.key_order = textlib.key_order(batch)
            if len(self.key_order) < 10 and verbose:
                print(f"  [!] rank_attrs включён, но порядок замерен только для "
                      f"{len(self.key_order)} категорий из 20 — "
                      f"остальные пойдут по алфавиту, как без него", flush=True)
        self._new_model()
        if self.grad_checkpoint:
            self.model.gradient_checkpointing_enable()

    def _announce(self, stream, amp: bool, fresh: bool,
                  batching: str | None = None) -> None:
        extra = "".join((
            "  +категория" if self.use_category else "",
            f"  +порядок ключей ({len(self.key_order)} кат.)" if self.rank_attrs else "",
            "  +чекпойнтинг" if self.grad_checkpoint else "",
            f"  +по длине (окно {self.bucket} батчей)" if self.bucket else "",
            "" if fresh else "  (продолжение)"))
        source = (f"{stream.n:,} пар" if stream.n_parts == 1 else
                  f"{stream.n:,} пар потоком из {stream.n_parts} кусков")
        print(f"  {self.base_model}: {source} × {self.epochs} эпох, "
              f"{batching or f'батч {self.batch_size}×{self.grad_accum}'}, "
              f"устройство {self.device.type}"
              + (" (bf16)" if amp else " (fp32)") + extra, flush=True)
        print(f"  optimizer step {self._optim_steps:,} из {self.total_steps:,}; "
              f"AdamW и расписание шага сквозные", flush=True)

    def _announce_infer(self, n: int) -> None:
        import collections

        import torch

        body, _ = collections.Counter(
            p.dtype for p in self.model.parameters()).most_common(1)[0]
        dtype = {torch.float16: "fp16", torch.bfloat16: "bf16",
                 torch.float32: "fp32"}.get(body, str(body))
        extra = "".join((
            "  +категория" if self.use_category else "",
            f"  +порядок ключей ({len(self.key_order)} кат.)" if self.rank_attrs else "",
            "  ×2 (flip)" if self.flip else ""))
        print(f"  {self.base_model}: {n:,} пар, батч {self.infer_batch}, "
              f"устройство {self.device.type} ({dtype})" + extra, flush=True)

    @staticmethod
    def _heartbeat(desc: str, seen: int, total: int, loss: float | None, t0: float,
                   lr: float | None = None) -> None:
        dt = max(time.time() - t0, 1e-6)
        rate = seen / dt
        left = (total - seen) / max(rate, 1e-6)
        eta = (f"{left / 3600:.1f} ч" if left >= 3600 else
               f"{left / 60:.0f} мин" if left >= 120 else f"{left:.0f} с")
        parts = [f"{seen:,}/{total:,} пар"]
        if loss is not None:
            parts.append(f"loss {loss:.4f}")
        parts.append(f"{rate:,.0f} пар/с")
        if lr is not None:
            parts.append(f"lr {lr:.2e}")
        print(f"  {desc}: " + " · ".join(parts) + f" · осталось {eta}", flush=True)

    def fit(self, source, verbose: bool = True, checkpoint=None):
        import torch
        import transformers

        stream = as_stream(source)
        if stream.n == 0:
            raise ValueError("в потоке нет пар — обучать не на чем")

        transformers.logging.set_verbosity_error()
        torch.manual_seed(self.seed)
        if self.total_steps is None:
            self.total_steps = self._horizon(stream.n * self.epochs)

        loss_fn = torch.nn.BCEWithLogitsLoss(reduction="none")
        amp = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        rng = np.random.default_rng(self.seed)
        fresh, started, lr_now = self.model is None, False, self.lr

        for ep in range(self.epochs):
            t0, run, seen, step, beat = time.time(), 0.0, 0, 0, HEARTBEAT
            with bar(total=stream.n, desc=f"эпоха {ep + 1}/{self.epochs}", unit="пара",
                     disable=not verbose) as pb:
                for part in stream:
                    if part.target is None:
                        raise ValueError("в батче нет target — обучать не на чем")
                    if self.model is None:
                        self._build(part, verbose)
                    if not started:
                        self._to_device(train=True)
                        self._ensure_optimizer()
                        self._optimizer.zero_grad(set_to_none=True)
                        if verbose:
                            self._announce(stream, amp, fresh)
                        started = True

                    cat_text, plain_text = self._texts(part)
                    ids1, ids2 = part.id1.tolist(), part.id2.tolist()
                    pref = part.prefix
                    y = np.clip(np.asarray(part.target, dtype=np.float32), 0.0, 1.0)
                    wts = (None if part.weight is None else
                           np.asarray(part.weight, dtype=np.float32))
                    perm = rng.permutation(part.n)
                    if self.bucket:
                        key = np.fromiter(
                            (len(cat_text.get(a, "")) + len(plain_text.get(b, ""))
                             for a, b in zip(ids1, ids2)),
                            dtype=np.int32, count=part.n)
                        if pref is not None:
                            key += np.fromiter((len(x) for x in pref),
                                               dtype=np.int32, count=part.n)
                        perm = self._by_length(perm, key, rng)
                    swap = rng.random(part.n) < 0.5
                    keep = (None if pref is None or not self.ctx_drop
                            else rng.random(part.n) >= self.ctx_drop)

                    for s in range(0, part.n, self.batch_size):
                        idx = perm[s:s + self.batch_size]
                        lids = [ids2[i] if swap[i] else ids1[i] for i in idx]
                        rids = [ids1[i] if swap[i] else ids2[i] for i in idx]
                        left = self._pick(cat_text, lids)
                        if pref is not None:
                            left = [pref[i] + t if keep is None or keep[i] else t
                                    for i, t in zip(idx, left)]
                        enc = self._encode(left, self._pick(plain_text, rids))
                        tgt = torch.from_numpy(y[idx]).to(self.device)

                        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp):
                            logits = self.model(**enc).logits
                        loss = loss_fn(logits.squeeze(-1).float(), tgt)
                        if wts is not None:
                            loss = loss * torch.from_numpy(wts[idx]).to(self.device)
                        loss = loss.mean()
                        (loss / self.grad_accum).backward()
                        step += 1
                        if step % self.grad_accum == 0:
                            lr_now = self._optimizer_step()
                            self._optimizer.zero_grad(set_to_none=True)

                        run += float(loss.detach()) * len(idx)
                        seen += len(idx)
                        pb.update(len(idx))
                        pb.set_postfix_str(f"loss {run / seen:.4f}", refresh=False)
                        if verbose and seen >= beat:
                            self._heartbeat(f"эпоха {ep + 1}/{self.epochs}", seen,
                                            stream.n, run / seen, t0, lr_now)
                            beat += HEARTBEAT

                    if checkpoint is not None:
                        self.save(checkpoint)
                        if verbose:
                            print(f"  чекпойнт после {seen:,} пар → {checkpoint}",
                                  flush=True)

                if step % self.grad_accum:
                    lr_now = self._optimizer_step()
                    self._optimizer.zero_grad(set_to_none=True)
            if verbose:
                dt = time.time() - t0
                print(f"  эпоха {ep + 1}/{self.epochs}: loss {run / max(seen, 1):.4f}   "
                      f"{dt:.0f}s ({seen / max(dt, 1e-6):,.0f} пар/с), "
                      f"lr {lr_now:.2e}", flush=True)
        return self

    def _score(self, left: list[str], right: list[str],
               desc: str = "инференс", verbose: bool = True) -> np.ndarray:
        import torch

        n = len(left)
        out = np.zeros(n, dtype=np.float64)
        lens = np.fromiter((len(x) + len(y) for x, y in zip(left, right)),
                           dtype=np.int32, count=n)
        order = np.argsort(lens, kind="stable")

        self.model.eval()
        t0, seen, beat = time.time(), 0, INFER_HEARTBEAT
        with torch.inference_mode(), bar(total=n, desc=desc, unit="пара",
                                         disable=not verbose) as pb:
            for s in range(0, n, self.infer_batch):
                idx = order[s:s + self.infer_batch]
                enc = self._encode([left[i] for i in idx], [right[i] for i in idx])
                logits = self.model(**enc).logits.double().squeeze(-1)
                out[idx] = torch.sigmoid(logits).cpu().numpy()
                seen += len(idx)
                pb.update(len(idx))
                if verbose and beat <= seen < n:
                    self._heartbeat(desc, seen, n, None, t0)
                    beat += INFER_HEARTBEAT
        if verbose:
            dt = max(time.time() - t0, 1e-6)
            print(f"  {desc}: {n:,} пар за {dt:.0f}s ({n / dt:,.0f} пар/с)", flush=True)
        return out

    def predict(self, batch: Batch, verbose: bool = True) -> np.ndarray:
        if self.model is None:
            raise RuntimeError("модель не обучена и не загружена")
        self._to_device(train=False)
        if verbose:
            self._announce_infer(batch.n)
        cat_text, plain_text = self._texts(batch)
        ids1, ids2 = batch.id1.tolist(), batch.id2.tolist()

        p = self._score(self._pick(cat_text, ids1), self._pick(plain_text, ids2),
                        "инференс 1/2" if self.flip else "инференс", verbose)
        if self.flip:
            p = 0.5 * (p + self._score(self._pick(cat_text, ids2),
                                       self._pick(plain_text, ids1), "инференс 2/2",
                                       verbose))
        return p

    def save(self, path) -> None:
        path = Path(path)
        (path / "model").mkdir(parents=True, exist_ok=True)
        dtype = self._eval_dtype()
        half = {k: (v.to(dtype) if v.is_floating_point() else v)
                for k, v in self.model.state_dict().items()}
        self.model.save_pretrained(path / "model", state_dict=half,
                                   safe_serialization=True)
        self.tok.save_pretrained(path / "model")
        (path / "meta.json").write_text(json.dumps(
            {**{k: getattr(self, k) for k in self.PARAMS}, "key_order": self.key_order},
            ensure_ascii=False, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path):
        from transformers import AutoModelForSequenceClassification, AutoTokenizer

        path = Path(path)
        meta = json.loads((path / "meta.json").read_text(encoding="utf-8-sig"))
        obj = cls(**{k: v for k, v in meta.items() if k in cls.PARAMS})
        obj.key_order = meta.get("key_order") or {}
        obj.tok = AutoTokenizer.from_pretrained(path / "model", local_files_only=True)
        obj.model = AutoModelForSequenceClassification.from_pretrained(
            path / "model", local_files_only=True)
        obj._to_device(train=False)
        return obj


In [ ]:
%%writefile src/solution/progress.py
from __future__ import annotations

import sys

TTY_INTERVAL = 0.3
LOG_INTERVAL = 10.0


class Silent:

    def update(self, n: int = 1) -> None:
        pass

    def set_postfix_str(self, s: str = "", refresh: bool = True) -> None:
        pass

    def close(self) -> None:
        pass

    def __enter__(self):
        return self

    def __exit__(self, *exc) -> bool:
        return False


def _tqdm():
    try:
        from tqdm.auto import tqdm
        return tqdm
    except Exception:
        return None


def _interval() -> float:
    try:
        return TTY_INTERVAL if sys.stderr.isatty() else LOG_INTERVAL
    except Exception:
        return LOG_INTERVAL


def bar(total=None, desc: str = "", unit: str = "it", disable: bool = False,
        initial: int = 0):
    cls = None if disable else _tqdm()
    if cls is None:
        return Silent()
    return cls(total=total, desc=desc, unit=unit, unit_scale=True, initial=initial,
               mininterval=_interval(), dynamic_ncols=True, leave=False)


def track(iterable, desc: str = "", total=None, unit: str = "it",
          disable: bool = False):
    cls = None if disable else _tqdm()
    if cls is None:
        return iterable
    return cls(iterable, desc=desc, total=total, unit=unit, unit_scale=True,
               mininterval=_interval(), dynamic_ncols=True, leave=False)


In [ ]:
%%writefile src/solution/report.py
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

WIDTH = 76
CAT_W = 24
BAR_W = 16

FULL, FLOOR, EMPTY = "█", "▒", "·"


@dataclass
class Row:

    category: str
    value: float
    n: int | None = None
    floor: float | None = None
    delta: float | None = None
    share: float | None = None
    hi: float | None = None
    sd: float | None = None

    @property
    def lift(self) -> float | None:
        return None if self.floor is None else self.value - self.floor


def _bar(value: float, floor: float | None) -> str:
    if not np.isfinite(value):
        return "?" * BAR_W
    v = min(max(value, 0.0), 1.0)
    fill = int(round(v * BAR_W))
    base = 0 if floor is None else min(int(round(min(floor, v) * BAR_W)), fill)
    return FLOOR * base + FULL * (fill - base) + EMPTY * (BAR_W - fill)


def _rule(char: str = "─") -> str:
    return char * WIDTH


def _cell(x, spec: str) -> str:
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "—"
    return format(x, spec).replace(",", " ")


def _columns(rows: list[Row], value_header: str) -> list:
    def has(field):
        return any(getattr(r, field) is not None for r in rows)

    cols = [("категория", CAT_W, "<", lambda r: r.category[:CAT_W])]
    if has("n"):
        cols.append(("пар", 8, ">", lambda r: _cell(r.n, ",")))
    if has("share"):
        cols.append(("доля", 7, ">", lambda r: _cell(r.share, ".1%")))
    cols.append((value_header, 9, ">", lambda r: _cell(r.value, ".4f")))
    if has("delta"):
        cols.append(("Δ", 8, ">", lambda r: _cell(r.delta, "+.4f")))
    if has("hi"):
        cols.append((">0.5", 8, ">", lambda r: _cell(r.hi, ".1%")))
    return cols


def render(kind: str, subject: str, rows: list[Row],
           summary: list[tuple[str, str]] | None = None,
           value_header: str = "PR-AUC", footer: str | None = None) -> str:
    rows = sorted(rows, key=lambda r: (not np.isfinite(r.value), r.value))
    cols = _columns(rows, value_header)

    def line(cells) -> str:
        return "  " + "".join(f"{c:{a}{w}}" for c, (_, w, a, _g) in zip(cells, cols))

    out = [_rule("═"), f"  {kind}" + (f" · {subject}" if subject else ""), _rule("═")]
    out += [f"  {label:<14} {value}" for label, value in (summary or [])]
    out += [_rule(),
            line([h for h, *_ in cols]) + "  распределение",
            _rule()]
    out += [f"{line([g(r) for *_, g in cols])}  {_bar(r.value, r.floor)}"
            for r in rows]
    out += [_rule(), f"  {_spread(rows) if footer is None else footer}", _rule()]
    return "\n".join(out)


def _spread(rows: list[Row]) -> str:
    v = np.array([r.value for r in rows if np.isfinite(r.value)], dtype=float)
    if len(v) < 4:
        return f"категорий {len(v)}"
    k = max(1, len(v) // 5)
    share = v[:k].sum() / v.sum() if v.sum() else float("nan")
    return (f"худшие {k} из {len(v)} дают {share:.1%} суммы   "
            f"медиана {np.median(v):.4f}   разброс {v.min():.4f}…{v.max():.4f}")


def predictions(category, pred, subject: str = "", kind: str = "ТЕСТ") -> str:
    cat = np.asarray(category)
    p = np.asarray(pred, dtype=np.float64)
    total = max(len(p), 1)

    rows = []
    for c in sorted(set(cat.tolist())):
        m = cat == c
        rows.append(Row(str(c) or "(без категории)", float(p[m].mean()),
                        n=int(m.sum()), share=int(m.sum()) / total,
                        hi=float((p[m] > 0.5).mean())))

    summary = [("пар", f"{total:,}".replace(",", " ") +
                f"   категорий {len(rows)}"),
               ("скор", f"среднее {p.mean():.4f}   медиана {np.median(p):.4f}   "
                        f"разброс {p.min():.4f}…{p.max():.4f}")]
    footer = (f"выше 0.5: {(p > 0.5).mean():.1%} пар   "
              f"крупнейшая категория {max(r.share for r in rows):.1%} набора")
    text = render(kind, subject, rows, summary, value_header="ср. скор", footer=footer)
    print(text, flush=True)
    return text


def validation(score, subject: str = "", local=None) -> str:
    rows = [Row(c.category, c.pr_auc, n=c.n, floor=c.pos_rate)
            for c in score.per_category]
    matched = getattr(score, "matched", False)
    summary = [("macro PR-AUC", f"{score.macro:.4f}   категорий {len(rows)}")]
    if score.skipped:
        summary.append(("пропущено", f"{len(score.skipped)} категорий без обоих "
                                     f"классов: {', '.join(score.skipped[:4])} — "
                                     f"macro не в шкале доски, она считает по 20"))
    text = render("ВАЛИДАЦИЯ" + (" · доли позитивов теста" if matched else ""),
                  subject, rows, summary)
    print(text, flush=True)
    return text


In [ ]:
%%writefile src/solution/runner.py
import argparse
import importlib
import json
import os
import sys
import time
from pathlib import Path

_T0 = time.time()

_CPU = str(min(os.cpu_count() or 20, 20))
os.environ.setdefault("OMP_NUM_THREADS", _CPU)
os.environ.setdefault("MKL_NUM_THREADS", _CPU)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_MODULE_LOADING", "LAZY")

ROOT = Path(__file__).resolve().parents[2]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

try:
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")
except Exception:
    pass

FALLBACK_SCORE = 0.0


def log(msg: str) -> None:
    print(f"[{time.time() - _T0:7.1f}s] {msg}", flush=True)


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser()
    p.add_argument("--items_path", "--items-path", "-i", dest="items_path")
    p.add_argument("--matches_path", "--matches-path", dest="matches_path")
    p.add_argument("--output_path", "--output-path", "-o", dest="output_path")
    args, unknown = p.parse_known_args()
    if unknown:
        log(f"игнорирую неизвестные аргументы: {unknown}")
    for name in ("items_path", "matches_path", "output_path"):
        if not getattr(args, name):
            p.error(f"не задан --{name}")
    return args


def load_model(cfg: dict):
    module = importlib.import_module(cfg["module"])
    cls = getattr(module, cfg["class"])
    return cls.load(ROOT / "artifacts" / cfg["approach"])


def main() -> None:
    args = parse_args()
    cfg = json.loads((ROOT / "config.json").read_text(encoding="utf-8-sig"))
    log(f"подход: {cfg['approach']}  ({cfg['module']}.{cfg['class']})")

    import numpy as np
    import pandas as pd

    from .batch import read_batch

    batch = read_batch(args.items_path, args.matches_path)
    log(f"данные: {batch.n:,} пар, {batch.n_items:,} товаров")

    try:
        model = load_model(cfg)
        log("модель загружена")
        pred = model.predict(batch)
    except Exception as exc:
        log(f"ОШИБКА: {type(exc).__name__}: {exc}")
        log("отдаю константу, чтобы пройти Result-стадию")
        import traceback
        traceback.print_exc()
        pred = np.full(batch.n, FALLBACK_SCORE, dtype=np.float32)

    pred = np.nan_to_num(np.asarray(pred, dtype=np.float64).ravel(),
                         nan=FALLBACK_SCORE, posinf=1.0, neginf=0.0)
    if len(pred) != batch.n:
        log(f"[!] модель вернула {len(pred)} предсказаний на {batch.n} пар — добиваю")
        fixed = np.full(batch.n, FALLBACK_SCORE, dtype=np.float64)
        fixed[:min(len(pred), batch.n)] = pred[:batch.n]
        pred = fixed
    log(f"инференс готов: min={pred.min():.4f} max={pred.max():.4f} "
        f"mean={pred.mean():.4f}")

    try:
        from .report import predictions
        predictions(batch.pair_category(), pred,
                    subject=f"{cfg['approach']} · {batch.n:,} пар")
    except Exception as exc:
        log(f"[i] разбивку по категориям напечатать не вышло: "
            f"{type(exc).__name__}: {exc}")

    pd.DataFrame({"id1": batch.id1, "id2": batch.id2, "predict": pred}).to_csv(
        args.output_path, index=False)
    log(f"записано {batch.n:,} строк → {args.output_path}")


In [ ]:
%%writefile src/solution/text.py
from __future__ import annotations

from collections import defaultdict

import numpy as np

from .batch import Batch, strip_seller

VAL_CHARS = 40
MIN_PAIRS = 40
PRIOR = 40.0


def key_order(batch: Batch, min_pairs: int = MIN_PAIRS,
              prior: float = PRIOR) -> dict[str, list[str]]:
    if batch.target is None:
        raise ValueError("в батче нет target — силу ключей мерить не на чем")

    y = np.asarray(batch.target) > 0.5
    cats = batch.pair_category()
    stat: dict = defaultdict(lambda: [0, 0, 0, 0])
    empty: dict = {}

    for a, b, t, c in zip(batch.id1.tolist(), batch.id2.tolist(), y.tolist(),
                          cats.tolist()):
        da, db = batch.attrs.get(a, empty), batch.attrs.get(b, empty)
        if not da or not db:
            continue
        for k in da.keys() & db.keys():
            s = stat[(c, k)]
            same = da[k] == db[k]
            if t:
                s[0] += 1
                s[1] += same
            else:
                s[2] += 1
                s[3] += same

    scored: dict = defaultdict(list)
    for (c, k), (npos, eqpos, nneg, eqneg) in stat.items():
        if npos < min_pairs or nneg < min_pairs:
            continue
        gap = eqpos / npos - eqneg / nneg
        n = npos + nneg
        scored[c].append((abs(gap) * n / (n + prior), k))

    return {c: [k for _, k in sorted(v, key=lambda x: (-x[0], x[1]))]
            for c, v in scored.items()}


def side_text(name: str, attrs: dict, category: str = "",
              order: dict | None = None, attr_chars: int = 120,
              val_chars: int = 0, show_category: bool = True) -> str:
    head = strip_seller(name)
    if category and show_category:
        head = f"{category}: {head}" if head else category
    if not attrs or attr_chars <= 0:
        return head

    rank = (order or {}).get(category)
    if rank:
        pos = {k: i for i, k in enumerate(rank)}
        keys = sorted(attrs, key=lambda k: (pos.get(k, len(pos)), k))
    else:
        keys = sorted(attrs)

    cut = (lambda v: v[:val_chars]) if val_chars > 0 else (lambda v: v)
    tail = "; ".join(f"{k}: {cut(attrs[k])}" for k in keys)[:attr_chars]
    return f"{head} | {tail}" if tail else head


In [ ]:
%%writefile src/utils/__init__.py
from __future__ import annotations


def human(n: float) -> str:
    for u in ("B", "KB", "MB", "GB"):
        if abs(n) < 1024:
            return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}TB"


In [ ]:
%%writefile src/utils/blend.py
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

from . import human
from ..config import (ARTIFACTS, BUILD, DOCKER_IMAGE, ENTRY_POINT, ROOT,
                      SUBMISSIONS)
from .packaging import EXCLUDE_DIRS, MAX_SIZE, _keep

HALF1_ZIP = SUBMISSIONS / "submission_k48b1.zip"
LB_HALF1 = Path(__file__).with_name("lb_k48b1.json")
LB_HALF2 = Path(__file__).with_name("lb_bge_b_all.json")
SPREAD = 0.22

TUNED_W: dict[str, float] = {
    "Автотовары": 0.386,
    "Аптека": 0.462,
    "Бытовая техника": 0.551,
    "Бытовая химия": 0.432,
    "Галантерея и аксессуары": 0.558,
    "Детские товары": 0.422,
    "Дом и сад": 0.631,
    "Канцелярские товары": 0.592,
    "Красота и гигиена": 0.753,
    "Мебель": 0.444,
    "Музыкальные инструменты": 0.463,
    "Обувь": 0.519,
    "Одежда": 0.382,
    "Продукты питания": 0.277,
    "Спорт и отдых": 0.539,
    "Строительство и ремонт": 0.448,
    "Товары для животных": 0.367,
    "Хобби и творчество": 0.386,
    "Электроника": 0.505,
    "Ювелирные изделия": 0.550,
}

W_CLIP = (0.12, 0.88)

RUN_PY = '''"""Точка входа архива-ансамбля — обёртка над src/solution/blend.py."""
from src.solution.blend import main

main()
'''

CATEGORIES: tuple[str, ...] = (
    "Автотовары", "Аптека", "Бытовая техника", "Бытовая химия",
    "Галантерея и аксессуары", "Детские товары", "Дом и сад",
    "Канцелярские товары", "Красота и гигиена", "Мебель",
    "Музыкальные инструменты", "Обувь", "Одежда", "Продукты питания",
    "Спорт и отдых", "Строительство и ремонт", "Товары для животных",
    "Хобби и творчество", "Электроника", "Ювелирные изделия",
)


def equal_weights() -> dict:
    return {c: 0.5 for c in CATEGORIES}


def lb_breakdown(path: Path) -> tuple[dict, dict]:
    if not path.is_file():
        raise SystemExit(
            f"нет {path.name} — официальной разбивки сабмита по категориям. "
            f"Файл лежит в дереве рядом с blend.py и разворачивается "
            f"bootstrap.ipynb; без него веса смешивания вывести не из чего")
    d = json.loads(path.read_text(encoding="utf-8"))
    per = {str(k): float(v) for k, v in d["per_category_prauc"].items()}
    if set(per) != set(CATEGORIES):
        raise SystemExit(
            f"{path.name}: категории разбивки не совпадают с двадцатью "
            f"категориями теста — файл не от этого соревнования")
    return per, {k: d[k] for k in ("submission_id", "file_name", "created_at",
                                   "total_prauc")}


def tuned_weights(verbose: bool = True) -> dict:
    per1, meta1 = lb_breakdown(LB_HALF1)
    per2, meta2 = lb_breakdown(LB_HALF2)
    lo, hi = W_CLIP
    w = {c: round(min(max(0.5 + (per2[c] - per1[c]) / SPREAD, lo), hi), 3)
         for c in CATEGORIES}
    if w != TUNED_W:
        diff = {c: (TUNED_W.get(c), w[c]) for c in w if w[c] != TUNED_W.get(c)}
        raise SystemExit(
            f"веса, выведенные из разбивок доски, разошлись с замороженной "
            f"копией TUNED_W: {diff}. Либо подменены файлы lb_*.json, либо "
            f"изменена формула — и то и другое меняет решение, которое доска "
            f"уже измерила. Восстановите исходные файлы или пересчитайте "
            f"TUNED_W осознанно")
    if verbose:
        print(f"веса половины 2 из разбивок доски: w = 0.5 + (AP2 - AP1) / "
              f"{SPREAD}, зажим {W_CLIP}")
        print(f"  половина 1: {meta1['file_name']} = {meta1['total_prauc']:.4f} "
              f"({meta1['submission_id'][:8]}, {meta1['created_at'][:10]})")
        print(f"  половина 2: {meta2['file_name']} = {meta2['total_prauc']:.4f} "
              f"({meta2['submission_id'][:8]}, {meta2['created_at'][:10]})")
        for c in sorted(w, key=lambda c: w[c]):
            print(f"  {c:<26} w={w[c]:.3f}   AP2 {per2[c]:.4f} / AP1 {per1[c]:.4f}")
    return w


def stage(approach: str, weights: dict, verbose: bool = True,
          half1_zip: Path | str | None = None) -> Path:
    half1_zip = HALF1_ZIP if half1_zip is None else Path(half1_zip)
    if not half1_zip.exists():
        raise SystemExit(
            f"нет {half1_zip.relative_to(ROOT).as_posix()} — из него берутся движок "
            f"`src.ecup` и его чекпойнт. Соберите его в 01_mmbert.ipynb "
            f"(src/scripts/40_build_submission.py).")
    art = ARTIFACTS / approach
    if not art.is_dir():
        raise SystemExit(f"нет весов {art} — сначала model.save(config.ARTIFACTS / "
                         f"'{approach}')")

    root = BUILD / "blend"
    if root.exists():
        shutil.rmtree(root)
    root.mkdir(parents=True)

    with zipfile.ZipFile(half1_zip) as z:
        members = [m for m in z.namelist()
                   if m.startswith(("src/ecup/", "artifacts/"))]
        z.extractall(root, members)
    if not (root / "src" / "ecup").is_dir():
        raise SystemExit(f"в {half1_zip.name} нет каталога src/ecup/ — не тот архив?")

    (root / "src").mkdir(exist_ok=True)
    (root / "src" / "__init__.py").write_text("", encoding="utf-8")
    shutil.copytree(ROOT / "src" / "solution", root / "src" / "solution",
                    ignore=shutil.ignore_patterns(*EXCLUDE_DIRS))
    shutil.copytree(art, root / "artifacts" / approach,
                    ignore=shutil.ignore_patterns(*EXCLUDE_DIRS))

    (root / "run.py").write_text(RUN_PY, encoding="utf-8")
    (root / "metadata.json").write_text(
        json.dumps({"image": DOCKER_IMAGE, "entry_point": ENTRY_POINT}, indent=4),
        encoding="utf-8")
    (root / "config.json").write_text(json.dumps(
        {"bge_dir": approach, "w_bge": weights, "w_default": 0.5,
         "min_pairs_for_bge": 5000}, ensure_ascii=False, indent=1),
        encoding="utf-8")

    if verbose:
        print(f"[stage] {root.relative_to(ROOT).as_posix()}  "
              f"(bge из {art.relative_to(ROOT).as_posix()}, ecup из {half1_zip.name})")
    return root


def preflight(root: Path) -> list[str]:
    problems = []
    for req in ("run.py", "config.json", "metadata.json",
                "src/solution/blend.py", "src/ecup/solve.py"):
        if not (root / req).exists():
            problems.append(f"в архиве нет {req}")

    cfg = json.loads((root / "config.json").read_text(encoding="utf-8-sig"))
    if not (root / "artifacts" / cfg["bge_dir"]).is_dir():
        problems.append(f"нет весов artifacts/{cfg['bge_dir']}/")
    if not sorted((root / "artifacts").glob("ce_f*")):
        problems.append("нет чекпойнта ce_f* — движку src.ecup нечем считать")
    lo, hi = W_CLIP
    if bad := [c for c, w in cfg["w_bge"].items() if not lo <= w <= hi]:
        problems.append(f"вес вне {W_CLIP} у категорий: {bad}")
    return problems


def build(weights: dict,
          approach: str = "bge_10m_len576_B_all",
          name: str = "blend", force: bool = False, verbose: bool = True,
          half1_zip: Path | str | None = None) -> Path:
    if not isinstance(weights, dict) or not weights:
        raise SystemExit("weights обязателен: blend.equal_weights() либо "
                         "blend.tuned_weights()")
    if missing := set(CATEGORIES) - set(weights):
        raise SystemExit(f"нет весов для категорий: {sorted(missing)}")
    root = stage(approach, weights, verbose, half1_zip)

    if problems := preflight(root):
        print("Замечания:")
        for m in problems:
            print(f"  [!] {m}")
        if not force:
            raise SystemExit("Исправьте или вызовите build(..., force=True).")
        print("  force=True: продолжаю")

    files = [p for p in sorted(root.rglob("*")) if p.is_file() and _keep(p)]
    raw = sum(f.stat().st_size for f in files)
    if verbose:
        print(f"файлов: {len(files)}   суммарно {human(raw)}")
        for f in sorted(files, key=lambda x: -x.stat().st_size)[:6]:
            print(f"   {human(f.stat().st_size):>9}  {f.relative_to(root).as_posix()}")

    SUBMISSIONS.mkdir(parents=True, exist_ok=True)
    out = SUBMISSIONS / f"{name}.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for f in files:
            z.write(f, f.relative_to(root).as_posix())

    size = out.stat().st_size
    if verbose:
        print(f"Готово: {out.relative_to(ROOT).as_posix()}  {human(size)} "
              f"(сжатие {size / max(raw, 1):.0%})")
    if size > MAX_SIZE:
        raise SystemExit(f"превышен лимит 5GB на {human(size - MAX_SIZE)}")
    return out


In [ ]:
%%writefile src/utils/bootstrap.py
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

from ..config import ARTIFACTS, PROC, RAW, ROOT, SUBMISSIONS, ensure_dirs

NEEDED = ["numpy", "pandas", "pyarrow", "polars", "scipy", "sklearn", "requests",
          "torch", "transformers"]
OPTIONAL = ["tqdm"]
PIP_NAME = {"sklearn": "scikit-learn"}

CUDA_INDEX = "https://download.pytorch.org/whl/cu128"
TORCH_INSTALL = f"!python3 -m pip install torch --index-url {CUDA_INDEX}"
TORCH_REINSTALL = ["!python3 -m pip uninstall -y torch torchvision torchaudio",
                   TORCH_INSTALL]


def torch_has_cuda() -> bool:
    try:
        import torch
        return torch.version.cuda is not None
    except Exception:
        return False


def install_commands(missing: list[str], cpu_torch: bool) -> list[str]:
    lines = []
    if "torch" in missing:
        lines.append(TORCH_INSTALL)
    elif cpu_torch:
        lines.extend(TORCH_REINSTALL)
    if rest := [PIP_NAME.get(n, n) for n in missing if n != "torch"]:
        lines.append("!python3 -m pip install " + " ".join(rest))
    return lines


def check_env(verbose: bool = True) -> dict:
    found = {}
    for name in NEEDED + OPTIONAL:
        try:
            found[name] = getattr(importlib.import_module(name), "__version__", "?")
        except Exception:
            found[name] = None

    missing = [n for n in NEEDED if not found[n]]
    cpu_torch = bool(found["torch"]) and not torch_has_cuda()
    if verbose:
        print(f"python {sys.version.split()[0]}")
        for name in NEEDED:
            mark = "OK  " if found[name] else "НЕТ "
            note = "  ← без CUDA" if name == "torch" and cpu_torch else ""
            print(f"  {mark} {name:<14} {found[name] or ''}{note}")
        opt = [f"{n} {found[n]}" for n in OPTIONAL if found[n]]
        if opt:
            print("  опционально: " + ", ".join(opt))

        if cpu_torch:
            print(f"\n[!] torch {found['torch']} собран без CUDA — обучение пойдёт "
                  f"на процессоре, а это два порядка, а не проценты")
        if missing:
            print(f"\n[!] не хватает: {', '.join(missing)}")
        if cmds := install_commands(missing, cpu_torch):
            print("\n    " + "\n    ".join(cmds)
                  + "\n\n    команды по одной, torch — первой и отдельно; "
                    "после установки перезапустить ядро")
    return found


def gpu_info(verbose: bool = True) -> dict:
    info = {"available": False, "cuda_build": torch_has_cuda()}
    try:
        import torch
        info["available"] = torch.cuda.is_available()
        if info["available"]:
            info["name"] = torch.cuda.get_device_name(0)
            info["memory_gb"] = round(
                torch.cuda.get_device_properties(0).total_memory / 2 ** 30, 1)
    except Exception:
        pass
    if verbose:
        if info["available"]:
            print(f"GPU: {info['name']} ({info['memory_gb']} ГБ)")
        elif not info["cuda_build"]:
            print("GPU: нет — torch собран без CUDA, см. команды выше")
        else:
            print("GPU: не видна — torch с CUDA есть, устройства нет: драйвер, "
                  "доступ или занята другим процессом")
    return info


def hf_offline(verbose: bool = True) -> bool:
    cache = Path(os.environ.get(
        "HF_HOME", str(Path.home() / ".cache" / "huggingface"))) / "hub"
    if not any(cache.glob("models--*")):
        return False
    os.environ.setdefault("HF_HUB_OFFLINE", "1")
    if verbose:
        print("HF-кэш найден → HF_HUB_OFFLINE=1 (новую базу качать: "
              "os.environ.pop('HF_HUB_OFFLINE'))")
    return True


def describe() -> str:
    lines = [f"ROOT       {ROOT}"]
    for name, p in (("RAW", RAW), ("PROC", PROC), ("ARTIFACTS", ARTIFACTS),
                    ("SUBMISSIONS", SUBMISSIONS)):
        lines.append(f"{name:<12} {'есть' if p.exists() else 'нет '}  {p}")
    return "\n".join(lines)


def setup(group: str = "light", jobs: int = 8, make_folds: bool = True,
          verbose: bool = True) -> dict:
    ensure_dirs()
    if verbose:
        print(describe(), "", sep="\n")

    env = check_env(verbose)
    if missing := [n for n in NEEDED if not env[n]]:
        raise SystemExit(f"Не хватает пакетов: {', '.join(missing)}. "
                         f"Установите и перезапустите ядро.")
    if verbose:
        print()
    gpu = gpu_info(verbose)
    hf_offline(verbose)

    if verbose:
        print()
    from . import data
    data.ensure_data(group=group, jobs=jobs, quiet=not verbose)

    folds_path = PROC / "folds.parquet"
    if make_folds and not folds_path.exists():
        if verbose:
            print("\nстрою фолды...")
        from . import folds
        folds.build(verbose=verbose)
    elif verbose:
        print(f"\nфолды на месте: {folds_path}")

    if verbose:
        print("\nготово.")
    return {"env": env, "gpu": gpu, "root": str(ROOT)}


In [ ]:
%%writefile src/utils/data.py
from __future__ import annotations

import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

from . import human
from ..config import BASE_URL, LOCAL_RUN, PROC, RAW, ensure_dirs
from ..solution.progress import bar

CHUNK = 1024 * 1024
RETRIES = 8
PASSES = 3
LOCK_STALE = 300
LOCK_POLL = 2
LOCK_SAY = 30
PARTS = 8
PART_MIN = 64 * 1024 * 1024
TIMEOUT = 60


@dataclass(frozen=True)
class Asset:
    name: str
    size: int
    groups: tuple[str, ...]


ASSETS: tuple[Asset, ...] = (
    Asset("matches.parquet", 4_120_668, ("light", "all")),
    Asset("matches_llm.parquet", 104_690_716, ("light", "all")),
    Asset("items_human.parquet", 214_210_451, ("light", "all")),
    Asset("items.parquet", 4_104_103_411, ("all",)),
)

GROUPS = {
    "light": "всё, кроме items.parquet (~340 МБ)",
    "all": "вместе с каталогом items.parquet (~4.5 ГБ) — он нужен только тем, кто "
           "берётся за LLM-пары: товаров из них нет в items_human",
}


def target_path(a: Asset) -> Path:
    return RAW / a.name


def url_of(a: Asset) -> str:
    return f"{BASE_URL}/{a.name}"


def is_ready(a: Asset) -> bool:
    p = target_path(a)
    if not p.exists():
        return False
    got = p.stat().st_size
    return got == a.size if a.size else got > 0


def probe(session, url: str) -> tuple[bool, int]:
    with session.get(url, headers={"Range": "bytes=0-0"}, stream=True,
                     timeout=TIMEOUT, allow_redirects=True) as r:
        if r.status_code not in (200, 206):
            raise OSError(f"HTTP {r.status_code}")
        if r.status_code == 206:
            tail = r.headers.get("Content-Range", "").rsplit("/", 1)[-1]
            return True, int(tail) if tail.isdigit() else 0
        return False, int(r.headers.get("Content-Length", 0))


def fetch_stream(session, url: str, part: Path, total: int, desc: str,
                 quiet: bool) -> None:
    have = part.stat().st_size if part.exists() else 0
    if total and have > total:
        part.unlink()
        have = 0
    headers = {"Range": f"bytes={have}-"} if have else {}
    with session.get(url, headers=headers, stream=True, timeout=TIMEOUT) as r:
        if r.status_code not in (200, 206):
            raise OSError(f"HTTP {r.status_code}")
        if r.status_code == 200:
            have = 0
        with open(part, "ab" if have else "wb") as f, \
                bar(total=total or None, desc=desc, unit="B", initial=have,
                    disable=quiet) as b:
            for blk in r.iter_content(CHUNK):
                f.write(blk)
                b.update(len(blk))


def fetch_range(session, url: str, path: Path, start: int, end: int,
                progress) -> None:
    need = end - start + 1
    have = path.stat().st_size if path.exists() else 0
    if have > need:
        path.unlink()
        have = 0
    if have == need:
        return
    with session.get(url, headers={"Range": f"bytes={start + have}-{end}"},
                     stream=True, timeout=TIMEOUT) as r:
        if r.status_code != 206:
            raise OSError(f"HTTP {r.status_code} на диапазоне {start}-{end}")
        with open(path, "ab" if have else "wb") as f:
            for blk in r.iter_content(CHUNK):
                f.write(blk)
                progress.update(len(blk))
    got = path.stat().st_size
    if got != need:
        raise OSError(f"диапазон {start}-{end}: {human(got)} из {human(need)}")


def spans_of(total: int, parts: int) -> list[tuple[int, int, int]]:
    size = (total + parts - 1) // parts
    out = []
    for i in range(parts):
        st, en = i * size, min(total, (i + 1) * size) - 1
        if st <= en:
            out.append((i, st, en))
    return out


def fetch_parts(a: Asset, url: str, dest: Path, total: int, parts: int,
                quiet: bool) -> None:
    import requests

    pdir = Path(f"{dest}.parts")
    pdir.mkdir(parents=True, exist_ok=True)
    spans = spans_of(total, parts)
    done = min(sum(p.stat().st_size for p in pdir.iterdir() if p.is_file()), total)
    last: Exception | None = None
    with bar(total=total, desc=a.name[:34], unit="B", initial=done,
             disable=quiet) as b:
        for attempt in range(1, PASSES + 1):
            failed: list[int] = []
            with ThreadPoolExecutor(max_workers=len(spans)) as ex:
                futs = {ex.submit(fetch_range, requests.Session(), url,
                                  pdir / str(i), st, en, b): i
                        for i, st, en in spans}
                for f in as_completed(futs):
                    try:
                        f.result()
                    except Exception as exc:
                        last = exc
                        failed.append(futs[f])
            if not failed:
                break
            if attempt == PASSES:
                raise OSError(f"диапазоны {sorted(failed)} не добрались — {last}")
            if not quiet:
                print(f"  {a.name}: не добрались {len(failed)} диапазонов, "
                      f"проход {attempt + 1} из {PASSES}")
    got = sum((pdir / str(i)).stat().st_size for i, _, _ in spans)
    if got != total:
        raise OSError(f"диапазоны собраны не полностью: {human(got)} из {human(total)}")
    with open(dest, "wb") as out:
        for i, _, _ in spans:
            with open(pdir / str(i), "rb") as src:
                while blk := src.read(CHUNK * 8):
                    out.write(blk)
    for p in pdir.iterdir():
        p.unlink()
    pdir.rmdir()


def written_bytes(dest: Path) -> int:
    total = 0
    part = dest.with_suffix(dest.suffix + ".part")
    if part.exists():
        total += part.stat().st_size
    pdir = Path(f"{dest}.parts")
    if pdir.is_dir():
        total += sum(p.stat().st_size for p in pdir.iterdir() if p.is_file())
    return total


def take_lock(a: Asset, dest: Path, quiet: bool) -> int | None:
    lock = Path(f"{dest}.lock")
    lock.parent.mkdir(parents=True, exist_ok=True)
    seen, idle, said = -1, 0.0, 0.0
    while True:
        try:
            return os.open(lock, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
        except FileExistsError:
            if is_ready(a):
                return None
            now = written_bytes(dest)
            idle = 0.0 if now != seen else idle + LOCK_POLL
            seen = now
            if idle >= LOCK_STALE:
                if not quiet:
                    print(f"  {a.name}: замок висит {LOCK_STALE}с без движения — "
                          f"беру себе")
                lock.unlink(missing_ok=True)
                continue
            if not quiet and (said == 0.0 or said >= LOCK_SAY):
                print(f"  {a.name}: качает другой процесс ({human(now)}) — жду")
                said = 0.0
            said += LOCK_POLL
            time.sleep(LOCK_POLL)


def download_one(a: Asset, session=None, quiet: bool = False,
                 parts: int = PARTS) -> Path:
    import requests

    session = session or requests.Session()
    dest = target_path(a)
    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    url = url_of(a)

    if is_ready(a):
        if not quiet:
            print(f"  {a.name[:34]:<36} уже есть ({human(dest.stat().st_size)})")
        return dest

    fd = take_lock(a, dest, quiet)
    if fd is None:
        if not quiet:
            print(f"  {a.name[:34]:<36} скачал другой процесс "
                  f"({human(dest.stat().st_size)})")
        return dest

    try:
        if is_ready(a):
            if not quiet:
                print(f"  {a.name[:34]:<36} скачал другой процесс "
                      f"({human(dest.stat().st_size)})")
            return dest
        if dest.exists():
            if not quiet:
                print(f"  {a.name}: на диске {human(dest.stat().st_size)} "
                      f"вместо {human(a.size)} — качаю заново")
            dest.unlink()

        last: Exception | None = None
        for attempt in range(1, RETRIES + 1):
            try:
                ranges, seen = probe(session, url)
                total = a.size or seen
                if ranges and parts > 1 and total >= PART_MIN:
                    fetch_parts(a, url, dest, total, parts, quiet)
                else:
                    fetch_stream(session, url, part, total, a.name[:34], quiet)
                    part.replace(dest)
                got = dest.stat().st_size
                if total and got != total:
                    dest.unlink()
                    raise OSError(f"пришло {human(got)} из {human(total)}")
                return dest
            except Exception as exc:
                last = exc
                if attempt == RETRIES:
                    break
                wait = min(2 ** attempt, 30)
                if not quiet:
                    print(f"  {a.name}: попытка {attempt} из {RETRIES} — {exc}; "
                          f"жду {wait}с")
                time.sleep(wait)
        raise RuntimeError(f"не скачался: {a.name} — {last}")
    finally:
        os.close(fd)
        Path(f"{dest}.lock").unlink(missing_ok=True)


def select(group: str = "light", names: list[str] | None = None) -> list[Asset]:
    if names:
        by_name = {a.name: a for a in ASSETS}
        missing = set(names) - set(by_name)
        if missing:
            raise SystemExit(f"нет таких файлов: {', '.join(sorted(missing))}")
        return [by_name[n] for n in names]
    if group not in GROUPS:
        raise SystemExit(f"группа '{group}' неизвестна. Есть: {', '.join(GROUPS)}")
    return [a for a in ASSETS if group in a.groups]


def ensure_data(group: str = "light", names: list[str] | None = None,
                jobs: int = PARTS, quiet: bool = False) -> None:
    import requests

    ensure_dirs()
    assets = select(group, names)
    todo = [a for a in assets if not is_ready(a)]
    if not todo:
        if not quiet:
            print(f"данные на месте ({len(assets)} файлов), качать нечего")
        return

    session = requests.Session()
    if not quiet:
        print(f"качаю {len(todo)} из {len(assets)} файлов, суммарно "
              f"~{human(sum(a.size for a in todo))} с {BASE_URL}")
    for a in todo:
        download_one(a, session, quiet, parts=max(1, jobs))


def load_items(path: Path | None = None):
    import polars as pl
    return pl.read_parquet(path or (RAW / "items_human.parquet"))


def load_matches(path: Path | None = None):
    import polars as pl
    return pl.read_parquet(path or (RAW / "matches.parquet"))


def load_folds():
    import polars as pl

    p = PROC / "folds.parquet"
    if not p.exists():
        raise SystemExit(f"нет {p}. Сначала: from src.utils import folds; folds.build()")
    return pl.read_parquet(p)


def make_check_slice(n_pairs: int = 1000, seed: int = 42, force: bool = False):
    import polars as pl

    items_p = LOCAL_RUN / "check_items.parquet"
    matches_p = LOCAL_RUN / "check_matches.parquet"
    if items_p.exists() and not force:
        return items_p, matches_p

    pairs = load_matches()
    items = load_items()
    pairs = pairs.join(items.select(["id", "category"]),
                       left_on="id1", right_on="id", how="left")

    per_cat = max(1, n_pairs // max(pairs["category"].n_unique(), 1))
    sub = (pairs.with_columns(pl.int_range(pl.len()).shuffle(seed)
                                .over("category").alias("_r"))
                .filter(pl.col("_r") < per_cat).drop("_r"))

    ids = pl.concat([sub["id1"], sub["id2"]]).unique()
    LOCAL_RUN.mkdir(parents=True, exist_ok=True)
    items.filter(pl.col("id").is_in(ids)).write_parquet(items_p)
    sub.select(["id1", "id2"]).write_parquet(matches_p)
    print(f"[check] {len(sub):,} пар / {len(ids):,} товаров → {LOCAL_RUN}")
    return items_p, matches_p


In [ ]:
%%writefile src/utils/folds.py
from __future__ import annotations

import numpy as np

from ..config import N_FOLDS, PROC, RAW, SEED


def components(id1: np.ndarray, id2: np.ndarray) -> np.ndarray:
    from scipy.sparse import coo_matrix
    from scipy.sparse.csgraph import connected_components

    all_ids, inv = np.unique(np.concatenate([id1, id2]), return_inverse=True)
    ia, ib = inv[: len(id1)], inv[len(id1):]
    g = coo_matrix((np.ones(len(ia), dtype=np.int8), (ia, ib)),
                   shape=(len(all_ids), len(all_ids)))
    _, labels = connected_components(g, directed=False)
    return labels[ia]


def assign(comp: np.ndarray, category: np.ndarray, n_folds: int, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    out = np.full(len(comp), -1, dtype=np.int8)

    for cat in np.unique(category):
        m = category == cat
        comps, sizes = np.unique(comp[m], return_counts=True)
        order = rng.permutation(len(comps))
        comps, sizes = comps[order], sizes[order]
        order = np.argsort(-sizes, kind="stable")
        comps, sizes = comps[order], sizes[order]

        load = np.zeros(n_folds, dtype=np.int64)
        where = {}
        for c, s in zip(comps.tolist(), sizes.tolist()):
            f = int(np.argmin(load))
            where[c] = f
            load[f] += s
        idx = np.flatnonzero(m)
        out[idx] = [where[c] for c in comp[idx].tolist()]
    return out


def build(n_folds: int = N_FOLDS, seed: int = SEED, verbose: bool = True):
    import polars as pl

    PROC.mkdir(parents=True, exist_ok=True)
    m = pl.read_parquet(RAW / "matches.parquet")
    items = pl.read_parquet(RAW / "items_human.parquet", columns=["id", "category"])

    cat_map = dict(zip(items["id"].to_list(), items["category"].to_list()))
    c1 = m["id1"].replace_strict(cat_map, default=None)
    c2 = m["id2"].replace_strict(cat_map, default=None)
    mismatch = int((c1 != c2).sum())
    if mismatch and verbose:
        print(f"[!] у {mismatch:,} пар категории концов различаются — беру категорию id1")
    m = m.with_columns(c1.alias("category"))

    comp = components(m["id1"].to_numpy(), m["id2"].to_numpy())
    fold = assign(comp, m["category"].to_numpy(), n_folds, seed)
    out = m.with_columns(pl.Series("component", comp), pl.Series("fold", fold))
    out.write_parquet(PROC / "folds.parquet")

    if verbose:
        print(f"пар {len(m):,}, компонент {len(np.unique(comp)):,}, фолдов {n_folds}")
        st = (out.group_by("fold")
                 .agg(pl.len().alias("n"), pl.col("target").mean().alias("pos"),
                      pl.col("category").n_unique().alias("cats"))
                 .sort("fold"))
        for r in st.iter_rows(named=True):
            print(f"  фолд {r['fold']}: {r['n']:>7,} пар   pos {r['pos']:.4f}   "
                  f"категорий {r['cats']}")
        print(f"сохранено: {(PROC / 'folds.parquet')}")
    return out


In [ ]:
%%writefile src/utils/lb_bge_b_all.json
{
  "submission_id": "fe51c7e9-3d93-4aaa-afbe-72264182c33f",
  "file_name": "bge_10m_len576_B_all.zip",
  "created_at": "2026-08-26T13:06:10.415Z",
  "total_prauc": 0.53820046655603,
  "per_category_prauc": {
    "Автотовары": 0.8488185111051255,
    "Аптека": 0.6955610763007344,
    "Бытовая техника": 0.6068356169945289,
    "Бытовая химия": 0.6720876780380213,
    "Галантерея и аксессуары": 0.31775550642432765,
    "Детские товары": 0.5973958732469269,
    "Дом и сад": 0.5457369532824454,
    "Канцелярские товары": 0.44648175989471556,
    "Красота и гигиена": 0.6070562985544159,
    "Мебель": 0.4375588389727143,
    "Музыкальные инструменты": 0.5859501884221496,
    "Обувь": 0.1675499091158892,
    "Одежда": 0.15594410830569946,
    "Продукты питания": 0.7414495044430798,
    "Спорт и отдых": 0.4499703214548964,
    "Строительство и ремонт": 0.6255560456627404,
    "Товары для животных": 0.623442843091613,
    "Хобби и творчество": 0.6013831701442066,
    "Электроника": 0.6209909453132997,
    "Ювелирные изделия": 0.41648418235307294
  }
}


In [ ]:
%%writefile src/utils/lb_k48b1.json
{
  "submission_id": "d91e5009-eb9a-42f3-886b-bc1e8d460edd",
  "file_name": "submission_k48b1.zip",
  "created_at": "2026-08-25T12:44:31.768Z",
  "total_prauc": 0.5418665947474801,
  "per_category_prauc": {
    "Автотовары": 0.8738266449525166,
    "Аптека": 0.7040230917791144,
    "Бытовая техника": 0.5955068486603243,
    "Бытовая химия": 0.6869527021784159,
    "Галантерея и аксессуары": 0.3049359026170025,
    "Детские товары": 0.6145817213924227,
    "Дом и сад": 0.517015676404338,
    "Канцелярские товары": 0.4262416983146892,
    "Красота и гигиена": 0.5513938312642508,
    "Мебель": 0.44983706186027667,
    "Музыкальные инструменты": 0.5941610175663955,
    "Обувь": 0.16334998247468765,
    "Одежда": 0.18201105151462882,
    "Продукты питания": 0.7905590185677567,
    "Спорт и отдых": 0.44134814344169093,
    "Строительство и ремонт": 0.6370210584918314,
    "Товары для животных": 0.6527018594819961,
    "Хобби и творчество": 0.6265223438281822,
    "Электроника": 0.6198129864029648,
    "Ювелирные изделия": 0.4055292537561191
  }
}


In [ ]:
%%writefile src/utils/llm.py
from __future__ import annotations

import numpy as np

from . import human
from ..config import PROC, RAW, SEED

CACHE = PROC / "llm"
SPLITS = ("train", "val")

VAL_SHARE = 10


def _paths(split: str) -> tuple:
    return CACHE / f"{split}_pairs.parquet", CACHE / f"{split}_items.parquet"


def ready(split: str = "val") -> bool:
    return all(p.exists() for p in _paths(split))


def ensure(n_train: int = 2_000_000, n_val: int = 200_000, seed: int = SEED,
           holdout: bool = True, force: bool = False, verbose: bool = True) -> dict:
    import polars as pl

    if not force and all(ready(s) for s in SPLITS):
        sizes = {s: len(pl.read_parquet(_paths(s)[0], columns=["id1"]))
                 for s in SPLITS}
        want = {"train": n_train, "val": n_val}
        if small := [s for s in SPLITS if sizes[s] < want[s] * 0.95]:
            print("кэш меньше запрошенного (" + ", ".join(
                f"{s}: {sizes[s]:,} против {want[s]:,}" for s in small)
                + ") — пересобираю", flush=True)
        else:
            if verbose:
                print("LLM-нарезка на месте: " +
                      ", ".join(f"{s} {n:,} пар" for s, n in sizes.items()))
            return {s: _paths(s) for s in SPLITS}

    from .folds import assign, components

    CACHE.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)

    want = int(max(n_train * VAL_SHARE / (VAL_SHARE - 1), n_val * VAL_SHARE) * 1.25)
    pairs = pl.read_parquet(RAW / "matches_llm.parquet")
    if verbose:
        print(f"LLM-пар всего {len(pairs):,}, беру {want:,}", flush=True)
    pairs = pairs.sample(min(want, len(pairs)), seed=seed)

    need = pl.concat([pairs["id1"], pairs["id2"]]).unique()
    if verbose:
        print(f"читаю каталог: нужно {len(need):,} карточек из items.parquet "
              f"(4 ГБ, около минуты)", flush=True)
    items = (pl.scan_parquet(RAW / "items.parquet")
               .select(["id", "name", "attributes", "category"])
               .filter(pl.col("id").is_in(need.implode()))
               .collect())

    cat = dict(zip(items["id"].to_list(), items["category"].to_list()))
    pairs = (pairs.with_columns(
        pl.col("id1").replace_strict(cat, default=None).alias("category"))
        .filter(pl.col("category").is_not_null()))

    comp = components(pairs["id1"].to_numpy(), pairs["id2"].to_numpy())
    fold = assign(comp, pairs["category"].to_numpy(), VAL_SHARE, seed)
    pairs = pairs.with_columns(pl.Series("component", comp),
                               pl.Series("fold", fold))

    val_fold = int(np.argmin(np.bincount(fold, minlength=VAL_SHARE)))
    train_mask = np.ones(len(fold), dtype=bool) if not holdout else fold != val_fold
    out = {}
    for split, mask, n_take in (("val", fold == val_fold, n_val),
                                ("train", train_mask, n_train)):
        sub = pairs.filter(pl.Series(mask))
        if len(sub) > n_take:
            keep = rng.permutation(len(sub))[:n_take]
            sub = sub[np.sort(keep)]
        ids = pl.concat([sub["id1"], sub["id2"]]).unique()
        part = items.filter(pl.col("id").is_in(ids.implode()))

        p_pairs, p_items = _paths(split)
        sub.select(["id1", "id2", "target", "category", "component"]).write_parquet(p_pairs)
        part.write_parquet(p_items)
        out[split] = (p_pairs, p_items)
        if verbose:
            size = sum(p.stat().st_size for p in (p_pairs, p_items))
            print(f"  {split:<6}{len(sub):>10,} пар / {len(part):>9,} карточек   "
                  f"позитивов {(sub['target'] > 0.5).mean():.1%}   {human(size)}")

    if verbose:
        tr = pl.read_parquet(out["train"][1], columns=["id"])["id"]
        va = pl.read_parquet(out["val"][1], columns=["id"])["id"]
        print(f"  общих карточек у train и val: {len(set(tr) & set(va))}")
    return out


def meta(split: str = "val"):
    import polars as pl

    p_pairs, _ = _paths(split)
    if not p_pairs.exists():
        raise SystemExit(f"нет {p_pairs}. Сначала: llm.ensure()")
    return pl.read_parquet(p_pairs)


def _pairs_frame(split: str, n: int | None, seed: int, ctx=None):
    import numpy as np
    import polars as pl

    if split not in SPLITS:
        raise SystemExit(f"split — одно из {SPLITS}, а не '{split}'")
    p_pairs, p_items = _paths(split)
    if not p_pairs.exists():
        raise SystemExit(f"нет {p_pairs}. Сначала: llm.ensure()")

    pairs = pl.read_parquet(p_pairs)
    if n and n < len(pairs):
        keep = np.sort(np.random.default_rng(seed).permutation(len(pairs))[:n])
        pairs = pairs[keep]
    if ctx is not None:
        cx = pl.read_parquet(ctx)
        pairs = pairs.join(cx, on=["id1", "id2"], how="left")
        miss = pairs["ctx"].null_count()
        if miss:
            print(f"  [!] контекста нет у {miss:,} пар из {len(pairs):,} — "
                  f"им пустой префикс", flush=True)
    return pairs, p_items


def _cards(p_items, pairs):
    import polars as pl

    ids = pl.concat([pairs["id1"], pairs["id2"]]).unique()
    return (pl.scan_parquet(p_items)
              .filter(pl.col("id").is_in(ids.implode())).collect())


def load(split: str = "val", n: int | None = None, seed: int = SEED, ctx=None):
    from ..solution.batch import from_frames

    pairs, p_items = _pairs_frame(split, n, seed, ctx)
    return from_frames(_cards(p_items, pairs), pairs)


def stream(split: str = "train", n: int | None = None, chunk: int = 500_000,
           seed: int = SEED, ctx=None):
    from ..solution.batch import Stream, from_frames

    pairs, p_items = _pairs_frame(split, n, seed, ctx)
    total = len(pairs)
    n_parts = max(-(-total // chunk), 1)

    def parts():
        for i in range(n_parts):
            sub = pairs[i * chunk:(i + 1) * chunk]
            yield from_frames(_cards(p_items, sub), sub)

    return Stream(parts=parts, n=total, n_parts=n_parts)


In [ ]:
%%writefile src/utils/noise.py
from __future__ import annotations

import re

import numpy as np

from ..solution.batch import Batch

WORD_RE = re.compile(r"\w+")

W = 0.1
HI = 0.8
LO = 0.3
AGREE = 0.8


def name_tokens(names: dict) -> dict:
    return {i: set(WORD_RE.findall(n or "")) for i, n in names.items()}


def jaccard(tok: dict, id1, id2) -> np.ndarray:
    return np.array([len(tok[a] & tok[b]) / max(len(tok[a] | tok[b]), 1)
                     for a, b in zip(id1, id2)])


def agreement(batch: Batch) -> np.ndarray:
    out = np.full(batch.n, np.nan)
    for k, (a, b) in enumerate(zip(batch.id1.tolist(), batch.id2.tolist())):
        da, db = batch.attrs.get(a, {}), batch.attrs.get(b, {})
        if common := da.keys() & db.keys():
            out[k] = sum(da[c] == db[c] for c in common) / len(common)
    return out


def by_name(target, jac, hi: float = HI, lo: float = LO) -> np.ndarray:
    pos = np.asarray(target) > 0.5
    return (~pos & (np.asarray(jac) >= hi)) | (pos & (np.asarray(jac) <= lo))


def mask(batch: Batch, hi: float = HI, lo: float = LO,
         agree: float = AGREE) -> np.ndarray:
    jac = jaccard(name_tokens(batch.name), batch.id1.tolist(), batch.id2.tolist())
    pos = np.asarray(batch.target) > 0.5
    ag = agreement(batch)
    explained = np.where(pos, ag >= agree, ag < agree)
    return by_name(batch.target, jac, hi, lo) & ~explained


def weights(batch: Batch, w: float = W, hi: float = HI, lo: float = LO,
            agree: float = AGREE, gate: tuple = (), w_gate: float | None = None,
            verbose: bool = True) -> np.ndarray:
    if w_gate is None:
        w_gate = w
    noisy = mask(batch, hi, lo, agree)
    gated = np.isin(batch.pair_category(), gate)
    if verbose:
        print(f"  вес {w}: маской {noisy.mean():.1%}; вес {w_gate}: гейтом "
              f"{gated.mean():.1%} — из {batch.n:,} пар", flush=True)
    out = np.where(noisy, w, 1.0).astype(np.float32)
    out[gated] = w_gate
    return out


In [ ]:
%%writefile src/utils/packaging.py
from __future__ import annotations

import json
import shutil
import zipfile
from pathlib import Path

import numpy as np

from . import human
from ..config import (ARTIFACTS, BUILD, DOCKER_IMAGE, ENTRY_POINT, LOCAL_RUN,
                      ROOT, SUBMISSIONS)

MAX_SIZE = 5 * 1024 ** 3

EXCLUDE_DIRS = {"__pycache__", ".ipynb_checkpoints", ".cache", ".git", ".pytest_cache"}
EXCLUDE_SUFFIX = {".pyc", ".pyo", ".log", ".part"}

RUN_PY = '''"""Точка входа архива — обёртка над src/solution/runner.py."""
from src.solution.runner import main

main()
'''


def _keep(p: Path) -> bool:
    if any(part in EXCLUDE_DIRS for part in p.parts):
        return False
    return p.suffix not in EXCLUDE_SUFFIX


def stage(approach: str, model_class: str, artifacts_dir: Path | None = None,
          verbose: bool = True) -> Path:
    if "." not in model_class:
        raise SystemExit("model_class задаётся как 'модуль.Класс', например "
                         "'src.solution.cross_encoder.CrossEncoder'")
    module, cls = model_class.rsplit(".", 1)

    root = BUILD / approach
    if root.exists():
        shutil.rmtree(root)
    (root / "src").mkdir(parents=True)

    (root / "src" / "__init__.py").write_text("", encoding="utf-8")
    shutil.copytree(ROOT / "src" / "solution", root / "src" / "solution",
                    ignore=shutil.ignore_patterns(*EXCLUDE_DIRS))
    (root / "run.py").write_text(RUN_PY, encoding="utf-8")

    art = Path(artifacts_dir) if artifacts_dir else (ARTIFACTS / approach)
    if not art.is_dir():
        raise SystemExit(f"нет весов {art} — сначала model.save(config.ARTIFACTS / "
                         f"'{approach}')")
    shutil.copytree(art, root / "artifacts" / approach,
                    ignore=shutil.ignore_patterns(*EXCLUDE_DIRS))

    (root / "metadata.json").write_text(
        json.dumps({"image": DOCKER_IMAGE, "entry_point": ENTRY_POINT}, indent=4),
        encoding="utf-8")
    (root / "config.json").write_text(
        json.dumps({"approach": approach, "module": module, "class": cls}, indent=2),
        encoding="utf-8")

    if verbose:
        print(f"[stage] {root.relative_to(ROOT).as_posix()}  "
              f"({module}.{cls}, веса из {art.relative_to(ROOT).as_posix()})")
    return root


def preflight(root: Path) -> list[str]:
    problems = []
    for req in ("run.py", "config.json", "metadata.json", "src/solution/runner.py"):
        if not (root / req).exists():
            problems.append(f"в архиве нет {req}")
    if (root / "requirements.txt").exists():
        problems.append("есть requirements.txt при базовом образе — интернета "
                        "на проверке нет, ставить будет неоткуда")

    cfg = json.loads((root / "config.json").read_text(encoding="utf-8-sig"))
    mod = Path(*cfg["module"].split("."))
    if not ((root / mod.with_suffix(".py")).exists() or (root / mod).is_dir()):
        problems.append(f"config.json ссылается на модуль '{cfg['module']}', "
                        f"которого нет в архиве")

    art = root / "artifacts" / cfg["approach"]
    if not art.is_dir() or not any(art.rglob("*")):
        problems.append(f"нет весов artifacts/{cfg['approach']}/")
    return problems


def build(approach: str, model_class: str, name: str | None = None,
          artifacts_dir: Path | None = None, force: bool = False,
          verbose: bool = True) -> Path:
    root = stage(approach, model_class, artifacts_dir, verbose)

    if problems := preflight(root):
        print("Замечания:")
        for m in problems:
            print(f"  [!] {m}")
        if not force:
            raise SystemExit("Исправьте или вызовите build(..., force=True).")
        print("  force=True: продолжаю")

    files = [p for p in sorted(root.rglob("*")) if p.is_file() and _keep(p)]
    raw = sum(f.stat().st_size for f in files)
    if verbose:
        print(f"файлов: {len(files)}   суммарно {human(raw)}")
        for f in sorted(files, key=lambda x: -x.stat().st_size)[:6]:
            print(f"   {human(f.stat().st_size):>9}  {f.relative_to(root).as_posix()}")

    SUBMISSIONS.mkdir(parents=True, exist_ok=True)
    out = SUBMISSIONS / f"{name or approach}.zip"
    with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for f in files:
            z.write(f, f.relative_to(root).as_posix())

    size = out.stat().st_size
    if verbose:
        print(f"Готово: {out.relative_to(ROOT).as_posix()}  {human(size)} "
              f"(сжатие {size / max(raw, 1):.0%})")
    if size > MAX_SIZE:
        raise SystemExit(f"превышен лимит 5GB на {human(size - MAX_SIZE)}")
    return out


def verify(zip_path, items_path=None, matches_path=None, gpu: bool = True,
           timeout_s: int = 900) -> bool:
    import subprocess
    import tempfile

    if shutil.which("docker") is None:
        print("[i] docker недоступен — проверка пропущена")
        return False

    if items_path is None and matches_path is None:
        from .data import make_check_slice
        make_check_slice()
    items = Path(items_path or (LOCAL_RUN / "check_items.parquet")).resolve()
    matches = Path(matches_path or (LOCAL_RUN / "check_matches.parquet")).resolve()
    if not items.exists() or not matches.exists():
        print(f"[i] нет данных для проверки ({items.name}) — пропускаю")
        return False

    with tempfile.TemporaryDirectory() as tmp:
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(tmp)
        out = Path(tmp) / "_out"
        out.mkdir()
        cmd = ["docker", "run", "--rm", "--network", "none"]
        if gpu:
            cmd += ["--gpus", "all"]
        cmd += ["-v", f"{tmp}:/workspace", "-v", f"{items.parent}:/data:ro",
                "-w", "/workspace", DOCKER_IMAGE, *ENTRY_POINT.split(),
                "--items_path", f"/data/{items.name}",
                "--matches_path", f"/data/{matches.name}",
                "--output_path", "/workspace/_out/verify.csv"]
        rc = subprocess.call(cmd, timeout=timeout_s)
        if rc != 0 or not (out / "verify.csv").exists():
            print(f"[!] архив падает, код {rc}")
            return False

        import pandas as pd
        import pyarrow.parquet as pq

        df = pd.read_csv(out / "verify.csv")
        expected = pq.read_table(matches).num_rows
        if len(df) != expected:
            print(f"[!] строк {len(df):,}, а пар {expected:,} — Result-стадия откажет")
            return False
        if set(df.columns) != {"id1", "id2", "predict"}:
            print(f"[!] колонки {list(df.columns)}, ждали id1,id2,predict")
            return False

        p = df["predict"].to_numpy()
        if not np.isfinite(p).all():
            print("[!] в predict есть nan/inf")
            return False
        if p.min() == p.max():
            print(f"[!] все предсказания равны {p[0]} — модель упала внутри runner, "
                  f"смотрите traceback выше. На лидерборде это уровень случайного.")
            return False

        print(f"архив рабочий: {len(df):,} строк, predict {p.min():.4f}..{p.max():.4f}")
    return True


In [ ]:
%%writefile src/utils/runlog.py
from __future__ import annotations

import atexit
import sys
from datetime import datetime
from pathlib import Path

from ..config import ROOT


class _Tee:

    def __init__(self, stream, log):
        self.stream, self.log = stream, log

    def write(self, text: str):
        result = self.stream.write(text)
        try:
            self.log.write(text.replace("\r", "\n"))
            self.log.flush()
        except OSError:
            pass
        return result

    def flush(self) -> None:
        self.stream.flush()
        try:
            self.log.flush()
        except OSError:
            pass

    def isatty(self) -> bool:
        return self.stream.isatty()

    def __getattr__(self, name):
        return getattr(self.stream, name)


def start(name: str) -> Path:
    active = getattr(sys, "_ecup_run_log", None)
    if active is not None:
        return active[0]

    logs = ROOT / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    path = logs / f"{name}_{stamp}.log"
    log = path.open("a", encoding="utf-8", buffering=1)
    sys.stdout = _Tee(sys.stdout, log)
    sys.stderr = _Tee(sys.stderr, log)
    sys._ecup_run_log = (path, log)
    atexit.register(log.flush)
    print(f"Лог ноутбука: {path.resolve()}", flush=True)
    return path


## 3. Проверка, что дерево на месте

In [ ]:
import os

expected = [
"run.py",
"requirements.txt",
"src/__init__.py",
"src/config.py",
"src/ecup/__init__.py",
"src/ecup/arrowstr.py",
"src/ecup/backbones.py",
"src/ecup/bucket.py",
"src/ecup/config.py",
"src/ecup/folds.py",
"src/ecup/infer.py",
"src/ecup/lb.py",
"src/ecup/losses.py",
"src/ecup/metric.py",
"src/ecup/oof.py",
"src/ecup/serialize.py",
"src/ecup/solve.py",
"src/ecup/testrate.json",
"src/ecup/textprep.py",
"src/scripts/01_fetch_data.py",
"src/scripts/05_build_index.py",
"src/scripts/07_noise_mask.py",
"src/scripts/08_context_feats.py",
"src/scripts/10_prep_text.py",
"src/scripts/13_prep_llm_chunked.py",
"src/scripts/20_pretrain.py",
"src/scripts/30_finetune.py",
"src/scripts/36_clean_metric.py",
"src/scripts/40_build_submission.py",
"src/scripts/50_selftest.py",
"src/solution/__init__.py",
"src/solution/batch.py",
"src/solution/blend.py",
"src/solution/cross_encoder.py",
"src/solution/progress.py",
"src/solution/report.py",
"src/solution/runner.py",
"src/solution/text.py",
"src/utils/__init__.py",
"src/utils/blend.py",
"src/utils/bootstrap.py",
"src/utils/data.py",
"src/utils/folds.py",
"src/utils/lb_bge_b_all.json",
"src/utils/lb_k48b1.json",
"src/utils/llm.py",
"src/utils/noise.py",
"src/utils/packaging.py",
"src/utils/runlog.py"
]

missing = [p for p in expected if not os.path.exists(p)]
if missing:
    raise SystemExit('не записались: ' + ', '.join(missing))
print('все 49 файлов на месте')

## 4. Интерпретатор

`!python3` и `!pip` в тетрадях обязаны попадать в тот же интерпретатор, что и
ядро Jupyter — иначе зависимости ставятся в один Python, а скрипты идут в другой.
Ячейка проверяет это и там, где команды нет или она смотрит не туда, кладёт
шим (`exec <ядро> "$@"`) в первый доступный на запись каталог из `PATH`:
`/usr/local/bin`, иначе `~/.local/bin`. Шимы — файлы, поэтому переживают
смену тетради; shell-алиасы в `!`-командах не живут.

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

PY = sys.executable
KERNEL = pathlib.Path(PY).resolve()
print('ядро:', PY, sys.version.split()[0])


def points_to_kernel(name):
    exe = shutil.which(name)
    if exe is None:
        return None
    if name.startswith('python'):
        r = subprocess.run([exe, '-c', 'import sys; print(sys.executable)'],
                           capture_output=True, text=True)
        return r.returncode == 0 and pathlib.Path(r.stdout.strip()).resolve() == KERNEL
    r = subprocess.run([exe, '--version'], capture_output=True, text=True)
    return r.returncode == 0 and str(pathlib.Path(sys.prefix).resolve()) in r.stdout


if os.name != 'posix':
    print('не posix: шимы не ставятся, тетради зовут ядро как есть')
else:
    candidates = ['/usr/local/bin', os.path.expanduser('~/.local/bin')]
    bindir = next((pathlib.Path(d) for d in candidates
                   if pathlib.Path(d).is_dir() and os.access(d, os.W_OK)), None)
    if bindir is None:
        bindir = pathlib.Path(candidates[-1])
        bindir.mkdir(parents=True, exist_ok=True)
    if str(bindir) not in os.environ['PATH'].split(os.pathsep):
        os.environ['PATH'] = str(bindir) + os.pathsep + os.environ['PATH']

    shims = {'python': [PY], 'python3': [PY],
             'pip': [PY, '-m', 'pip'], 'pip3': [PY, '-m', 'pip']}
    for name, cmd in shims.items():
        state = points_to_kernel(name)
        if state:
            print(f'  {name:<8} уже ядро: {shutil.which(name)}')
            continue
        shim, tmp = bindir / name, bindir / f'.{name}.shim'
        tmp.write_text('#!/bin/sh\nexec ' + ' '.join(f'"{c}"' for c in cmd) + ' "$@"\n')
        tmp.chmod(0o755)
        os.replace(tmp, shim)
        why = 'не найден' if state is None else 'смотрел на другой интерпретатор'
        print(f'  {name:<8} {why} -> {shim}')

    for name in ('python3', 'pip'):
        if not points_to_kernel(name):
            raise SystemExit(f'{name} не совпадает с ядром даже после шима — '
                             f'проверьте PATH: ' + os.environ['PATH'])
    print('python3 и pip совпадают с ядром Jupyter, каталог шимов:', bindir)

## 5. Зависимости

Ставятся через `python3 -m pip` — тем интерпретатором, который раздел 4 закрепил
за ядром. `torch` ставится с индекса CUDA 12.8 — на PyPI лежит другая сборка.
Остальное из `requirements.txt`, который только что записан выше.

In [ ]:
!python3 -m pip install -q torch==2.11.0 --index-url https://download.pytorch.org/whl/cu128
!python3 -m pip install -q -r requirements.txt

In [ ]:
# Проверка окружения: версии и наличие карты.
import importlib

for m in ('torch', 'transformers', 'numpy', 'pyarrow', 'polars', 'sklearn'):
    try:
        mod = importlib.import_module(m)
        print(f'  {m:<14} {getattr(mod, "__version__", "?")}')
    except Exception as exc:
        print(f'  {m:<14} НЕ СТАВИТСЯ: {type(exc).__name__}: {exc}')

import torch
print('  cuda:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print()
import src.ecup.solve, src.solution, src.utils.blend
print('дерево импортируется — можно запускать 01_mmbert.ipynb')